# Revizyon deneyleri: tek parça, tek tıkla

Bu defter, makalenin revize sürümüne eklenen deneylerin hepsini makaledeki veri seti ve protokolle çalıştırır. Sonuçları tek bir zip dosyası olarak indirir. Yeni sentetik veri üretmez; makaledeki 2.000 paylaşımı Drive'dan ya da GitHub deposundan okur.

**Nasıl çalıştırılır**

1. *Çalışma zamanı > Çalışma zamanı türünü değiştir*: **GPU A100** (yoksa L4), **Yüksek RAM açık**.
2. Sol menüdeki anahtar simgesinden üç gizli anahtar ekleyin: `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `GOOGLE_API_KEY`. Her birinde "Not defteri erişimi" açık olmalı.
3. *Çalışma zamanı > Tümünü çalıştır*. Drive erişimi bir kez onay ister; başka bir şey yapmanız gerekmez.
4. Bitince `revizyon_sonuclari_revizyon.zip` kendiliğinden iner; aynı dosya `MyDrive/NLP_Disaster/` altında da durur. Revizyon sonuçlarının tamamı bu dosyadadır.

**Süre ve maliyet:** Yaklaşık 2-4 saat sürer. API maliyeti tahminen 20-25 $; büyük kısmı Claude Opus 5.5'ten gelir. `BUDGET_GUARD_USD` (40 $) aşılırsa yeni çağrı yapılmaz. Gemini ücretsiz kademedeyse günlük istek sınırına takılabilir; faturalandırmayı açın ya da ertesi gün yeniden çalıştırın.

**Yarıda kesilirse:** Tüm API yanıtları ve üretim parçaları Drive'da saklanır. Yeniden *Tümünü çalıştır* dediğinizde yalnızca eksikler istenir; ödenmiş çağrılar tekrar ödenmez.

**Gerçek veri:** İki etiketlemenin tam uyuştuğu paylaşımlardan oluşan altın etiket dosyası (`gercek_veri_altin.csv`; tweet metinleri platform koşulları nedeniyle depoda yoktur, kimlikler ve etiketler `runs/revizyon/data/gercek_veri_altin_kimlikler.csv` dosyasındadır) `MyDrive/NLP_Disaster/gercek_veri/` klasörüne konduğunda gerçek veri deneyleri de çalışır. Dosya yoksa bu adımlar atlanır; dosya hazır olunca defteri yeniden çalıştırmak yeterli.

**Deneyler ve makaledeki bölümleri**

| Deney | Makaledeki bölüm |
|---|---|
| Aynı planlarla Gemini 3.8 Flash ve GPT-5 mini'ye yazdırılan alt küme, çapraz denetim | II.C, III.F |
| Gerçek paylaşımlar: İngilizce (HumAID) ve Türkçe (Kaggle, Türkiye-Suriye depremi) | II.D, III.G |
| BDM'lerin özgün Türkçe metinle değerlendirilmesi | III.B |
| mE5-large-instruct, BGE-M3, Qwen3-Embedding (0,6B ve 4B) | II.F, III.A |
| İnce ayar ablasyonu (ham, NLI, karşıtsal) | III.A |
| İhtiyaç adlarına kısa tanım eklenmesi | II.E, III.A, III.C |
| 66 örnekli BDM istemi ve SetFit | II.H, III.E |
| En fazla 3 sıralı ihtiyaç; mikro/makro F1, tüm ihtiyaçları bulma oranı | III.D |
| Sektör düzeyinde (hiyerarşik) puanlama | II.E, III.D |
| GPT-5 mini'nin 210 paylaşımı ve BDM tekrarları | II.G, III.B |
| İlçe ve konum çıkarımı: sözlük, bulanık eşleme, Türkçe NER | II.J, III.H |
| Her yapılandırmanın benzerlik puanları (kademeli analiz için) | II.I, III.I |

Persona dağılımı, ilçe başına paylaşım duyarlılığı ve kademeli tasarımın isabet ile maliyet analizi mevcut tahmin dosyalarından `analiz/revizyon_*.py` betikleriyle ayrıca hesaplanır.

**Hızlı deneme:** R0 hücresinde `SMOKE_TEST = True` yaparsanız API'ye ve büyük modellere dokunmadan, sahte çıktılarla bütün akış birkaç dakikada çalışır.

In [ ]:
# =============================================================================
# 0. MAKALEDEKİ ÇALIŞTIRMANIN AYARLARI: DEĞİŞTİRMEYİN (revizyon ayarları bir sonraki hücrede)
# =============================================================================
SMOKE_TEST = False            # True: API ve büyük modeller olmadan sahte veri/modelle 2-3 dakikalık deneme

# --- Klasörler -----------------------------------------------------------------
DRIVE_FOLDER_NAME = "NLP_Disaster"   # Google Drive > MyDrive altındaki klasör (yoksa oluşturulur)
RUN_TAG = None                       # None: otomatik tarih-saat etiketi; örn. "deneme1"

# --- Veri üretimi (Claude API) -------------------------------------------------
N_MESSAGES = 2000                    # hedef kayıt sayısı (39 ilçeye eşit dağıtılır)
GEN_MODEL = "claude-opus-5-5"        # üretim modeli. Tahmini üretim maliyeti (2.000 kayıt, THINKING_MODE="minimal"):
                                     #   claude-opus-5-5 ~12-15 $ (düşünme kapatılamaz, effort ile sınırlanır) | claude-sonnet-5-5 ~4-6 $ | claude-haiku-5-5 ~0,5 $
GEN_EFFORT = "low"                   # düşünme derinliği: "low" en ucuz/hızlı; None: API varsayılanı (Opus 5.5'te "medium", çok daha pahalı)
THINKING_MODE = "minimal"            # "minimal": Sonnet 5.5 -> between_tools, Haiku 5.5 -> disabled, Opus/Fable -> yalnız effort; "default": adaptif düşünme açık (pahalı)
GEN_RECORDS_PER_CALL = 20            # bir API çağrısında üretilen kayıt sayısı (üretim imzasına girer; değiştirirseniz önceki parçalar kullanılmaz)
GEN_WORKERS = 4                      # eş zamanlı çağrı sayısı (hız limiti aşılırsa otomatik bekler)
GEN_MAX_TOKENS_PER_RECORD = 700      # kayıt başına çıktı üst sınırı (düşünme payı dâhil; kullanılmayan token ücretlendirilmez)
BUDGET_GUARD_USD = 30.0              # bu oturumda tahmini toplam API maliyeti bunu aşarsa yeni çağrı yapılmaz (None: kapalı)
GEN_TEMPERATURE = None               # None: API varsayılanı. Güncel SDK (>=1.15) ve yeni modeller sıcaklık parametresi almaz; eski SDK'da sayı verilirse kullanılır
GEN_OVERSAMPLE = 1.15                # tekrar/kalite elemesi için fazladan üretim payı
IMPLICIT_RATIO = 0.30                # ihtiyacı adıyla söylemeyip durumdan çıkarılmasını gerektiren paylaşım oranı
MIN_PER_NEED = 12                    # her ihtiyacın etiket olarak en az kaç kez geçeceği
DEDUP_COSINE_THRESHOLD = 0.92        # bu benzerliğin üzerindeki paylaşımlar tekrar sayılır ve atılır
FORCE_REGENERATE = False             # True: Drive'daki önceki üretim parçalarını yok sayıp yeniden üret (ücret!)
RUN_LABEL_QC = True                  # üretilen her kaydı BAĞIMSIZ bir modelle denetle (etiketler metinde var mı, fazladan ihtiyaç var mı)
QC_MODEL = "claude-sonnet-5-5"       # kalite kontrol modeli (üretim modelinden farklı olmalı); tahmini ~1,5-2 $
QC_EFFORT = "low"
QC_EXCLUDE_FAILED = False            # True: denetimden geçemeyen kayıtları veri setinden çıkar (ilçe kotaları bozulabilir); False: yalnız işaretle

# --- İhtiyaç listesi -----------------------------------------------------------
FIX_NEED_TRANSLATIONS = True         # True: hatalı Google Translate karşılıkları düzeltilir ("voluntarily" vb.)
                                     # False: makine çevirisi karşılıkları düzeltilmeden kullanılır (64 benzersiz etiket)
PREPROCESS_NEEDS_LIKE_MESSAGES = True  # True: lemma/gövde deneylerinde ihtiyaç adları da aynı işlemden geçer
                                       # False: ihtiyaç adları ham bırakılır

# --- Dil temsil modelleri ---------------------------------------------------------
MODELS_TO_RUN = ["BERT", "ALBERT", "RoBERTa", "GloVe", "Word2Vec", "USE", "FastText"]  # makaledeki 7 model
EXTRA_MODELS = ["SBERT-mpnet", "BERT-raw", "ALBERT-paraphrase"]  # ek modeller: güncel SBERT + ham/ince ayarlı eş çiftler; kapatmak için []
TURKISH_MODELS = ["SBERT-çokdilli (TR)", "BERTurk-STS (TR)"]      # Türkçe metin + Türkçe ihtiyaç listesi (yalnız düz metin); kapatmak için []
BASELINES = ["Temel: TF-IDF", "Temel: Rastgele", "Temel: Çoğunluk"]  # şans düzeyi ve sözcüksel temel çizgi
TEXT_TYPES = ["plain", "lemma", "stem"]   # düz / lemmalanmış / gövdelenmiş
LATENCY_ON_CPU = True                # tek mesaj gecikmesi tüm yerel modeller için CPU'da ölçülür (eşit koşul); toplu GPU süresi ayrıca verilir
SIGNIFICANCE_TESTS = True            # en iyi modele karşı eşleştirilmiş McNemar + bootstrap fark testleri (Holm düzeltmeli)
GLOVE_VARIANT = "840B"               # "840B" (makaledeki Common Crawl 840B) veya "wiki300" (küçük, hızlı yedek)
CACHE_MODELS_TO_DRIVE = False        # True: indirilen büyük modelleri Drive'a da kopyala (Drive kotası ~15 GB ister)
MODEL_DIR_CANDIDATES = {             # kullanıcının Drive'ında zaten bulunan model dosyaları (ilk bulunan kullanılır)
    "Word2Vec": ["MyDrive/Colab Notebooks/NLP Modeller/Word2Vec Files/word2vec-google-news-300.bin",
                 "MyDrive/NLP_Disaster/models/word2vec-google-news-300.bin"],
    "GloVe":    ["MyDrive/NLP_Disaster/models/glove.840B.300d.kv",
                 "MyDrive/Colab Notebooks/NLP Modeller/GloVe File/glove.840B.300d.txt",
                 "MyDrive/Colab Notebooks/NLP Modeller/GloVe File/glove.840B.300d-001.txt"],
    "FastText": ["MyDrive/Colab Notebooks/NLP Modeller/FastText Files/cc.en.300.bin",
                 "MyDrive/NLP_Disaster/models/cc.en.300.bin"],
    "USE":      ["MyDrive/Colab Notebooks/NLP Modeller/USE Files",
                 "MyDrive/NLP_Disaster/models/USE"],
}
BOOTSTRAP_N = 1000                   # güven aralığı için yeniden örnekleme sayısı

# --- BDM (büyük dil modeli) değerlendirmesi, Tablo 5 ----------------------------------
LLM_EVAL_MODELS = [                  # (sağlayıcı, model). openai/google satırları anahtar yoksa atlanır
    ("anthropic", "claude-opus-5-5"),
    ("anthropic", "claude-sonnet-5-5"),
    ("anthropic", "claude-haiku-5-5"),
    ("openai", "gpt-5-mini"),
    ("google", "gemini-3.8-flash"),   # gemini-2.5-flash yeni kullanıcılara kapatıldı (API 404)
]
LLM_EVAL_MESSAGES_PER_CALL = 10      # 1: her paylaşım ayrı istekte gönderilir (daha yavaş)
LLM_EVAL_EFFORT = "low"              # BDM değerlendirmesinde düşünme derinliği ("low": en düşük ve en ucuz düzey)
LLM_EVAL_WORKERS = 4                 # Anthropic için eş zamanlı istek sayısı
LLM_EVAL_WORKERS_BY_PROVIDER = {"openai": 2, "google": 2}   # diğer sağlayıcılar (düşük kademe hız sınırları için düşük tutuldu)
LLM_EVAL_RPM_BY_PROVIDER = {"openai": None, "google": None}  # dakikadaki istek üst sınırı; None: uyarlanabilir (429 geldikçe bekler).
                                                             # Düşük kademe (yeni hesap) ise örn. "openai": 8 yazın.
LLM_MIN_COVERAGE = 0.98              # bir BDM'nin mesajların en az bu oranı için yanıtı yoksa (kota/limit) Tablo 5'e ALINMAZ,
                                     # "eksik" olarak ayrı dosyaya yazılır; yeniden çalıştırınca önbellekten kaldığı yerden devam eder
LLM_EVAL_MAX_MESSAGES = None         # None: tüm mesajlar; örn. 500: rastgele alt küme (hızlı/ucuz)
RUN_LATENCY_BENCHMARK = True         # tüm BDM'ler için aynı paylaşımlarla, sıralı çağrılarla gecikme ve maliyet ölçümü (Tablo 6b)
BENCH_BATCH_REQUESTS = 15            # toplu kip: 15 istek x 10 paylaşım (ms/paylaşım ve USD/1.000 paylaşım)
BENCH_SINGLE_REQUESTS = 15           # tekil kip: 15 istek x 1 paylaşım (tek paylaşım yanıt süresi)
BENCH_FORCE = False                  # True: önceki ölçümleri yok sayıp yeniden ölç

# --- Genel ---------------------------------------------------------------------------
SEED = 42
AUTO_DOWNLOAD_ZIP = True             # bitince sonuç zip'ini tarayıcıya indir
PRICES_USD_PER_MTOK = {              # maliyet tahmini için (giriş, çıkış, önbellek okuma, önbellek yazma)
    "claude-opus-5-5":   (4.0, 20.0, 0.20, 5.0),
    "claude-sonnet-5-5": (2.0, 10.0, 0.10, 2.5),
    "claude-haiku-5-5":  (0.10, 0.50, 0.01, 0.125),
    "gpt-5-mini":        (0.25, 2.00, 0.025, 0.0),    # OpenAI liste fiyatı (Ekim 2026); önbellek yazma ücretsiz
    "gemini-3.8-flash":  (0.75, 3.75, 0.075, 0.0),    # Google liste fiyatı (Ekim 2026; 31 Aralık 2026'ya kadar geçerli; ai.google.dev/gemini-api/docs/pricing)
}

In [ ]:
# =============================================================================
# R0. REVİZYON AYARLARI (makalenin revize sürümüne eklenen deneyler)
# -----------------------------------------------------------------------------
# Yukarıdaki hücre makaledeki çalıştırmanın ayarlarıdır; DEĞİŞTİRMEYİN (istemler ve imzalar onlara bağlı).
# Bu defter yeni sentetik veri ÜRETMEZ: makaledeki 2.000 paylaşımı Drive'dan ya da GitHub deposundan okur.
# Yalnızca bu hücredeki ayarları değiştirmeniz yeterlidir.
# =============================================================================
import os

SMOKE_TEST = False           # True: API ve büyük modeller olmadan sahte çıktılarla akış denemesi (ücretsiz, birkaç dakika)
RUN_TAG = "revizyon_smoke" if SMOKE_TEST else "revizyon"   # çıktılar: MyDrive/NLP_Disaster/runs/revizyon
BUDGET_GUARD_USD = 40.0      # bu oturumdaki TÜM API çağrılarının (Anthropic + OpenAI + Google) tahmini toplam üst sınırı

PAPER_RUN_TAG = "20261016_1116"         # makalede raporlanan çalıştırma
PAPER_GEN_SIGNATURE = "a8bfe68ee1"      # makaledeki veri setinin üretim imzası
REPO_RAW = "https://raw.githubusercontent.com/kurbankotan/disaster-needs-assessment-nlp/main"
REPO_LOCAL_DIR = os.environ.get("NLP_REPO_DIR")   # Colab dışı deneme için deponun yerel kopyası (isteğe bağlı)

# --- Hangi deneyler çalışsın (False: atla) -------------------------------------------------
REV = {
    "yeni_kodlayicilar": True,     # mE5, BGE-M3, Qwen3-Embedding (İngilizce ve Türkçe; ihtiyaç adı ve ad + tanım)
    "ince_ayar_ablasyonu": True,   # RoBERTa ham ve güncel eşi (ham / NLI / karşıtsal ince ayar karşılaştırması)
    "tanimli_ihtiyaclar": True,    # ihtiyaç adına kısa tanım eklenmesi (makaledeki modellerle de)
    "setfit": True,                # az örnekli öğrenme: SetFit ve dondurulmuş gömme + lojistik regresyon
    "ner_ilce": True,              # ilçe/konum çıkarımı: alt dize, bulanık sözlük, Türkçe NER; bozulmuş ilçe adları
    "bdm_tekrar": True,            # makale protokolüyle BDM tekrarları (çalıştırmalar arası değişkenlik)
    "bdm_gpt210": True,            # GPT-5 mini'nin 210 paylaşımını minimal düşünmeyle yeniden sor
    "bdm_turkce": True,            # Türkçe paylaşım + Türkçe istem + Türkçe ihtiyaç listesi
    "bdm_coklu": True,             # en fazla 3 sıralı ihtiyaç (çok etiketli ölçütler), İngilizce ve Türkçe
    "bdm_az_ornekli": True,        # her ihtiyaçtan bir örnek içeren (66 örnekli) istem, test bölümünde
    "alternatif_uretici": True,    # aynı planlarla Gemini ve GPT-5 mini'ye yazdırılan alt küme + çapraz denetim
    "gercek_veri": True,           # etiketlenmiş gerçek paylaşımlar (altın dosya Drive'da varsa)
    "humaid_insan": True,          # HumAID'in insan (kitle) etiketleriyle somut ihtiyaç/istek ayrımı (dış geçerlilik)
}

# --- BDM'ler (makaledekiyle aynı protokol: 10 paylaşım/istek, en düşük düşünme) ------------------
REV_LLMS = [
    ("anthropic", "claude-opus-5-5"),
    ("anthropic", "claude-sonnet-5-5"),
    ("anthropic", "claude-haiku-5-5"),
    ("openai", "gpt-5-mini"),
    ("google", "gemini-3.8-flash"),
]
BDM_TEKRAR = 2                                    # makaledeki çalıştırmaya EK tekrar sayısı
LLM_EVAL_RPM_BY_PROVIDER = {"openai": None, "google": None}   # dakikadaki istek üst sınırı; Gemini ücretsiz kademedeyse "google": 9

# --- Yeni kodlayıcılar (Hugging Face; ilk çalıştırmada indirilir) ----------------------------------
NEW_ENCODERS = {
    "mE5-large-instruct": {"name": "intfloat/multilingual-e5-large-instruct", "query_prompt": "Instruct: {task}\nQuery: "},
    "BGE-M3":             {"name": "BAAI/bge-m3", "query_prompt": ""},
    "Qwen3-Emb-0.6B":     {"name": "Qwen/Qwen3-Embedding-0.6B", "query_prompt": "Instruct: {task}\nQuery:"},
    "Qwen3-Emb-4B":       {"name": "Qwen/Qwen3-Embedding-4B", "query_prompt": "Instruct: {task}\nQuery:", "fp16": True},
}
EMB_TASK = "Given a social media post written after an earthquake, retrieve the relief item that the author needs most urgently"
ABLATION_MODELS = {"RoBERTa-raw": "roberta-base", "RoBERTa-güncel": "all-distilroberta-v1"}
CPU_LATENCY_MAX_PARAMS = 1.0e9    # bu boyutun altındaki yeni kodlayıcılar için işlemcide tek paylaşım gecikmesi de ölçülür

# --- Az örnekli öğrenme -----------------------------------------------------------------------------
FEW_K = 8                                         # her ihtiyaç için eğitim örneği (66 x 8 = 528 paylaşım; geri kalanı test)
SETFIT_BASE = {"en": "sentence-transformers/all-mpnet-base-v2",
               "tr": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"}
SETFIT_SEEDS = [42, 43, 44]
SETFIT_ITERATIONS = 20                            # SetFit: her örnek için yineleme başına bir pozitif ve bir negatif çift
SETFIT_EPOCHS = 1
SETFIT_BATCH = 32
SETFIT_LR = 2e-5
SETFIT_FULL_FOR_REAL = True                       # gerçek veri için ek model: 2.000 sentetik paylaşımın tamamıyla eğitim
SETFIT_FULL_ITERATIONS = 5

# --- Alternatif üreticiler (öz-tercih yanlılığı denetimi) ---------------------------------------------
ALT_PER_DISTRICT = 26                             # makaledeki veriden ilçe başına seçilen plan sayısı (39 x 26 = 1.014)
ALT_GENERATORS = [
    {"ad": "gemini", "uretici": ("google", "gemini-3.8-flash"), "denetci": ("openai", "gpt-5-mini")},
    {"ad": "gpt",    "uretici": ("openai", "gpt-5-mini"),       "denetci": ("google", "gemini-3.8-flash")},
]
ALT_CLAUDE_QC = ("openai", "gpt-5-mini")          # aynı alt kümedeki Claude metinlerinin çapraz denetimi

# --- Gerçek veri ---------------------------------------------------------------------------------------
REAL_DIR_NAME = "gercek_veri"                     # MyDrive/NLP_Disaster/gercek_veri
REAL_GOLD_FILE = "gercek_veri_altin.csv"          # uzlaştırılmış altın etiketler (metinler yalnız sizin Drive'ınızda kalır)
GERCEK_ISTEM_EK = ""                              # uzlaştırmada netleşen kod kitabı kuralları (açık küme istemine eklenir)
MT_MODEL = "Helsinki-NLP/opus-mt-tr-en"           # Türkçe tweetlerin yalnız İngilizce modeller için makine çevirisi
NER_MODELS = ["savasy/bert-base-turkish-ner-cased", "akdeniz27/bert-base-turkish-cased-ner"]
FUZZY_MIN = 88                                    # bulanık ilçe eşlemesinde en düşük benzerlik (0-100)
HUMAID_DIR_NAME = "humaid"                        # MyDrive/NLP_Disaster/humaid: train/dev/test (.json, .jsonl ya da .tsv); yoksa resmî zip indirilir
HUMAID_NEG_RATIO = 2                              # her "requests_or_urgent_needs" tweetine karşı örneklenen diğer sınıf tweeti

In [ ]:
# =============================================================================
# 1. KURULUM: paketler, Drive, klasörler, günlük, çalışma zamanı bilgisi
# =============================================================================
import os, sys, re, io, json, time, math, gc, random, hashlib, shutil, subprocess, warnings, datetime, zipfile, urllib.request
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings("ignore")

def pip_install(pkgs, upgrade=False):
    """Sessiz pip kurulumu (hata olursa akışı durdurmaz, uyarı verir). Colab'ın hazır numpy/pandas/torch sürümlerine dokunmaz."""
    if os.environ.get("NLP_SKIP_PIP"):
        return
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + (["--upgrade"] if upgrade else []) + pkgs
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("UYARI: pip kurulumu başarısız:", pkgs, "\n", r.stderr[-800:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

pip_install(["anthropic>=1.0"], upgrade=True)          # output_config (yapılandırılmış JSON) için güncel SDK gerekir
pip_install(["folium", "nltk", "inflect", "openpyxl", "tqdm", "psutil"])
if not SMOKE_TEST:
    pip_install(["sentence-transformers", "gensim>=4.3.3", "tensorflow_hub"])

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from tqdm.auto import tqdm
import nltk
os.environ.setdefault("NLTK_ALLOW_PROXIED_URLOPEN", "1")   # vekil sunucu arkasındaki ortamlarda NLTK indirmesine izin ver
for res in ["wordnet", "omw-1.4", "stopwords", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(res, quiet=True)
    except Exception as e:
        print("NLTK indirme uyarısı:", res, e)

# --- Rastgelelik ---------------------------------------------------------------
random.seed(SEED); np.random.seed(SEED)
RNG = random.Random(SEED)
NP_RNG = np.random.default_rng(SEED)

# --- Drive ve klasörler --------------------------------------------------------
if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive"
    BASE_DIR = os.path.join(DRIVE_ROOT, "MyDrive", DRIVE_FOLDER_NAME)
    LOCAL_SCRATCH = "/content/nlp_scratch"
else:
    DRIVE_ROOT = os.path.abspath("./drive_local")           # Colab dışı deneme için yerel klasör
    BASE_DIR = os.path.join(DRIVE_ROOT, "MyDrive", DRIVE_FOLDER_NAME)
    LOCAL_SCRATCH = os.path.abspath("./nlp_scratch")

if RUN_TAG is None:
    RUN_TAG = datetime.datetime.now().strftime("%Y%m%d_%H%M") + ("_smoke" if SMOKE_TEST else "")
RUN_DIR   = os.path.join(BASE_DIR, "runs", RUN_TAG)
DATA_DIR  = os.path.join(BASE_DIR, "data")
MODEL_DIR = os.path.join(BASE_DIR, "models")
DIRS = {
    "data": DATA_DIR, "models": MODEL_DIR, "run": RUN_DIR,
    "results": os.path.join(RUN_DIR, "results"), "figures": os.path.join(RUN_DIR, "figures"),
    "maps": os.path.join(RUN_DIR, "maps"), "logs": os.path.join(RUN_DIR, "logs"),
    "preds": os.path.join(RUN_DIR, "results", "tahminler"), "llm_cache": os.path.join(DATA_DIR, "bdm_cache"),
    "scratch": LOCAL_SCRATCH,
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

# --- Günlük --------------------------------------------------------------------
LOG_PATH = os.path.join(DIRS["logs"], "log.txt")
def log(*args):
    msg = " ".join(str(a) for a in args)
    stamp = datetime.datetime.now().strftime("%H:%M:%S")
    line = f"[{stamp}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# --- Çalışma zamanı bilgisi -----------------------------------------------------
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "yok"
except Exception:
    torch = None; DEVICE = "cpu"; GPU_NAME = "yok (torch bulunamadı)"
try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / 1e9
except Exception:
    RAM_GB = float("nan")
log(f"Colab: {IN_COLAB} | GPU: {GPU_NAME} | RAM: {RAM_GB:.0f} GB | Python {sys.version.split()[0]}")
log(f"Çıktı klasörü: {RUN_DIR}")
if not SMOKE_TEST and RAM_GB < 25:
    log("UYARI: RAM 25 GB'ın altında. FastText (cc.en.300.bin, ~7 GB) ve GloVe 840B yüklemesi çökebilir. "
        "Runtime > Change runtime type > High-RAM önerilir.")

# --- API anahtarları -----------------------------------------------------------
def get_secret(name):
    val = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
        except Exception:
            val = None
    return val or os.environ.get(name)

ANTHROPIC_API_KEY = get_secret("ANTHROPIC_API_KEY")
OPENAI_API_KEY    = get_secret("OPENAI_API_KEY")
GOOGLE_API_KEY    = get_secret("GOOGLE_API_KEY")
if not SMOKE_TEST and not ANTHROPIC_API_KEY:
    raise RuntimeError("ANTHROPIC_API_KEY bulunamadı. Colab'da sol menü > Anahtar simgesi > "
                       "ANTHROPIC_API_KEY ekleyip 'Notebook access' açın.")
log("Anahtarlar: Anthropic=%s | OpenAI=%s | Google=%s" % tuple("var" if k else "yok" for k in (ANTHROPIC_API_KEY, OPENAI_API_KEY, GOOGLE_API_KEY)))

# --- Yardımcılar ------------------------------------------------------------------
def tr_lower(s):
    """Türkçe'ye uygun küçük harf (İ→i, I→ı)."""
    return str(s).replace("İ", "i").replace("I", "ı").lower()

def fmt_tr(x, nd=2):
    """Ondalık virgüllü sayı biçimi (makale Türkçe)."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    return f"{x:.{nd}f}".replace(".", ",")

def save_table(df, name, index=False, float_nd=3):
    """Tabloyu CSV + XLSX + Markdown olarak kaydeder."""
    p = os.path.join(DIRS["results"], name)
    df.to_csv(p + ".csv", index=index, encoding="utf-8-sig")
    try:
        df.to_excel(p + ".xlsx", index=index)
    except Exception as e:
        log("XLSX kaydedilemedi:", name, e)
    with open(p + ".md", "w", encoding="utf-8") as f:
        f.write(df_to_markdown(df, index=index, float_nd=float_nd))
    return p

def df_to_markdown(df, index=False, float_nd=3):
    d = df.reset_index() if index else df
    cols = list(d.columns)
    def cell(v):
        if isinstance(v, float):
            return fmt_tr(v, float_nd)
        return str(v)
    lines = ["| " + " | ".join(str(c) for c in cols) + " |", "|" + "---|" * len(cols)]
    for _, row in d.iterrows():
        lines.append("| " + " | ".join(cell(row[c]) for c in cols) + " |")
    return "\n".join(lines) + "\n"

plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
TR_DECIMAL = FuncFormatter(lambda v, pos: f"{v:.2f}".replace(".", ","))
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000", "#999999"]  # Okabe-Ito

CONFIG = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, list, dict, tuple, type(None)))
          and k not in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_API_KEY")
          and not re.search(r"(KEY|TOKEN|SECRET|PASSWORD)", k)}   # anahtar içerebilecek hiçbir değişken kayda yazılmaz
with open(os.path.join(DIRS["logs"], "config.json"), "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, ensure_ascii=False, indent=2, default=str)
log("Kurulum tamam.")

In [ ]:
# =============================================================================
# R0b. EK PAKETLER VE MAKALEDEKİ ÇALIŞTIRMANIN DOSYALARINA ERİŞİM
# =============================================================================
import urllib.parse
pip_install(["openai>=1.40", "google-genai>=1.0", "rapidfuzz"])
if not SMOKE_TEST:
    # Qwen3-Embedding için transformers >= 4.51 gerekir; yüklü sürüm yeterliyse pip dokunmaz
    pip_install(["sentence-transformers>=3.0", "transformers>=4.51", "sentencepiece", "sacremoses"])

PAPER_RUN_DIR = os.path.join(BASE_DIR, "runs", PAPER_RUN_TAG)
PAPER_CACHE = os.path.join(DATA_DIR, "makale_dosyalari"); os.makedirs(PAPER_CACHE, exist_ok=True)

def paper_file(rel, required=True):
    """Makaledeki çalıştırmanın bir dosyası: önce Drive'daki çalıştırma klasörü, sonra yerel depo kopyası,
    en son GitHub deposundan indirme (DATA_DIR/makale_dosyalari altında saklanır). Türkçe karakterli adlar için
    hem birleşik (NFC) hem ayrık (NFD) yazım denenir (depoda bazı adlar NFD'dir). Bulunamazsa None ya da hata."""
    import unicodedata
    variants = list(dict.fromkeys([rel, unicodedata.normalize("NFC", rel), unicodedata.normalize("NFD", rel)]))
    cands = []
    for r in variants:
        cands.append(os.path.join(PAPER_RUN_DIR, r))
        if r.startswith("data/"):
            cands.append(os.path.join(DATA_DIR, r[len("data/"):]))
        if REPO_LOCAL_DIR:
            cands.append(os.path.join(REPO_LOCAL_DIR, "runs", PAPER_RUN_TAG, r))
    for p in cands:
        if os.path.exists(p):
            return p
    dest = os.path.join(PAPER_CACHE, unicodedata.normalize("NFC", rel))
    if os.path.exists(dest):
        return dest
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    last = None
    for r in variants:
        try:
            urllib.request.urlretrieve(f"{REPO_RAW}/runs/{PAPER_RUN_TAG}/" + urllib.parse.quote(r), dest + ".part")
            os.replace(dest + ".part", dest)
            log(f"GitHub deposundan indirildi: {rel}")
            return dest
        except Exception as e:
            last = e
    if required:
        raise RuntimeError(f"Makale dosyası bulunamadı: {rel} ({last})")
    return None

log(f"Makaledeki çalıştırma klasörü: {PAPER_RUN_DIR} ({'Drive’da var' if os.path.isdir(PAPER_RUN_DIR) else 'Drive’da yok, GitHub deposu kullanılacak'})")

In [ ]:
# =============================================================================
# 2. İHTİYAÇ LİSTESİ VE İLÇELER
# =============================================================================
# 66 maddelik ihtiyaç listesi (sıra korunmuştur) ve makine çevirisiyle (Google Translate) elde edilen İngilizce karşılıklar.
NEEDS_TR = [
    "içme suyu", "bebek maması", "biberon", "ıslak mendil", "peçete", "gıda", "fener", "elektrikli ısıtıcı", "soba", "odun",
    "pil", "battaniye", "pilli radyo", "diş fırçası", "diş macunu", "hijyenik ped", "bebek bezi", "sabun", "yedek giysi", "mont",
    "maske", "nakit para", "konserve", "sıcak yemek", "ilaç", "ağrı kesici", "antibiyotik", "soğuk algınlığı ilacı", "ateş düşürücü", "yarabandı",
    "bandaj", "alerji ilaçları", "astım ilacı", "insülin", "seyyar tuvalet", "çadır", "uyku tulumu", "arama kurtarma ekibi", "sağlık ekibi", "psikolojik destek ekibi",
    "güvenlik güçleri", "jenaratör", "powerbank", "glutesiz yiyecek", "tuzsuz-şekersiz yiyecek", "gönüllü", "kuru gıda", "ekmek", "ilk yardım kiti", "kap kacak",
    "engelli ekipmanı", "kan", "aydınlatıcı", "hayvan maması", "kalp ilacı", "konteynır", "temizlik ürünü", "hırdavat", "çocuk oyun alanı", "oksijen cihazı",
    "elektrik saha ekibi", "tercüman", "kan toplama merkezi", "can yeleği", "mutfak eşyası", "el feneri",
]
NEEDS_EN_ORIG = [
    "drinking water", "baby food", "bottle", "baby wipes", "napkin", "food", "torch", "electric heater", "stove", "wood",
    "battery", "blanket", "battery radio", "toothbrush", "toothpaste", "sanitary pad", "diaper", "soap", "spare clothes", "coat",
    "mask", "cash money", "canned", "hot meal", "medicine", "painkiller", "antibiotic", "cold medicine", "antipyretic", "bandage",
    "bandage", "allergy medications", "asthma medicine", "insulin", "portable toilet", "tent", "sleeping bag", "search and rescue team", "medical team", "psychological support team",
    "security forces", "generator", "power bank", "gluten free food", "salt-free-sugar-free food", "voluntarily", "dry food", "bread", "first aid kit", "utensils",
    "disabled equipment", "blood", "enlightening", "pet food", "heart medicine", "container", "cleaning product", "hardware", "children's playground", "oxygen device",
    "electrical field team", "interpreter", "blood collection center", "life jacket", "kitchenware", "torch",
]
assert len(NEEDS_TR) == len(NEEDS_EN_ORIG) == 66

# Makine çevirisindeki hatalı / yinelenen karşılıklar için düzeltmeler (FIX_NEED_TRANSLATIONS=True iken uygulanır)
NEEDS_EN_FIX = {
    "bebek maması": "baby formula", "biberon": "baby bottle", "ıslak mendil": "wet wipes", "fener": "lantern",
    "el feneri": "flashlight", "soba": "heating stove", "odun": "firewood", "pil": "batteries", "pilli radyo": "battery-powered radio",
    "nakit para": "cash", "konserve": "canned food", "ateş düşürücü": "fever reducer", "yarabandı": "adhesive bandage",
    "alerji ilaçları": "allergy medication", "tuzsuz-şekersiz yiyecek": "salt-free and sugar-free food", "gönüllü": "volunteers",
    "kap kacak": "pots and pans", "engelli ekipmanı": "equipment for disabled people", "aydınlatıcı": "lighting equipment",
    "konteynır": "container housing", "hırdavat": "hardware tools", "oksijen cihazı": "oxygen concentrator",
}
NEED_TABLE = pd.DataFrame({"tr": NEEDS_TR, "en_orig": NEEDS_EN_ORIG})
NEED_TABLE["en_fixed"] = [NEEDS_EN_FIX.get(tr, en) for tr, en in zip(NEEDS_TR, NEEDS_EN_ORIG)]
NEED_TABLE["en"] = NEED_TABLE["en_fixed"] if FIX_NEED_TRANSLATIONS else NEED_TABLE["en_orig"]
TR2EN = dict(zip(NEED_TABLE["tr"], NEED_TABLE["en"]))
EN_LABELS = list(dict.fromkeys(NEED_TABLE["en"]))          # benzersiz İngilizce etiketler (model çıktı uzayı)
EN2TR = {}
for tr, en in zip(NEED_TABLE["tr"], NEED_TABLE["en"]):
    EN2TR.setdefault(en, tr)                                 # yinelenen etikette ilk Türkçe karşılık
log(f"İhtiyaç listesi: {len(NEEDS_TR)} Türkçe madde, {len(EN_LABELS)} benzersiz İngilizce etiket "
    f"(çeviri düzeltme: {'açık' if FIX_NEED_TRANSLATIONS else 'kapalı'})")
if FIX_NEED_TRANSLATIONS:
    changed = NEED_TABLE[NEED_TABLE.en_orig != NEED_TABLE.en_fixed][["tr", "en_orig", "en_fixed"]]
    log("Düzeltilen çeviriler:\n" + changed.to_string(index=False))
NEED_TABLE.to_csv(os.path.join(DIRS["results"], "ihtiyac_listesi.csv"), index=False, encoding="utf-8-sig")

# Yazım farkları: etiketlerde ve üretim çıktılarında görülebilen farklı yazımlar liste adına eşlenir
NEED_ALIASES_TR = {"jeneratör": "jenaratör", "jenerator": "jenaratör", "glutensiz yiyecek": "glutesiz yiyecek", "glütensiz yiyecek": "glutesiz yiyecek",
                   "power bank": "powerbank", "konteyner": "konteynır", "yara bandı": "yarabandı", "hijyenik bağ": "hijyenik ped"}
NEED_DISPLAY_TR = {"jenaratör": "jeneratör", "glutesiz yiyecek": "glutensiz yiyecek"}   # tablo/rapor gösterimi için doğru yazım
def canon_need_tr(x):
    """Türkçe ihtiyaç adını liste biçimine getirir (küçük harf, boşluk, yazım eşlemesi); listede yoksa None."""
    if not isinstance(x, str):
        return None
    k = " ".join(tr_lower(x).split())
    k = NEED_ALIASES_TR.get(k, k)
    return k if k in TR2EN else None

def get_english_need(tr_need):
    return TR2EN.get(tr_lower(tr_need).strip(), None)

# --- 39 ilçe ve merkez koordinatları (Nominatim ile belirlenmiştir) ------------
DISTRICTS = [
    "Adalar", "Arnavutköy", "Ataşehir", "Avcılar", "Bağcılar", "Bahçelievler", "Bakırköy", "Başakşehir", "Bayrampaşa", "Beşiktaş",
    "Beykoz", "Beylikdüzü", "Beyoğlu", "Büyükçekmece", "Çatalca", "Çekmeköy", "Esenler", "Esenyurt", "Eyüpsultan", "Fatih",
    "Gaziosmanpaşa", "Güngören", "Kadıköy", "Kağıthane", "Kartal", "Küçükçekmece", "Maltepe", "Pendik", "Sancaktepe", "Sarıyer",
    "Silivri", "Sultanbeyli", "Sultangazi", "Şile", "Şişli", "Tuzla", "Ümraniye", "Üsküdar", "Zeytinburnu",
]
DISTRICT_COORDS = {
    "Adalar": (40.8741659, 29.1293251), "Arnavutköy": (41.184471, 28.7412446), "Ataşehir": (40.9929379, 29.1135187),
    "Avcılar": (40.9799389, 28.7216689), "Bağcılar": (41.034547, 28.8567558), "Bahçelievler": (41.0030495, 28.8657701),
    "Bakırköy": (40.9782585, 28.8744461), "Başakşehir": (41.1075794, 28.7950711), "Bayrampaşa": (41.0345549, 28.9118417),
    "Beşiktaş": (41.0428465, 29.0075283), "Beykoz": (41.1343001, 29.0920378), "Beylikdüzü": (41.0037541, 28.6372583),
    "Beyoğlu": (41.0284233, 28.9736808), "Büyükçekmece": (41.021654, 28.579757), "Çatalca": (41.1436804, 28.4605154),
    "Çekmeköy": (41.0351579, 29.1739149), "Esenler": (41.0376175, 28.8824519), "Esenyurt": (41.0342862, 28.6801113),
    "Eyüpsultan": (41.0478358, 28.9327383), "Fatih": (41.0192846, 28.9479296), "Gaziosmanpaşa": (41.0578305, 28.9122452),
    "Güngören": (41.0252832, 28.8726498), "Kadıköy": (40.9912955, 29.0245631), "Kağıthane": (41.0796544, 28.9731198),
    "Kartal": (40.88858, 29.1856536), "Küçükçekmece": (40.9918737, 28.7711956), "Maltepe": (40.9247539, 29.1310782),
    "Pendik": (40.8768715, 29.2349672), "Sancaktepe": (40.9905196, 29.2288624), "Sarıyer": (41.1685803, 29.0572623),
    "Silivri": (41.0742476, 28.2481709), "Sultanbeyli": (40.9670242, 29.2671314), "Sultangazi": (41.1043344, 28.8614367),
    "Şile": (41.1744067, 29.6125216), "Şişli": (41.0637891, 28.9831642), "Tuzla": (40.8161732, 29.3034194),
    "Ümraniye": (41.0256362, 29.0963049), "Üsküdar": (41.0265498, 29.0151321), "Zeytinburnu": (40.9898653, 28.9037467),
}
assert len(DISTRICTS) == 39 and all(d in DISTRICT_COORDS for d in DISTRICTS)
DISTRICTS_LOWER = {tr_lower(d): d for d in DISTRICTS}

def detect_districts(text):
    """Alt dize eşleşmesiyle ilçe adı tespiti; Türkçe büyük/küçük harf kurallarına uygun ve harf duyarsız."""
    t = tr_lower(text)
    return [d for d in DISTRICTS if tr_lower(d) in t]

# --- İlçe sınırları (statik haritalar için) -------------------------------------------------------
# Kaynak: geoBoundaries (gbOpen, TUR ADM2; OpenStreetMap türevi, ODbL / CC BY 4.0, kullanım koşulu: kaynak belirtme).
# Dosya bir kez indirilip Drive'da saklanır (data/geo); indirme başarısız olursa haritalar nokta işaretleriyle çizilir.
import unicodedata
GEO_DIR = os.path.join(DATA_DIR, "geo"); os.makedirs(GEO_DIR, exist_ok=True)
GEO_PATH = os.path.join(GEO_DIR, "istanbul_ilceler.geojson")
GEO_SOURCE_NOTE = "geoBoundaries gbOpen TUR ADM2 (OpenStreetMap verisi, ODbL / CC BY 4.0)"

def _geo_norm(s):
    s = str(s).replace("İ", "i").replace("I", "ı").lower()
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    return s.replace("ı", "i").replace("ş", "s").replace("ğ", "g").replace("ç", "c").replace("ö", "o").replace("ü", "u").replace(" ", "")

_GEO_ALIASES = {"eyup": "Eyüpsultan", "eyupsultan": "Eyüpsultan"}
_IST_BBOX = (27.95, 40.70, 29.95, 41.62)   # lon_min, lat_min, lon_max, lat_max

def _geom_lon_lat(geom):
    """Poligonların tüm (lon, lat) noktalarını düz liste olarak verir."""
    pts = []
    if geom["type"] == "Polygon":
        for ring in geom["coordinates"]:
            pts.extend(ring)
    elif geom["type"] == "MultiPolygon":
        for poly in geom["coordinates"]:
            for ring in poly:
                pts.extend(ring)
    return pts

def fetch_istanbul_geojson(force=False):
    """İstanbul'un 39 ilçesinin sınırlarını geoBoundaries'ten alır ve GEO_PATH'e yazar; yol ya da None döner."""
    if os.path.exists(GEO_PATH) and not force:
        return GEO_PATH
    try:
        import requests
        meta = requests.get("https://www.geoboundaries.org/api/current/gbOpen/TUR/ADM2/", timeout=60).json()
        url = meta.get("simplifiedGeometryGeoJSON") or meta.get("gjDownloadURL")
        gj = requests.get(url, timeout=300).json()
        wanted = {_geo_norm(d): d for d in DISTRICTS}
        feats = []
        for f in gj["features"]:
            name = f["properties"].get("shapeName", "")
            d = wanted.get(_geo_norm(name)) or _GEO_ALIASES.get(_geo_norm(name))
            if not d:
                continue
            pts = _geom_lon_lat(f["geometry"])
            if not pts:
                continue
            lon = sum(p[0] for p in pts) / len(pts); lat = sum(p[1] for p in pts) / len(pts)
            if not (_IST_BBOX[0] <= lon <= _IST_BBOX[2] and _IST_BBOX[1] <= lat <= _IST_BBOX[3]):
                continue   # aynı adlı başka ilçe (İstanbul dışı)
            feats.append({"type": "Feature", "properties": {"ilce": d, "shapeName": name, "shapeID": f["properties"].get("shapeID", "")}, "geometry": f["geometry"]})
        found = {f["properties"]["ilce"] for f in feats}
        missing = [d for d in DISTRICTS if d not in found]
        if len(found) < 30:
            log(f"İlçe sınırları: yalnızca {len(found)} ilçe eşleşti, sınırlar kullanılmayacak. Eksik: {missing}")
            return None
        out = {"type": "FeatureCollection", "name": "istanbul_ilceler", "source": GEO_SOURCE_NOTE,
               "boundary_year": meta.get("boundaryYearRepresented"), "license": meta.get("licenseDetail"), "features": feats}
        with open(GEO_PATH, "w", encoding="utf-8") as fh:
            json.dump(out, fh, ensure_ascii=False)
        log(f"İlçe sınırları indirildi: {len(found)} ilçe ({GEO_SOURCE_NOTE})" + (f"; eksik: {missing}" if missing else ""))
        return GEO_PATH
    except Exception as e:
        log("İlçe sınırları indirilemedi (haritalar nokta işaretleriyle çizilecek):", repr(e))
        return None

GEOJSON_PATH = fetch_istanbul_geojson()

In [ ]:
# =============================================================================
# 3. VERİ ÜRETİMİ: etiket-öncelikli (label-first) sentetik paylaşım üretimi
# -----------------------------------------------------------------------------
# Tasarım: Önce her kayıt için ilçe, 1-3 gerçek ihtiyaç ve üslup özellikleri (persona, kayıt dili,
# uzunluk, yazım hatası, hashtag, emoji, mahalle, sayı, açık/örtük anlatım) örneklenir ("plan").
# Model yalnızca bu plana uygun Türkçe paylaşımı ve İngilizce karşılığını yazar. Böylece etiketler
# kesin olur, ihtiyaç ve ilçe dağılımı kontrol edilir, her ilçenin "baskın ihtiyaç profili" olur.
# =============================================================================
PROMPT_VERSION = "v1"   # DİKKAT: üretim imzasına girer; değiştirmeyin (önbellekteki parçalar yeniden kullanılsın)

PERSONAS = [
    "yaşlı bir kadın", "yaşlı bir adam", "üniversite öğrencisi", "üç çocuklu bir anne", "yeni doğum yapmış bir anne",
    "apartman yöneticisi", "mahalle muhtarı", "gönüllü bir arama kurtarma eri", "enkaz altında kalmış, telefon şarjı azalan bir kişi",
    "komşusu adına yazan bir kişi", "tekerlekli sandalye kullanan engelli bir birey", "hamile bir kadın", "evcil hayvan sahibi",
    "bakkal (esnaf)", "ilkokul öğretmeni", "Türkçesi sınırlı bir mülteci", "sağlık çalışanı (hemşire)",
    "şehir dışındaki, ailesine ulaşamayan bir akraba", "cami görevlisi", "site güvenlik görevlisi", "taksi şoförü", "lise öğrencisi",
    "diyaliz hastası bir yakını olan kişi", "yurtta kalan öğrenci", "inşaat işçisi", "ev hanımı", "emekli öğretmen", "kafe çalışanı",
    "sokak hayvanlarına bakan bir gönüllü", "yabancı turist (basit Türkçe)",
]
REGISTERS = [("panik, büyük harfler ve ünlemler", 0.25), ("sakin ve bilgilendirici", 0.25), ("kısa ve telgraf gibi", 0.15),
             ("duygusal ve yalvaran", 0.15), ("argo ve kısaltmalı gündelik dil", 0.12), ("resmi ve kurumsal", 0.08)]
LENGTHS = [("kısa (6-14 kelime)", 0.35), ("orta (15-30 kelime)", 0.45), ("uzun (31-50 kelime)", 0.20)]
PRIOR_WEIGHTS = defaultdict(lambda: 1.0, {
    "arama kurtarma ekibi": 6, "içme suyu": 5, "gıda": 4, "battaniye": 4, "çadır": 4, "sağlık ekibi": 4, "sıcak yemek": 3,
    "ilaç": 3, "bebek maması": 3, "bebek bezi": 2, "jenaratör": 2, "elektrikli ısıtıcı": 2, "el feneri": 2, "powerbank": 2,
    "kuru gıda": 2, "ekmek": 2, "seyyar tuvalet": 2, "uyku tulumu": 2, "mont": 2, "ilk yardım kiti": 2, "psikolojik destek ekibi": 2,
    "insülin": 1.5, "soba": 1.5, "pil": 1.5, "maske": 1.5, "yedek giysi": 1.5, "konteynır": 1.5, "hayvan maması": 1.5,
})
GEN_SIGNATURE = hashlib.md5(json.dumps([N_MESSAGES, GEN_MODEL, SEED, PROMPT_VERSION, IMPLICIT_RATIO, FIX_NEED_TRANSLATIONS,
                                        GEN_RECORDS_PER_CALL, MIN_PER_NEED, SMOKE_TEST]).encode()).hexdigest()[:10]
PARTS_DIR = os.path.join(DATA_DIR, "gen_parts", GEN_SIGNATURE)
os.makedirs(PARTS_DIR, exist_ok=True)
log(f"Üretim imzası: {GEN_SIGNATURE} (parçalar: {PARTS_DIR})")

def weighted_choice(rng, items_with_w):
    items, ws = zip(*items_with_w)
    return rng.choices(items, weights=ws, k=1)[0]

# --- İlçe profilleri: her ilçenin 1 birincil, 1 ikincil, 1 üçüncül ihtiyacı -----------------
prior_arr = np.array([PRIOR_WEIGHTS[n] for n in NEEDS_TR], dtype=float)
primary_needs = [str(x) for x in NP_RNG.choice(NEEDS_TR, size=len(DISTRICTS), replace=False, p=prior_arr / prior_arr.sum())]
DISTRICT_PROFILE = {}
for d, p1 in zip(DISTRICTS, primary_needs):
    rest = [n for n in NEEDS_TR if n != p1]
    w = np.array([PRIOR_WEIGHTS[n] for n in rest]); w = w / w.sum()
    p2, p3 = NP_RNG.choice(rest, size=2, replace=False, p=w)
    DISTRICT_PROFILE[d] = [p1, str(p2), str(p3)]

def sample_needs(rng, district):
    p1, p2, p3 = DISTRICT_PROFILE[district]
    r = rng.random()
    if r < 0.40:   first = p1
    elif r < 0.58: first = p2
    elif r < 0.70: first = p3
    else:          first = weighted_choice(rng, [(n, PRIOR_WEIGHTS[n]) for n in NEEDS_TR])
    n_needs = weighted_choice(rng, [(1, 0.60), (2, 0.30), (3, 0.10)])
    needs = [str(first)]
    while len(needs) < n_needs:
        cand = weighted_choice(rng, [(n, PRIOR_WEIGHTS[n]) for n in NEEDS_TR if n not in needs])
        needs.append(str(cand))
    return needs

def sample_style(rng, n_needs):
    r = rng.random()
    if r < IMPLICIT_RATIO:               explicitness = "implicit"
    elif r < IMPLICIT_RATIO + 0.15 and n_needs > 1: explicitness = "mixed"
    else:                                explicitness = "explicit"
    return {
        "persona": rng.choice(PERSONAS), "register": weighted_choice(rng, REGISTERS), "length": weighted_choice(rng, LENGTHS),
        "explicitness": explicitness, "typos": rng.random() < 0.30, "no_diacritics": rng.random() < 0.20,
        "hashtag": rng.random() < 0.30, "emoji": rng.random() < 0.20, "neighbourhood": rng.random() < 0.50, "numbers": rng.random() < 0.50,
    }

def build_specs(quota_by_district, round_idx):
    """Plan: ilçe kotalarına göre kayıt özellikleri. Etiket tabanları (MIN_PER_NEED) 0. turda uygulanır."""
    rng = random.Random(SEED * 1000 + round_idx)
    specs = []
    for d in DISTRICTS:
        for _ in range(int(quota_by_district.get(d, 0))):
            needs = sample_needs(rng, d)
            specs.append({"district": d, "needs_tr": needs, **sample_style(rng, len(needs))})
    if round_idx == 0:
        counts = Counter(n for s in specs for n in s["needs_tr"])
        bg = [i for i, s in enumerate(specs) if s["needs_tr"][0] not in DISTRICT_PROFILE[s["district"]]]
        rng.shuffle(bg)
        for need in NEEDS_TR:
            deficit = MIN_PER_NEED - counts[need]
            while deficit > 0 and bg:
                i = bg.pop()
                old = specs[i]["needs_tr"][0]
                if need in specs[i]["needs_tr"]:
                    continue
                specs[i]["needs_tr"][0] = need
                counts[old] -= 1; counts[need] += 1; deficit -= 1
    rng.shuffle(specs)
    for k, s in enumerate(specs):
        s["id"] = f"r{round_idx}_{k:05d}"
        s["needs_en"] = [TR2EN[n] for n in s["needs_tr"]]
        s["round"] = round_idx
    return specs

# --- İstem metinleri -------------------------------------------------------------
NEEDS_REFERENCE = "\n".join(f"- {tr} ({en})" for tr, en in zip(NEED_TABLE["tr"], NEED_TABLE["en"]))
SYSTEM_PROMPT_GEN = f"""You generate realistic simulated social media posts for an academic research dataset on rapid needs detection after a major earthquake in Istanbul, Türkiye (first 72 hours, winter). Posts are written by affected residents, relatives, neighbours and volunteers.

You receive a list of post specifications. For EACH specification write:
- "message_tr": the post in natural Turkish, exactly as a real person would post on X/Twitter in that situation (max 280 characters).
- "message_en": the same post as the same person would have written it in English (not a stiff translation; keep the meaning, the district name, the informality, roughly the same length; max 280 characters).

Hard rules:
1. Mention the assigned district exactly once, spelled exactly as given (e.g. "Kadıköy", "Şişli"), optionally with a Turkish suffix (Kadıköy'de, Şişli'den, Fatihte). Never mention any other Istanbul district from this list: {", ".join(DISTRICTS)}. Avoid neighbourhood or street names that contain another district's name (e.g. do not write "Kartaltepe" or "Maltepe Mahallesi" outside those districts). Keep the district name correctly spelled in BOTH languages, even when "no_diacritics" is true.
2. The post must convey ALL of the assigned needs ("needs_tr"/"needs_en") and no other item from the needs list below.
   - "explicit": name the item(s) plainly (everyday wording and synonyms are fine, e.g. "mama" for bebek maması).
   - "implicit": do NOT name the item; describe the situation so that a human would infer it (e.g. "my grandmother is diabetic and her injections stayed under the rubble" -> insülin; "the baby has had nothing since last night" -> bebek maması).
   - "mixed": the first need explicit, the others implicit.
3. Follow the persona, register and length. Feature flags: "typos": include 1-2 realistic Turkish typos or missing letters (never in the district name). "no_diacritics": drop Turkish diacritics in some words (never in the district name). "hashtag": add 1-2 relevant hashtags (#deprem, #İstanbul, a district tag). "emoji": 1-2 emojis. "neighbourhood": add a plausible real neighbourhood, street or landmark of that district. "numbers": include concrete numbers (people, hours, floors, ages).
4. No phone numbers, no real personal names, no URLs, no @mentions. Vary sentence openings and structure; do not reuse a template; do not start most posts with "Acil".
5. Return ONLY a JSON object with the key "records": a list with one object per specification, same "id" values, same order, fields: id, district, message_tr, message_en.

Needs list (Turkish (English)); for reference only, use everyday wording in posts:
{NEEDS_REFERENCE}
"""
GEN_SCHEMA = {
    "type": "object",
    "properties": {"records": {"type": "array", "items": {
        "type": "object",
        "properties": {"id": {"type": "string"}, "district": {"type": "string"},
                       "message_tr": {"type": "string"}, "message_en": {"type": "string"}},
        "required": ["id", "district", "message_tr", "message_en"], "additionalProperties": False}}},
    "required": ["records"], "additionalProperties": False,
}
SPEC_KEYS = ["id", "district", "needs_tr", "needs_en", "explicitness", "persona", "register", "length",
             "typos", "no_diacritics", "hashtag", "emoji", "neighbourhood", "numbers"]
def build_user_prompt(specs):
    lines = [json.dumps({k: s[k] for k in SPEC_KEYS}, ensure_ascii=False) for s in specs]
    return "Specifications (one JSON object per line):\n" + "\n".join(lines) + f"\n\nWrite {len(specs)} posts now."
with open(os.path.join(DIRS["logs"], "prompt_uretim.txt"), "w", encoding="utf-8") as f:
    f.write(SYSTEM_PROMPT_GEN + "\n\n--- örnek kullanıcı istemi ---\n" + build_user_prompt(build_specs({DISTRICTS[0]: 2}, 99)))

# --- Claude istemcisi ve yapılandırılmış JSON çağrısı ---------------------------------------
import threading
USAGE_LOCK = threading.Lock()
_API_TL = threading.local()   # son başarılı API çağrısının süresi (ms): bekleme/yeniden deneme süreleri hariç
USAGE = defaultdict(lambda: Counter())
class BudgetExceeded(RuntimeError):
    pass
_BUDGET_LOGGED = [False]
def add_usage(model, usage):
    if usage is None:
        return
    with USAGE_LOCK:
        for k in ("input_tokens", "output_tokens", "cache_creation_input_tokens", "cache_read_input_tokens"):
            USAGE[model][k] += int(getattr(usage, k, 0) or 0)
        det = getattr(usage, "output_tokens_details", None)
        USAGE[model]["thinking_tokens"] += int(getattr(det, "thinking_tokens", 0) or 0) if det is not None else 0
        USAGE[model]["calls"] += 1
def usage_cost_usd(model):
    u = USAGE.get(model, {})
    p = PRICES_USD_PER_MTOK.get(model)
    if not p:
        return float("nan")
    return (u.get("input_tokens", 0) * p[0] + u.get("output_tokens", 0) * p[1] +
            u.get("cache_read_input_tokens", 0) * p[2] + u.get("cache_creation_input_tokens", 0) * p[3]) / 1e6
def total_cost_usd():
    return float(np.nansum([usage_cost_usd(m) for m in USAGE])) if USAGE else 0.0
def usage_summary(model):
    u = USAGE.get(model, {}); n = max(1, u.get("calls", 0))
    return (f"{u.get('calls', 0)} çağrı | çağrı başına ort. çıktı {u.get('output_tokens', 0) / n:.0f} token "
            f"(düşünme {u.get('thinking_tokens', 0) / n:.0f}) | giriş {u.get('input_tokens', 0) / n:.0f} + önbellek {u.get('cache_read_input_tokens', 0) / n:.0f} | "
            f"tahmini ${usage_cost_usd(model):.2f}")
def check_budget():
    if BUDGET_GUARD_USD is not None and total_cost_usd() >= BUDGET_GUARD_USD:
        if not _BUDGET_LOGGED[0]:
            _BUDGET_LOGGED[0] = True
            log(f"BÜTÇE FRENİ: tahmini maliyet ${total_cost_usd():.2f} >= {BUDGET_GUARD_USD} $. Yeni API çağrısı yapılmıyor; "
                f"devam etmek için BUDGET_GUARD_USD'yi artırıp hücreyi yeniden çalıştırın (tamamlanan parçalar Drive'da).")
        raise BudgetExceeded("bütçe aşıldı")

def thinking_param(model, mode):
    """Düşünme (thinking) denetimi: Sonnet 5.5 ön düşünmeyi 'between_tools' ile kapatır, Haiku 5.5 'disabled' kabul eder,
    Opus 5.5 / Fable 5.1'de düşünme kapatılamaz (yalnız effort ile sınırlanır)."""
    if mode != "minimal":
        return None
    if "sonnet-5-5" in model:
        return {"type": "between_tools"}
    if "haiku-5-5" in model:
        return {"type": "disabled"}
    return None

def extract_json(text):
    try:
        return json.loads(text)
    except Exception:
        s, e = text.find("{"), text.rfind("}")
        return json.loads(text[s:e + 1])

ANTHROPIC_CLIENT = None
SDK_SUPPORTS = {"temperature": False, "output_config": True}
if not SMOKE_TEST:
    import anthropic, inspect as _inspect
    ANTHROPIC_CLIENT = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY, max_retries=4, timeout=600)
    try:
        _params = set(_inspect.signature(ANTHROPIC_CLIENT.messages.create).parameters)
        SDK_SUPPORTS = {"temperature": "temperature" in _params, "output_config": "output_config" in _params}
    except Exception:
        pass
    log(f"anthropic SDK {anthropic.__version__}: temperature={'var' if SDK_SUPPORTS['temperature'] else 'yok (API varsayılanı kullanılır)'}, "
        f"output_config={'var' if SDK_SUPPORTS['output_config'] else 'yok (düz JSON istemi)'}")

def claude_json(system_text, user_text, schema, model, max_tokens=8000, temperature=None, retries=6, effort=None, thinking_mode=None):
    """Yapılandırılmış JSON çıktısı (output_config.format). Düşünme/effort denetimi, SDK imzasına göre parametre seçimi,
    bütçe freni, kesilen yanıtta max_tokens artırımı, 429/5xx için üstel bekleme. Desteklenmeyen parametre 400 verirse düşürülür."""
    last_err = None
    thinking_mode = THINKING_MODE if thinking_mode is None else thinking_mode
    use_temp = SDK_SUPPORTS["temperature"] and temperature is not None
    use_output_config = SDK_SUPPORTS["output_config"]
    think = thinking_param(model, thinking_mode)
    eff = effort
    for attempt in range(retries):
        check_budget()
        try:
            kwargs = dict(model=model, max_tokens=max_tokens,
                          system=[{"type": "text", "text": system_text, "cache_control": {"type": "ephemeral"}}],
                          messages=[{"role": "user", "content": user_text}])
            if use_temp:
                kwargs["temperature"] = temperature
            oc = {}
            if use_output_config:
                oc["format"] = {"type": "json_schema", "schema": schema}
            else:
                kwargs["messages"][0]["content"] += "\n\nReturn ONLY valid JSON matching this schema:\n" + json.dumps(schema)
            if eff and use_output_config:
                oc["effort"] = eff
            if oc:
                kwargs["output_config"] = oc
            if think:
                kwargs["thinking"] = think
            _t0 = time.perf_counter()
            resp = ANTHROPIC_CLIENT.messages.create(**kwargs)
            _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
            add_usage(model, getattr(resp, "usage", None))
            if resp.stop_reason == "max_tokens":
                max_tokens = int(max_tokens * 1.5)
                raise RuntimeError(f"Yanıt max_tokens sınırında kesildi, sınır {max_tokens} olarak artırıldı")
            if resp.stop_reason == "refusal":
                raise RuntimeError("Model yanıtı reddetti (refusal)")
            text = "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")
            return extract_json(text)
        except BudgetExceeded:
            raise
        except TypeError as e:   # SDK imzası beklenenden farklı: parametreyi düşür ve hemen yeniden dene
            last_err = e; msg = str(e)
            if "temperature" in msg and use_temp:
                log("SDK temperature parametresini kabul etmiyor, parametresiz devam ediliyor"); use_temp = False; SDK_SUPPORTS["temperature"] = False; continue
            if "output_config" in msg and use_output_config:
                log("SDK output_config parametresini kabul etmiyor, düz JSON istemine geçiliyor"); use_output_config = False; SDK_SUPPORTS["output_config"] = False; continue
            if "thinking" in msg and think:
                log("SDK thinking parametresini kabul etmiyor, parametresiz devam ediliyor"); think = None; continue
            raise
        except Exception as e:
            last_err = e; msg = str(e)
            if e.__class__.__name__ == "BadRequestError":
                if use_temp and "temperature" in msg:
                    log(f"{model}: API temperature kabul etmedi, parametresiz yeniden deneniyor"); use_temp = False; continue
                if think and "thinking" in msg:
                    log(f"{model}: thinking={think} kabul edilmedi ({msg[:120]}), parametresiz yeniden deneniyor"); think = None; continue
                if eff and "effort" in msg:
                    log(f"{model}: effort={eff} kabul edilmedi ({msg[:120]}), effort olmadan yeniden deneniyor"); eff = None; continue
                raise  # diğer 400 hatalarında yeniden denemenin anlamı yok
            wait = min(90, (2 ** attempt) + random.random() * 2)
            log(f"API hatası ({model}, deneme {attempt + 1}/{retries}): {msg[:160]} -> {wait:.0f} sn bekleniyor")
            time.sleep(wait)
    raise RuntimeError(f"API çağrısı başarısız: {last_err}")

# --- Sahte üretici (SMOKE_TEST) ------------------------------------------------------
_MOCK_OPEN_TR = ["Acil yardım!", "Lütfen duyun,", "Durum çok kötü,", "Yardım edin,", "Arkadaşlar", "SOS", "Bilgi:", "Komşular için yazıyorum,"]
_MOCK_OPEN_EN = ["Urgent help!", "Please hear us,", "Situation is bad,", "Help us,", "Friends", "SOS", "Info:", "Writing for my neighbours,"]
def mock_generate(specs):
    rng = random.Random(hash(specs[0]["id"]) % 10**6)
    out = []
    for s in specs:
        d = s["district"]
        tr_needs = ", ".join(s["needs_tr"]); en_needs = ", ".join(s["needs_en"])
        k = rng.randint(0, 999)
        tr = f"{rng.choice(_MOCK_OPEN_TR)} {d}'de {rng.randint(2, 40)} kişiyiz, {tr_needs} lazım. {s['persona']} {k}"
        en = f"{rng.choice(_MOCK_OPEN_EN)} we are {rng.randint(2, 40)} people in {d}, we need {en_needs}. {s['persona']} {k}"
        if s["hashtag"]:
            tr += " #deprem"; en += " #earthquake"
        out.append({"id": s["id"], "district": d, "message_tr": tr, "message_en": en})
    return {"records": out}

# --- Üretim çalıştırıcı (parça önbelleği + eş zamanlı çağrılar) ------------------------------
def generate_chunk(chunk_specs, part_path):
    if os.path.exists(part_path) and not FORCE_REGENERATE:
        with open(part_path, encoding="utf-8") as f:
            return json.load(f)
    t0 = time.perf_counter()
    if SMOKE_TEST:
        data = mock_generate(chunk_specs)
    else:
        try:
            data = claude_json(SYSTEM_PROMPT_GEN, build_user_prompt(chunk_specs), GEN_SCHEMA, GEN_MODEL,
                               max_tokens=max(3000, GEN_MAX_TOKENS_PER_RECORD * len(chunk_specs)), temperature=GEN_TEMPERATURE, effort=GEN_EFFORT)
        except BudgetExceeded:
            return None
    part = {"specs": chunk_specs, "records": data.get("records", []), "model": GEN_MODEL,
            "seconds": round(time.perf_counter() - t0, 1), "ts": datetime.datetime.now().isoformat()}
    with open(part_path, "w", encoding="utf-8") as f:
        json.dump(part, f, ensure_ascii=False)
    return part

def run_generation(specs, round_idx):
    chunks = [specs[i:i + GEN_RECORDS_PER_CALL] for i in range(0, len(specs), GEN_RECORDS_PER_CALL)]
    parts = [None] * len(chunks)
    with ThreadPoolExecutor(max_workers=GEN_WORKERS) as ex:
        futs = {ex.submit(generate_chunk, ch, os.path.join(PARTS_DIR, f"r{round_idx}_c{i:04d}.json")): i for i, ch in enumerate(chunks)}
        done = 0
        for fut in tqdm(as_completed(futs), total=len(futs), desc=f"Üretim turu {round_idx}"):
            i = futs[fut]
            try:
                parts[i] = fut.result()
            except Exception as e:
                log(f"Parça {i} başarısız: {e}")
            done += 1
            if not SMOKE_TEST and (done % 10 == 0 or done == len(futs)):
                log(f"Üretim {done}/{len(futs)} | {GEN_MODEL}: {usage_summary(GEN_MODEL)} | oturum toplamı ${total_cost_usd():.2f}")
    return [p for p in parts if p]

PHONE_RE = re.compile(r"\d{3}[\s.-]?\d{3}[\s.-]?\d{2}[\s.-]?\d{2}")
def validate_records(parts):
    """Model çıktısını plana karşı denetler. Döndürür: kabul edilen kayıtlar (spec + metinler), ret nedenleri."""
    accepted, reasons = [], Counter()
    for part in parts:
        spec_by_id = {s["id"]: s for s in part["specs"]}
        seen = set()
        for r in part["records"]:
            s = spec_by_id.get(r.get("id"))
            if s is None or r["id"] in seen:
                reasons["kimlik uyuşmazlığı"] += 1; continue
            seen.add(r["id"])
            tr, en = str(r.get("message_tr", "")).strip(), str(r.get("message_en", "")).strip()
            d = s["district"]
            if not (10 <= len(tr) <= 420 and 10 <= len(en) <= 420):
                reasons["uzunluk"] += 1; continue
            if tr_lower(d) not in tr_lower(tr) or tr_lower(d) not in tr_lower(en):
                reasons["ilçe adı yok"] += 1; continue
            others_tr = [x for x in detect_districts(tr) if x != d]
            others_en = [x for x in detect_districts(en) if x != d]
            if others_tr or others_en:
                reasons["başka ilçe adı"] += 1; continue
            if PHONE_RE.search(tr) or PHONE_RE.search(en) or "http" in en.lower() or "@" in en:
                reasons["telefon/URL/@"] += 1; continue
            accepted.append({**s, "message_tr": tr, "message_en": en, "gen_model": part["model"]})
    return accepted, reasons

# --- Tekrar eleme (anlamsal) ---------------------------------------------------------------
def _hash_embed(texts, dim=256):
    out = np.zeros((len(texts), dim), dtype=np.float32)
    for i, t in enumerate(texts):
        for tok in re.findall(r"\w+", tr_lower(t)):
            h = int(hashlib.md5(tok.encode()).hexdigest(), 16)
            out[i, h % dim] += 1.0 if (h >> 8) % 2 else -1.0
    return out
_DEDUP_MODEL = None
def dedup_embed(texts):
    global _DEDUP_MODEL
    if SMOKE_TEST:
        return _hash_embed(texts)
    if _DEDUP_MODEL is None:
        from sentence_transformers import SentenceTransformer
        _DEDUP_MODEL = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
    return _DEDUP_MODEL.encode(texts, batch_size=128, convert_to_numpy=True, show_progress_bar=False)

def deduplicate(records, threshold):
    if not records:
        return records, 0
    norm = lambda t: re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", tr_lower(t))).strip()
    keep, seen_norm = [], set()
    for r in records:
        key = norm(r["message_en"])
        if key in seen_norm:
            continue
        seen_norm.add(key); keep.append(r)
    E = dedup_embed([r["message_en"] for r in keep]).astype(np.float32)
    E /= (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)
    S = E @ E.T
    np.fill_diagonal(S, 0.0)
    drop = np.zeros(len(keep), dtype=bool)
    for j in range(len(keep)):
        if j and S[j, :j][~drop[:j]].size and S[j, :j][~drop[:j]].max() > threshold:
            drop[j] = True
    final = [r for r, dr in zip(keep, drop) if not dr]
    return final, len(records) - len(final)

# --- Bağımsız modelle etiket kalite kontrolü (QC) ----------------------------------------------
# Üretici modelden farklı bir model, her kaydı plana karşı denetler: atanan ihtiyaçların hepsi metinde
# (açık veya örtük) var mı, listeden başka bir ihtiyaç geçiyor mu, ilçe doğru mu. Sonuç raporlanır
# (kalite_kontrol.csv), QC_EXCLUDE_FAILED=True ise başarısızlar veri setinden çıkarılır.
SYSTEM_PROMPT_QC = f"""You are a strict annotator for a disaster-informatics dataset. Each item has a Turkish social media post, its English version, an assigned Istanbul district and assigned needs taken from a fixed needs list.
For each item decide:
- all_needs_expressed: true only if EVERY assigned need is conveyed by the post (explicitly, or by clear implication a human reader would make; e.g. a diabetic relative without injections implies insülin).
- missing: assigned needs that are NOT conveyed (empty list if none).
- extra: items from the needs list that the post clearly asks for but that were NOT assigned (empty list if none; ignore generic words like "yardım").
- district_ok: true if the assigned district is mentioned and no other district from the list is mentioned.
Needs list (Turkish (English)):
{NEEDS_REFERENCE}
Districts: {", ".join(DISTRICTS)}
Return only JSON {{"results": [{{"id", "all_needs_expressed", "missing", "extra", "district_ok"}}]}} with one entry per item, same ids."""
QC_SCHEMA = {"type": "object", "properties": {"results": {"type": "array", "items": {
    "type": "object", "properties": {"id": {"type": "string"}, "all_needs_expressed": {"type": "boolean"},
                                     "missing": {"type": "array", "items": {"type": "string"}}, "extra": {"type": "array", "items": {"type": "string"}},
                                     "district_ok": {"type": "boolean"}},
    "required": ["id", "all_needs_expressed", "missing", "extra", "district_ok"], "additionalProperties": False}}},
    "required": ["results"], "additionalProperties": False}

def qc_chunk(chunk, part_path):
    if os.path.exists(part_path) and not FORCE_REGENERATE:
        with open(part_path, encoding="utf-8") as f:
            return json.load(f)
    if SMOKE_TEST:
        rng = random.Random(part_path)
        data = {"results": [{"id": r["id"], "all_needs_expressed": rng.random() < 0.95, "missing": [], "extra": [], "district_ok": True} for r in chunk]}
    else:
        user = "Items (one JSON object per line):\n" + "\n".join(json.dumps({k: r[k] for k in ("id", "district", "needs_tr", "needs_en", "explicitness", "message_tr", "message_en")}, ensure_ascii=False) for r in chunk)
        data = claude_json(SYSTEM_PROMPT_QC, user, QC_SCHEMA, QC_MODEL, max_tokens=200 * len(chunk) + 1500, temperature=0.0, effort=QC_EFFORT)
    with open(part_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    return data


# =============================================================================
# 3R. MAKALEDEKİ VERİ SETİNİN YÜKLENMESİ (bu defterde yeniden üretim yapılmaz)
# -----------------------------------------------------------------------------
# Yukarıdaki tanımlar (persona, üslup, plan, üretim istemi, doğrulama, tekrar eleme, kalite kontrol istemi)
# makaledeki defterle birebir aynıdır; alternatif üreticiler aynı istem ve aynı planlarla çalıştırılır.
# =============================================================================
if GEN_SIGNATURE != PAPER_GEN_SIGNATURE:
    log(f"Not: bu oturumun üretim imzası {GEN_SIGNATURE}, makaledeki {PAPER_GEN_SIGNATURE}"
        + (" (SMOKE_TEST)" if SMOKE_TEST else "; ayarlar hücresi değişmiş olabilir. Veri yine makaledeki dosyadan okunur."))
with open(paper_file("logs/ilce_profilleri.json"), encoding="utf-8") as _f:
    _paper_profile = json.load(_f)
if _paper_profile != DISTRICT_PROFILE:
    log("Not: ilçe profilleri bu oturumda farklı çıktı; makaledeki profiller kullanılıyor.")
    DISTRICT_PROFILE = _paper_profile

df_syn = pd.read_csv(paper_file(f"data/sentetik_paylasimlar_{PAPER_GEN_SIGNATURE}.csv"), encoding="utf-8-sig")
assert len(df_syn) == N_MESSAGES and (df_syn["id"].values == np.arange(len(df_syn))).all(), "Veri setinde beklenmeyen kimlikler"
for _c in ("label1", "label2", "label3"):
    df_syn[_c] = df_syn[_c].apply(lambda x: canon_need_tr(x) if isinstance(x, str) else None)
df_syn["labels_tr"] = df_syn[["label1", "label2", "label3"]].apply(lambda s: [x for x in s if isinstance(x, str)], axis=1)
df_syn["labels_en"] = df_syn["labels_tr"].apply(lambda L: list(dict.fromkeys(TR2EN[x] for x in L)))
df_syn["districts_detected"] = df_syn["Message"].apply(detect_districts)
for _c in ("typos", "no_diacritics", "hashtag", "emoji", "neighbourhood", "numbers"):
    df_syn[_c] = df_syn[_c].astype(str).str.strip().str.lower().eq("true")
DISTRICT_DETECT_AGREEMENT = float(df_syn.apply(lambda r: r["districts_detected"] == [r["district"]], axis=1).mean())
DATASET_BASENAME = f"sentetik_paylasimlar_{PAPER_GEN_SIGNATURE}"
assert all(len(L) >= 1 for L in df_syn["labels_tr"]), "Etiketsiz paylaşım var"
log(f"Makaledeki veri seti yüklendi: {len(df_syn)} paylaşım, {df_syn['district'].nunique()} ilçe, "
    f"QC geçen {int(df_syn['qc_ok'].astype(str).str.lower().eq('true').sum())}; ilçe adı tespiti uyumu {DISTRICT_DETECT_AGREEMENT:.3f}")

In [ ]:
# =============================================================================
# 4. ÖN İŞLEME (düz / lemma / gövde), VERİ İSTATİSTİKLERİ
# =============================================================================
import inflect
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer, PorterStemmer
INFLECT = inflect.engine()
STOP_EN = set(stopwords.words("english"))
LEMM, STEMMER = WordNetLemmatizer(), PorterStemmer()
LETTERS_RE = re.compile(r"[^a-zA-ZğĞçÇşŞüÜöÖıİ]")   # Türkçe harfler korunur (ilçe adları parçalanmaz)

def numbers_to_words(text):
    return re.sub(r"\d+", lambda m: " " + (INFLECT.number_to_words(int(m.group())) if len(m.group()) < 12 else "") + " ", text)

def en_lower(s):
    """İngilizce metin için küçük harf: 'İ' -> 'i' (Python lower() birleşik nokta üretir), 'I' -> 'i' (Türkçe kural UYGULANMAZ)."""
    return str(s).replace("İ", "i").lower()

def plain_text(text):
    """Düz metin (İngilizce hat): sayılar sözcüğe, yalnız harfler, İngilizce küçük harf; Türkçe yer adlarındaki harfler korunur."""
    if not isinstance(text, str):
        return ""
    t = LETTERS_RE.sub(" ", numbers_to_words(text))
    return " ".join(en_lower(t).split())

def _wn_pos(tag):
    return {"J": wordnet.ADJ, "V": wordnet.VERB, "R": wordnet.ADV}.get(tag[:1], wordnet.NOUN)

_POS_OK = True
try:
    nltk.pos_tag(["test"])
except Exception as e:
    _POS_OK = False
    log("UYARI: POS etiketleyici yüklenemedi, lemmalama yalnız isim kipinde:", e)

def lemmatize_text(text):
    toks = [w for w in plain_text(text).split() if w not in STOP_EN]
    if _POS_OK:
        return " ".join(LEMM.lemmatize(w, _wn_pos(t)) for w, t in nltk.pos_tag(toks))
    return " ".join(LEMM.lemmatize(w) for w in toks)

def stem_text(text):
    return " ".join(STEMMER.stem(w) for w in plain_text(text).split() if w not in STOP_EN)

TEXT_FUNCS = {"plain": plain_text, "lemma": lemmatize_text, "stem": stem_text}
TEXT_TYPE_TR = {"plain": "Düz", "lemma": "Lemma", "stem": "Gövde"}

def add_text_columns(df):
    df = df.copy()
    for tt, fn in TEXT_FUNCS.items():
        df["text_" + tt] = df["English"].apply(fn)
    return df

def need_texts(text_type):
    """Model girdisi olarak ihtiyaç adları (PREPROCESS_NEEDS_LIKE_MESSAGES=True: mesajla aynı işlem)."""
    if PREPROCESS_NEEDS_LIKE_MESSAGES:
        out = [TEXT_FUNCS[text_type](n) for n in EN_LABELS]
        return [o if o.strip() else n.lower() for o, n in zip(out, EN_LABELS)]
    return [n.lower() for n in EN_LABELS]

df_syn = add_text_columns(df_syn)
EVAL_SETS = {"sentetik": df_syn}

df_syn.drop(columns=["labels_tr", "labels_en", "districts_detected"]).to_excel(os.path.join(DIRS["results"], "veri_seti_sentetik_islenmis.xlsx"), index=False)

# --- Tablo 2 benzeri örnekler ---------------------------------------------------------------
ex = df_syn.sample(min(8, len(df_syn)), random_state=SEED)[["district", "Message", "English", "text_plain", "text_lemma", "text_stem", "labels_tr"]]
ex = ex.rename(columns={"district": "İlçe", "Message": "Paylaşım (TR)", "English": "Paylaşım (EN)", "text_plain": "Düz", "text_lemma": "Lemma", "text_stem": "Gövde", "labels_tr": "Gerçek ihtiyaç(lar)"})
save_table(ex, "tablo2_onislenmis_ornekler")

# --- Veri istatistikleri ---------------------------------------------------------------------
stats_d = (df_syn.groupby("district").agg(mesaj=("id", "size"), ort_kelime_en=("English", lambda s: s.str.split().str.len().mean()))
           .reindex(DISTRICTS).fillna(0))
stats_d["en_sik_ihtiyac"] = [", ".join(f"{n} ({c})" for n, c in Counter(x for L in df_syn[df_syn.district == d]["labels_tr"] for x in L).most_common(3)) for d in DISTRICTS]
stats_d["profil"] = [", ".join(DISTRICT_PROFILE[d]) for d in DISTRICTS]
save_table(stats_d.reset_index().rename(columns={"district": "İlçe"}), "veri_ilce_dagilimi")

cnt_any = Counter(x for L in df_syn["labels_tr"] for x in L); cnt_first = Counter(df_syn["label1"])
stats_n = pd.DataFrame({"İhtiyaç (TR)": NEEDS_TR, "İhtiyaç (EN)": [TR2EN[n] for n in NEEDS_TR],
                        "etiket_sayisi": [cnt_any[n] for n in NEEDS_TR], "birinci_etiket": [cnt_first[n] for n in NEEDS_TR]})
save_table(stats_n.sort_values("etiket_sayisi", ascending=False), "veri_ihtiyac_dagilimi")

summary = {
    "kayit": len(df_syn), "ilce": df_syn["district"].nunique(), "ilce_basina_ort": round(len(df_syn) / 39, 1),
    "benzersiz_etiket_tr": df_syn["labels_tr"].explode().nunique(),
    "ort_karakter_tr": round(df_syn["Message"].str.len().mean(), 1), "ort_kelime_tr": round(df_syn["Message"].str.split().str.len().mean(), 1),
    "ort_karakter_en": round(df_syn["English"].str.len().mean(), 1), "ort_kelime_en": round(df_syn["English"].str.split().str.len().mean(), 1),
    "etiket_1_2_3_oran": " / ".join(fmt_tr(v, 2) for v in df_syn["n_needs"].value_counts(normalize=True).reindex([1, 2, 3]).fillna(0)),
    "acik_karma_ortuk_oran": " / ".join(fmt_tr(v, 2) for v in df_syn["explicitness"].value_counts(normalize=True).reindex(["explicit", "mixed", "implicit"]).fillna(0)),
    "ilce_tespit_uyumu": round(DISTRICT_DETECT_AGREEMENT, 3), "uretim_modeli": GEN_MODEL,
}
save_table(pd.DataFrame([summary]).T.reset_index().rename(columns={"index": "ölçü", 0: "değer"}), "veri_ozet", float_nd=1)
log("Veri özeti:", summary)

# Şekiller: ilçe dağılımı ve ihtiyaç dağılımı
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(range(39), stats_d["mesaj"].values, color=PALETTE[0])
ax.set_xticks(range(39)); ax.set_xticklabels(DISTRICTS, rotation=75, ha="right", fontsize=8)
ax.set_ylabel("Paylaşım sayısı"); ax.set_title("İlçelere göre simüle paylaşım sayısı")
fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], "sekil_veri_ilce_dagilimi.png"), dpi=300); plt.close(fig)

top = stats_n.sort_values("etiket_sayisi", ascending=True).tail(30)
fig, ax = plt.subplots(figsize=(7.5, 8))
ax.barh(top["İhtiyaç (TR)"], top["etiket_sayisi"], color=PALETTE[2])
ax.set_xlabel("Etiket sayısı (bir paylaşımda 1-3 ihtiyaç)"); ax.set_title("En sık 30 ihtiyaç etiketi")
fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], "sekil_veri_ihtiyac_dagilimi.png"), dpi=300); plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(df_syn["Message"].str.split().str.len(), bins=25, alpha=0.7, label="Türkçe", color=PALETTE[0])
ax.hist(df_syn["English"].str.split().str.len(), bins=25, alpha=0.7, label="İngilizce", color=PALETTE[1])
ax.set_xlabel("Kelime sayısı"); ax.set_ylabel("Paylaşım"); ax.legend(frameon=False); ax.set_title("Paylaşım uzunluğu dağılımı")
fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], "sekil_veri_uzunluk.png"), dpi=300); plt.close(fig)
log("Ön işleme ve veri istatistikleri tamam.")
# --- Makaledeki ön işlenmiş metin sütunları ---------------------------------------------------------
# NLTK / inflect sürüm farkları lemma ve gövde sütunlarını değiştirebileceği için makaledeki sütunlar kullanılır.
_proc = pd.read_excel(paper_file("results/veri_seti_sentetik_islenmis.xlsx")).set_index("id").reindex(df_syn["id"].values)
for _c in ("text_plain", "text_lemma", "text_stem"):
    _new = _proc[_c].fillna("").astype(str).values
    log(f"{_c}: bu oturumdaki ön işleme makaledekiyle {float(np.mean(df_syn[_c].values == _new)):.3f} oranında aynı; makaledeki kullanılıyor")
    df_syn[_c] = _new
EVAL_SETS = {"sentetik": df_syn}

In [ ]:
# =============================================================================
# 5. DİL TEMSİL MODELLERİ, KOSİNÜS BENZERLİĞİ, DOĞRULUK ÖLÇÜTLERİ, HARİTALAR
# -----------------------------------------------------------------------------
# Eşit koşullar:
#  * Tüm modeller aynı mesajları, aynı ihtiyaç listesini, aynı ön işlemeyi, aynı karar kuralını
#    (kosinüs benzerliğinde en yüksek ihtiyaç = top-1) ve aynı ölçütleri kullanır; hiçbir model eğitilmez/ayarlanmaz.
#  * Her modelin çıkarım ayarları (kaynak, eğitim verisi, boyut, parametre sayısı, cümle temsili,
#    ince ayar durumu, kasa duyarlılığı, OOV davranışı) Tablo 3b'ye yazılır.
#  * Ham/ince ayarlı eş çiftler (BERT-raw ↔ BERT-nli, ALBERT ↔ ALBERT-paraphrase) ince ayar etkisini ayırır.
#  * Temel çizgiler (TF-IDF, rastgele, çoğunluk) şans düzeyini gösterir.
#  * Gecikme tüm yerel modeller için aynı cihazda (CPU, tek mesaj) ölçülür; toplu GPU süresi ayrıca verilir.
# Uygulama ayrıntıları: ihtiyaç gömmeleri bir kez hesaplanır; sıfır vektörlü mesaj "tahminsiz" sayılır;
# makaledeki ilçe ölçütü aynen korunur, mesaj düzeyi hit@1/hit@3 ve bootstrap %95 GA eklenir.
# =============================================================================
MODEL_SPECS = {
    # makaledeki 7 model
    "BERT":        {"kind": "st",  "name": "bert-base-nli-mean-tokens"},
    "ALBERT":      {"kind": "st",  "name": "albert-base-v2"},
    "RoBERTa":     {"kind": "st",  "name": "roberta-base-nli-mean-tokens"},
    "GloVe":       {"kind": "glove"},
    "Word2Vec":    {"kind": "w2v"},
    "USE":         {"kind": "use"},
    "FastText":    {"kind": "fasttext"},
    # ek modeller (eşit koşul çiftleri ve güncel SBERT)
    "SBERT-mpnet":       {"kind": "st", "name": "all-mpnet-base-v2"},
    "BERT-raw":          {"kind": "st", "name": "bert-base-uncased"},
    "ALBERT-paraphrase": {"kind": "st", "name": "paraphrase-albert-small-v2"},
    # Türkçe metin + Türkçe ihtiyaç listesi (yalnız düz metin)
    "SBERT-çokdilli (TR)": {"kind": "st_tr", "name": "paraphrase-multilingual-MiniLM-L12-v2"},
    "BERTurk-STS (TR)":    {"kind": "st_tr", "name": "emrecan/bert-base-turkish-cased-mean-nli-stsb-tr"},
    # temel çizgiler
    "Temel: TF-IDF":   {"kind": "tfidf"},
    "Temel: Rastgele": {"kind": "random"},
    "Temel: Çoğunluk": {"kind": "majority"},
}
MODEL_META = {   # Tablo 3b için sabit bilgiler (boyut ve parametre sayısı çalışma anında ölçülür)
    "BERT":     dict(tur="Transformer (BERT-base)", egitim="Wikipedia + BookCorpus; SNLI+MultiNLI ile cümle benzerliği ince ayarı", temsil="Ortalama havuzlama (sentence-transformers)", ince_ayar="Evet (NLI)", kasa="Duyarsız", oov="WordPiece alt kelime"),
    "ALBERT":   dict(tur="Transformer (ALBERT-base-v2)", egitim="Wikipedia + BookCorpus; benzerlik ince ayarı YOK", temsil="Ortalama havuzlama (otomatik)", ince_ayar="Hayır", kasa="Duyarsız", oov="SentencePiece alt kelime"),
    "RoBERTa":  dict(tur="Transformer (RoBERTa-base)", egitim="160 GB web metni; SNLI+MultiNLI ince ayarı", temsil="Ortalama havuzlama (sentence-transformers)", ince_ayar="Evet (NLI)", kasa="Duyarlı", oov="Byte-BPE alt kelime"),
    "GloVe":    dict(tur="Statik kelime vektörü", egitim="Common Crawl, 840 milyar token", temsil="Sözlükteki kelime vektörlerinin ortalaması", ince_ayar="Hayır", kasa="Duyarlı", oov="Yok (kelime atlanır)"),
    "Word2Vec": dict(tur="Statik kelime vektörü", egitim="Google News, 100 milyar token", temsil="Sözlükteki kelime vektörlerinin ortalaması", ince_ayar="Hayır", kasa="Duyarlı", oov="Yok (kelime atlanır)"),
    "USE":      dict(tur="Cümle kodlayıcı (DAN, USE v4)", egitim="Wikipedia, web haber/forum, SNLI (çok görevli)", temsil="Doğrudan cümle gömmesi", ince_ayar="Hayır (cümle düzeyinde eğitilmiş)", kasa="Duyarsız", oov="Model içi sözlük/karma"),
    "FastText": dict(tur="Statik kelime vektörü (CBOW + karakter n-gram)", egitim="Common Crawl + Wikipedia (cc.en.300)", temsil="Birim normlu kelime vektörlerinin ortalaması", ince_ayar="Hayır", kasa="Duyarlı", oov="Karakter n-gram (her kelimeye vektör)"),
    "SBERT-mpnet":       dict(tur="Transformer (MPNet-base)", egitim="1 milyar+ cümle çifti ile karşıtsal ince ayar", temsil="Ortalama havuzlama", ince_ayar="Evet (cümle çiftleri)", kasa="Duyarsız", oov="WordPiece alt kelime"),
    "BERT-raw":          dict(tur="Transformer (BERT-base, ham)", egitim="Wikipedia + BookCorpus; ince ayar YOK (ALBERT ile eşit koşul)", temsil="Ortalama havuzlama (otomatik)", ince_ayar="Hayır", kasa="Duyarsız", oov="WordPiece alt kelime"),
    "ALBERT-paraphrase": dict(tur="Transformer (ALBERT-small-v2)", egitim="Paraphrase cümle çiftleri ile ince ayar (BERT-nli ile eşit koşul)", temsil="Ortalama havuzlama", ince_ayar="Evet (paraphrase)", kasa="Duyarsız", oov="SentencePiece alt kelime"),
    "SBERT-çokdilli (TR)": dict(tur="Transformer (MiniLM-L12, çok dilli damıtma)", egitim="50+ dilde paralel cümle çiftleri (Türkçe dâhil)", temsil="Ortalama havuzlama", ince_ayar="Evet (çok dilli damıtma)", kasa="Duyarlı", oov="SentencePiece alt kelime"),
    "BERTurk-STS (TR)":    dict(tur="Transformer (BERTurk-base)", egitim="Türkçe korpus; NLI-TR + STSb-TR ince ayarı", temsil="Ortalama havuzlama", ince_ayar="Evet (NLI/STS, Türkçe)", kasa="Duyarlı", oov="WordPiece alt kelime"),
    "Temel: TF-IDF":   dict(tur="Sözcüksel temel çizgi", egitim="Yalnız bu veri seti (1-2 gram TF-IDF)", temsil="Seyrek TF-IDF vektörü", ince_ayar="Hayır", kasa="Duyarsız", oov="Yok"),
    "Temel: Rastgele": dict(tur="Şans düzeyi", egitim="Yok", temsil="Rastgele ihtiyaç", ince_ayar="Hayır", kasa="-", oov="-"),
    "Temel: Çoğunluk": dict(tur="Şans düzeyi", egitim="Yok", temsil="Veri setindeki en sık ihtiyaç", ince_ayar="Hayır", kasa="-", oov="-"),
}
MAIN_MODELS = [m for m in MODELS_TO_RUN if m in MODEL_SPECS]
ALL_MODELS = [m for m in MODELS_TO_RUN + EXTRA_MODELS + TURKISH_MODELS + BASELINES if m in MODEL_SPECS]
BASELINE_NAMES = [m for m in BASELINES if m in MODEL_SPECS]

def find_existing(cands):
    for rel in cands:
        p = os.path.join(DRIVE_ROOT, rel)
        if os.path.exists(p):
            return p
    return None

def download_file(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 1e6:
        return dest
    log(f"İndiriliyor: {url}")
    tmp = dest + ".part"
    with tqdm(unit="B", unit_scale=True, desc=os.path.basename(dest)) as bar:
        def hook(b, bs, total):
            if total > 0:
                bar.total = total
            bar.update(bs)
        urllib.request.urlretrieve(url, tmp, reporthook=hook)
    os.replace(tmp, dest)
    return dest

# --- Gömme sınıfları ---------------------------------------------------------------------
class STEmbedder:
    def __init__(self, name):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(name, device=DEVICE)
        self.info = f"sentence-transformers:{name}"
        self.n_params = int(sum(p.numel() for p in self.model.parameters()))
        self.dim = int(self.model.get_sentence_embedding_dimension() or 0)
    def encode(self, texts):
        return np.asarray(self.model.encode(list(texts), batch_size=64, convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)
    def set_device(self, dev):
        try:
            self.model.to(dev)
        except Exception as e:
            log("cihaz değiştirilemedi:", e)
    def close(self):
        del self.model

class KVEmbedder:
    """Kelime vektörlerinin ortalaması (Word2Vec/GloVe: yalnız sözlükteki kelimeler; FastText: alt kelimelerle tüm kelimeler, birim normlu)."""
    def __init__(self, kv, info, fasttext=False):
        self.kv, self.info, self.fasttext = kv, info, fasttext
        self.dim = int(kv.vector_size)
        self.n_params = int(len(kv.key_to_index) * kv.vector_size)
        if fasttext and hasattr(kv, "vectors_ngrams"):
            self.n_params += int(np.prod(kv.vectors_ngrams.shape))
    def _vec(self, text):
        words = text.split()
        if self.fasttext:
            vs = []
            for w in words:
                v = self.kv.get_vector(w, norm=True)
                if np.isfinite(v).all() and np.linalg.norm(v) > 0:
                    vs.append(v)
        else:
            vs = [self.kv[w] for w in words if w in self.kv.key_to_index]
        return np.mean(vs, axis=0) if vs else np.zeros(self.dim, dtype=np.float32)
    def encode(self, texts):
        return np.vstack([self._vec(t) for t in texts]).astype(np.float32)
    def close(self):
        del self.kv

class USEEmbedder:
    def __init__(self, model, info):
        self.model, self.info = model, info
        try:
            self.n_params = int(sum(int(np.prod(v.shape)) for v in model.variables))
        except Exception:
            self.n_params = None
        self.dim = int(self.encode(["test"]).shape[1])
    def encode(self, texts):
        out = []
        for i in range(0, len(texts), 256):
            out.append(self.model(list(texts[i:i + 256])).numpy())
        return np.vstack(out).astype(np.float32)
    def close(self):
        del self.model

class TfidfEmbedder:
    """Sözcüksel temel çizgi: mesajlar + ihtiyaç adları üzerinde 1-2 gram TF-IDF, kosinüs."""
    def __init__(self):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.vec_cls = TfidfVectorizer; self.info = "TF-IDF (sklearn, 1-2 gram, sublinear tf)"; self.n_params = 0; self.dim = 0; self.vec = None
    def fit(self, texts):
        self.vec = self.vec_cls(ngram_range=(1, 2), sublinear_tf=True, min_df=1).fit(texts)
        self.dim = len(self.vec.vocabulary_)
    def encode(self, texts):
        return self.vec.transform(list(texts)).toarray().astype(np.float32)
    def close(self):
        pass

class MockEmbedder:
    """SMOKE_TEST: kelime karma (hash) vektörleri. Gerçek anlam taşımaz, yalnız akışı test eder."""
    def __init__(self, name):
        self.info = f"mock:{name}"; self.seed = int(hashlib.md5(name.encode()).hexdigest(), 16) % 1000
        self.n_params = 0; self.dim = 128 + self.seed % 7
    def encode(self, texts):
        return _hash_embed(texts, dim=self.dim)
    def set_device(self, dev):
        pass
    def close(self):
        pass

# --- Yükleyiciler ----------------------------------------------------------------------------
def glove_txt_to_kv(txt_path, dim=300):
    """glove.840B.300d.txt içindeki boşluklu/bozuk satırlara dayanıklı yükleyici (gensim'in load_word2vec_format'ı bu dosyada hata verir)."""
    from gensim.models import KeyedVectors
    n = 0
    with open(txt_path, "rb") as f:
        for _ in f:
            n += 1
    words, mat, seen, bad = [], np.empty((n, dim), dtype=np.float32), set(), 0
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in tqdm(f, total=n, desc="GloVe okunuyor"):
            parts = line.rstrip("\n").split(" ")
            if len(parts) <= dim:
                bad += 1; continue
            word = " ".join(parts[:-dim])
            if word in seen:
                bad += 1; continue
            try:
                vec = np.asarray(parts[-dim:], dtype=np.float32)
            except ValueError:
                bad += 1; continue
            seen.add(word); words.append(word); mat[len(words) - 1] = vec
    kv = KeyedVectors(vector_size=dim)
    kv.add_vectors(words, mat[:len(words)])
    log(f"GloVe yüklendi: {len(words)} kelime, atlanan satır {bad}")
    return kv

def load_model(name):
    spec = MODEL_SPECS[name]
    if spec["kind"] == "tfidf":
        return TfidfEmbedder()
    if SMOKE_TEST:
        return MockEmbedder(name)
    if spec["kind"] in ("st", "st_tr"):
        return STEmbedder(spec["name"])
    from gensim.models import KeyedVectors
    import gensim.downloader as gapi
    if spec["kind"] == "w2v":
        p = find_existing(MODEL_DIR_CANDIDATES["Word2Vec"])
        if p:
            return KVEmbedder(KeyedVectors.load_word2vec_format(p, binary=True), f"word2vec-google-news-300 ({p})")
        return KVEmbedder(gapi.load("word2vec-google-news-300"), "word2vec-google-news-300 (gensim-data)")
    if spec["kind"] == "glove":
        p = find_existing(MODEL_DIR_CANDIDATES["GloVe"])
        local_kv = os.path.join(LOCAL_SCRATCH, "glove.840B.300d.kv")
        if GLOVE_VARIANT == "840B":
            if p and p.endswith(".kv"):
                return KVEmbedder(KeyedVectors.load(p, mmap="r"), f"glove.840B.300d ({p})")
            if os.path.exists(local_kv):
                return KVEmbedder(KeyedVectors.load(local_kv, mmap="r"), "glove.840B.300d (yerel önbellek)")
            txt = p if (p and p.endswith(".txt")) else None
            if txt is None:
                zp = os.path.join(LOCAL_SCRATCH, "glove.840B.300d.zip")
                for url in ("https://nlp.stanford.edu/data/glove.840B.300d.zip",
                            "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.840B.300d.zip"):
                    try:
                        download_file(url, zp); break
                    except Exception as e:
                        log("GloVe indirme hatası:", url, e)
                if os.path.exists(zp):
                    with zipfile.ZipFile(zp) as z:
                        z.extractall(LOCAL_SCRATCH)
                    txt = os.path.join(LOCAL_SCRATCH, "glove.840B.300d.txt")
            if txt and os.path.exists(txt):
                kv = glove_txt_to_kv(txt, 300)
                kv.save(local_kv)
                if CACHE_MODELS_TO_DRIVE:
                    kv.save(os.path.join(MODEL_DIR, "glove.840B.300d.kv"))
                return KVEmbedder(kv, f"glove.840B.300d ({txt})")
            log("UYARI: GloVe 840B bulunamadı/indirilemedi, 'glove-wiki-gigaword-300' yedeği kullanılıyor")
        return KVEmbedder(gapi.load("glove-wiki-gigaword-300"), "glove-wiki-gigaword-300 (gensim-data, YEDEK)")
    if spec["kind"] == "fasttext":
        from gensim.models.fasttext import load_facebook_vectors
        p = find_existing(MODEL_DIR_CANDIDATES["FastText"])
        if not p:
            gz = download_file("https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz", os.path.join(LOCAL_SCRATCH, "cc.en.300.bin.gz"))
            p = os.path.join(LOCAL_SCRATCH, "cc.en.300.bin")
            if not os.path.exists(p):
                import gzip
                with gzip.open(gz, "rb") as fi, open(p, "wb") as fo:
                    shutil.copyfileobj(fi, fo, length=64 * 1024 * 1024)
            if CACHE_MODELS_TO_DRIVE:
                shutil.copy(p, os.path.join(MODEL_DIR, "cc.en.300.bin"))
        return KVEmbedder(load_facebook_vectors(p), f"cc.en.300.bin ({p})", fasttext=True)
    if spec["kind"] == "use":
        import tensorflow as tf
        try:
            tf.config.set_visible_devices([], "GPU")   # USE CPU'da çalışır; GPU belleği torch modellerine kalır
        except Exception:
            pass
        import tensorflow_hub as hub
        p = find_existing(MODEL_DIR_CANDIDATES["USE"])
        if p:
            return USEEmbedder(hub.load(p), f"universal-sentence-encoder/4 ({p})")
        last = None
        for url in ("https://tfhub.dev/google/universal-sentence-encoder/4",
                    "https://www.kaggle.com/models/google/universal-sentence-encoder/TensorFlow2/universal-sentence-encoder/2"):
            try:
                return USEEmbedder(hub.load(url), f"universal-sentence-encoder/4 ({url})")
            except Exception as e:
                last = e; log("USE yükleme hatası:", url, e)
        raise RuntimeError(f"USE yüklenemedi: {last}")
    raise ValueError(name)

# --- Ölçütler -----------------------------------------------------------------------------
def top_modes(counter):
    if not counter:
        return set()
    m = max(counter.values())
    return {k for k, v in counter.items() if v == m}

def district_metric_paper(df, preds):
    """Makaledeki ölçüt: ilçede en sık tahmin edilen ihtiyaç(lar) ile en sık gerçek ihtiyaç(lar) kesişiyorsa 1, değilse 0."""
    acc = {}
    for d, idx in df.groupby("district").indices.items():
        actual = Counter(x for i in idx for x in df["labels_en"].iat[i])
        predicted = Counter(preds[i] for i in idx if preds[i] is not None)
        if not actual or not predicted:
            continue
        acc[d] = 1 if (top_modes(actual) & top_modes(predicted)) else 0
    return acc

def district_top_pred(df, preds):
    out = {}
    for d, idx in df.groupby("district").indices.items():
        c = Counter(preds[i] for i in idx if preds[i] is not None)
        if c:
            out[d] = c.most_common(1)[0]
    return out

def bootstrap_ci(values, n_boot=BOOTSTRAP_N, seed=SEED):
    v = np.asarray(values, dtype=float)
    if len(v) == 0:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    means = rng.choice(v, size=(n_boot, len(v)), replace=True).mean(axis=1)
    return (float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5)))

def evaluate_predictions(df, top1, top3, tag):
    labels = df["labels_en"].tolist()
    hit1 = np.array([(p is not None) and (p in L) for p, L in zip(top1, labels)], dtype=float)
    hit3 = np.array([any((p in L) for p in P if p is not None) for P, L in zip(top3, labels)], dtype=float)
    d_acc = district_metric_paper(df, top1)
    d_vals = list(d_acc.values())
    hit1_by_d = pd.Series(hit1, index=df["district"].values).groupby(level=0).mean()
    ci_d, ci_m = bootstrap_ci(d_vals), bootstrap_ci(hit1)
    res = {"ilce_dogruluk_makale": float(np.mean(d_vals)) if d_vals else float("nan"), "ilce_ga_alt": ci_d[0], "ilce_ga_ust": ci_d[1],
           "hit1": float(hit1.mean()), "hit1_ga_alt": ci_m[0], "hit1_ga_ust": ci_m[1],
           "hit1_ilce_ort": float(hit1_by_d.mean()), "hit3": float(hit3.mean()),
           "n_ilce": len(d_vals), "n_mesaj": len(df), "tahminsiz": int(sum(p is None for p in top1))}
    dtp = district_top_pred(df, top1)
    rows = []
    for d, idx in df.groupby("district").indices.items():
        actual = Counter(x for i in idx for x in df["labels_en"].iat[i])
        rows.append({"İlçe": d, "mesaj": len(idx), "gercek_en_sik": ", ".join(sorted(top_modes(actual))),
                     "tahmin_en_sik": dtp.get(d, ("", 0))[0], "tahmin_sayi": dtp.get(d, ("", 0))[1],
                     "dogru_makale_olcutu": d_acc.get(d, np.nan), "hit1_ort": float(hit1_by_d.get(d, np.nan))})
    pd.DataFrame(rows).to_csv(os.path.join(DIRS["preds"], f"ilce_{tag}.csv"), index=False, encoding="utf-8-sig")
    pd.DataFrame({"id": df["id"].values, "district": df["district"].values, "gercek": [";".join(L) for L in labels],
                  "top1": top1, "top2": [P[1] if len(P) > 1 else None for P in top3], "top3": [P[2] if len(P) > 2 else None for P in top3],
                  "hit1": hit1.astype(int)}).to_csv(os.path.join(DIRS["preds"], f"mesaj_{tag}.csv"), index=False, encoding="utf-8-sig")
    return res, hit1

def plain_text_tr(text):
    return " ".join(tr_lower(LETTERS_RE.sub(" ", str(text))).split())

def run_embedding_config(embedder, df, text_type, lang="en"):
    """Bir model × metin türü için top-1/top-3 tahminler ve süreler. lang='tr': Türkçe metin + Türkçe ihtiyaç adları."""
    if lang == "tr":
        msgs = df["Message"].apply(plain_text_tr).tolist()
        needs = [plain_text_tr(n) for n in NEEDS_TR]
        need_labels = [TR2EN[n] for n in NEEDS_TR]
    else:
        msgs = df["text_" + text_type].tolist()
        needs = need_texts(text_type)
        need_labels = list(EN_LABELS)
    if hasattr(embedder, "fit"):
        embedder.fit(msgs + needs)
    t0 = time.perf_counter(); M = embedder.encode(msgs); t_batch = (time.perf_counter() - t0) / max(1, len(msgs)) * 1000
    Nv = embedder.encode(needs)
    m_norm = np.linalg.norm(M, axis=1, keepdims=True); n_norm = np.linalg.norm(Nv, axis=1, keepdims=True)
    S = (M / (m_norm + 1e-9)) @ (Nv / (n_norm + 1e-9)).T
    zero = (m_norm[:, 0] == 0)
    order = np.argsort(-S, axis=1)
    top1, top3 = [], []
    for i in range(len(msgs)):
        if zero[i]:
            top1.append(None); top3.append([]); continue
        uniq = list(dict.fromkeys(need_labels[j] for j in order[i, :6]))[:3]   # aynı İngilizce etikete giden Türkçe adları birleştir
        top1.append(uniq[0]); top3.append(uniq)
    # tek mesaj gecikmesi: tüm yerel modeller için aynı cihazda (varsayılan CPU)
    if LATENCY_ON_CPU and hasattr(embedder, "set_device"):
        embedder.set_device("cpu")
    sample = random.Random(SEED).sample(msgs, min(30, len(msgs)))
    lat = []
    for s in sample:
        t0 = time.perf_counter(); embedder.encode([s]); lat.append((time.perf_counter() - t0) * 1000)
    if LATENCY_ON_CPU and hasattr(embedder, "set_device"):
        embedder.set_device(DEVICE)
    return top1, top3, t_batch, float(np.median(lat))

def baseline_predictions(kind, df, seed=SEED):
    rng = random.Random(seed)
    if kind == "random":
        top3 = [rng.sample(EN_LABELS, 3) for _ in range(len(df))]
        return [t[0] for t in top3], top3
    freq = Counter(L[0] for L in df["labels_en"] if L)
    top = [n for n, _ in freq.most_common(3)]
    return [top[0]] * len(df), [top] * len(df)

# --- Haritalar -------------------------------------------------------------------------------
import folium
def need_color_map(needs):
    cmap = plt.get_cmap("tab20").colors + plt.get_cmap("tab20b").colors + plt.get_cmap("tab20c").colors
    return {n: matplotlib.colors.to_hex(cmap[i % len(cmap)]) for i, n in enumerate(sorted(set(needs)))}

def save_folium_map(dtp, path, title):
    m = folium.Map(location=[41.05, 28.95], zoom_start=10, tiles="cartodbpositron")
    colors = need_color_map([v[0] for v in dtp.values()])
    for d, (need, cnt) in dtp.items():
        lat, lon = DISTRICT_COORDS[d]
        folium.CircleMarker(location=[lat, lon], radius=9, color=colors[need], fill=True, fill_color=colors[need], fill_opacity=0.85,
                            popup=f"{d}: {EN2TR.get(need, need)} ({cnt})", tooltip=f"{d}: {EN2TR.get(need, need)}").add_to(m)
    legend = "<br>".join(f'<i style="background:{c};width:12px;height:12px;border-radius:50%;display:inline-block"></i> {EN2TR.get(n, n)}' for n, c in colors.items())
    m.get_root().html.add_child(folium.Element(
        f'<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;padding:8px;border:1px solid grey;font-size:12px;max-height:400px;overflow:auto"><b>{title}</b><br>{legend}</div>'))
    m.save(path)

# İlçe poligonları (varsa) bir kez yüklenir
from matplotlib.patches import Polygon as MplPolygon
DISTRICT_POLYS = {}
if GEOJSON_PATH and os.path.exists(GEOJSON_PATH):
    with open(GEOJSON_PATH, encoding="utf-8") as fh:
        for f in json.load(fh)["features"]:
            g = f["geometry"]; rings = g["coordinates"] if g["type"] == "Polygon" else [r for poly in g["coordinates"] for r in poly]
            DISTRICT_POLYS[f["properties"]["ilce"]] = [np.array(r) for r in rings if len(r) >= 3]

def draw_static_map(ax, dtp, title):
    colors = need_color_map([v[0] for v in dtp.values()])
    for d in DISTRICTS:
        lat, lon = DISTRICT_COORDS[d]
        fc = colors[dtp[d][0]] if d in dtp else "white"
        if d in DISTRICT_POLYS:
            for ring in DISTRICT_POLYS[d]:
                ax.add_patch(MplPolygon(ring[:, :2], closed=True, facecolor=fc, edgecolor="#444444", linewidth=0.35, alpha=0.9, zorder=2))
            ax.annotate(d, (lon, lat), fontsize=3.6, ha="center", va="center", zorder=4)
        else:
            ax.scatter(lon, lat, s=60 if d in dtp else 22, color=fc, edgecolor="black" if d in dtp else "grey", linewidth=0.4, zorder=3)
            ax.annotate(d, (lon, lat), fontsize=3.8, xytext=(2, 2), textcoords="offset points", zorder=4)
    if DISTRICT_POLYS:
        ax.set_xlim(27.95, 29.95); ax.set_ylim(40.78, 41.60)
    else:
        ax.set_xlim(28.15, 29.75); ax.set_ylim(40.74, 41.30)
    ax.set_title(title, fontsize=8.5); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_linewidth(0.5)
    ax.set_aspect(1 / math.cos(math.radians(41)))
    handles = [plt.Line2D([], [], marker="o", color=c, linestyle="", markersize=4.5, label=EN2TR.get(n, n)) for n, c in colors.items()]
    ax.legend(handles=handles, fontsize=4.6, loc="upper center", bbox_to_anchor=(0.5, -0.02), frameon=False, ncol=4,
              handletextpad=0.3, columnspacing=0.8, borderaxespad=0.0)


In [ ]:
# =============================================================================
# 6. BDM DEĞERLENDİRMESİ (Tablo 5): sıfır atış istem, makaledeki protokolün API karşılığı
# =============================================================================
import difflib
LLM_PROMPT_VERSION = "v1"
SYSTEM_PROMPT_LLM = ("You are a disaster response assistant. Based on the social media message of an earthquake victim, "
                     "identify the single most urgently needed relief item from the following list of disaster relief needs:\n"
                     + "\n".join(f"- {n}" for n in EN_LABELS) +
                     "\nFor each message respond only with the name of the need, exactly as written in the list.")
LLM_SCHEMA = {"type": "object", "properties": {"predictions": {"type": "array", "items": {
    "type": "object", "properties": {"id": {"type": "integer"}, "need": {"type": "string", "enum": EN_LABELS}},
    "required": ["id", "need"], "additionalProperties": False}}}, "required": ["predictions"], "additionalProperties": False}
with open(os.path.join(DIRS["logs"], "prompt_bdm.txt"), "w", encoding="utf-8") as f:
    f.write(SYSTEM_PROMPT_LLM)

def strip_additional(schema):
    if isinstance(schema, dict):
        return {k: strip_additional(v) for k, v in schema.items() if k != "additionalProperties"}
    if isinstance(schema, list):
        return [strip_additional(x) for x in schema]
    return schema

_LABEL_LOWER = {l.lower(): l for l in EN_LABELS}
def normalize_need(x):
    if not isinstance(x, str):
        return None
    k = x.strip().lower()
    if k in _LABEL_LOWER:
        return _LABEL_LOWER[k]
    m = difflib.get_close_matches(k, list(_LABEL_LOWER), n=1, cutoff=0.6)
    return _LABEL_LOWER[m[0]] if m else None

def build_llm_user(batch):
    if len(batch) == 1:
        return f"Message (id {batch[0][0]}): {batch[0][1]}\nReturn JSON {{\"predictions\":[{{\"id\":{batch[0][0]},\"need\":...}}]}}."
    return "Messages:\n" + "\n".join(f"[id {i}] {t}" for i, t in batch) + "\nReturn JSON {\"predictions\":[{\"id\":..,\"need\":..}, ...]} with one entry per message id."

# --- Hız sınırı (429) ve kota yönetimi -------------------------------------------------------
class LLMQuotaExhausted(Exception):
    """Günlük istek sınırı ya da kredi/kota bitti: bu model için yeni istek göndermenin anlamı yok."""

class _Pacer:
    """Sağlayıcı başına istek aralığı (RPM) ve 429 sonrası ortak bekleme."""
    def __init__(self, rpm=None):
        self.lock = threading.Lock(); self.next_ok = 0.0; self.min_interval = (60.0 / rpm) if rpm else 0.0
    def wait(self):
        with self.lock:
            now = time.monotonic(); delay = max(0.0, self.next_ok - now)
            self.next_ok = max(now, self.next_ok) + self.min_interval
        if delay > 0:
            time.sleep(delay)
    def hold(self, seconds):
        with self.lock:
            self.next_ok = max(self.next_ok, time.monotonic() + seconds)

_PACERS = {}
def pacer_for(provider):
    if provider not in _PACERS:
        _PACERS[provider] = _Pacer(LLM_EVAL_RPM_BY_PROVIDER.get(provider))
    return _PACERS[provider]

def _retry_after_seconds(msg):
    m = re.search(r"try again in\s*([\d.]+)\s*(ms|s|m)\b", msg, flags=re.I)
    if not m:
        return None
    v = float(m.group(1)); u = m.group(2).lower()
    return v / 1000 if u == "ms" else (v * 60 if u == "m" else v)

def classify_api_error(e):
    """'quota' (kredi/kota bitti), 'daily' (günlük istek sınırı), 'rate' (dakikalık sınır), 'other'."""
    msg = str(e); low = msg.lower()
    is429 = ("429" in msg) or ("rate limit" in low) or ("resource_exhausted" in low) or ("quota" in low) or type(e).__name__ in ("RateLimitError", "ResourceExhausted")
    if any(k in low for k in ("insufficient_quota", "no credits remaining", "credit_balance_exhausted", "spend_limit_exceeded", "usage_limit_exceeded", "billing")):
        return "quota"
    if is429 and any(k in low for k in ("per day", "requests per day", "tokens per day", "rpd", "perday", "daily")):
        return "daily"
    if is429:
        return "rate"
    return "other"

def llm_call_paced(provider, model, batch, max_attempts=8):
    """llm_call'ı hız sınırına uyarak çağırır: dakikalık 429'da bekleyip yeniden dener; kota/günlük sınırda LLMQuotaExhausted fırlatır."""
    pacer = pacer_for(provider)
    for attempt in range(max_attempts):
        if not SMOKE_TEST:
            pacer.wait()
        try:
            _API_TL.last_ms = float("nan")
            preds = llm_call(provider, model, batch)
            return preds, float(getattr(_API_TL, "last_ms", float("nan")))
        except LLMQuotaExhausted:
            raise
        except BudgetExceeded:
            raise
        except Exception as e:
            kind = classify_api_error(e)
            if kind in ("quota", "daily"):
                raise LLMQuotaExhausted(f"{kind}: {str(e)[:300]}")
            wait_s = _retry_after_seconds(str(e))
            if kind == "rate":
                wait_s = wait_s if wait_s is not None else min(60.0, 3.0 * (2 ** attempt))
                pacer.hold(wait_s + 0.5)
                time.sleep(wait_s + 0.5 + random.random())
                continue
            if attempt >= 2:
                raise
            time.sleep(2.0 * (attempt + 1))
    raise RuntimeError(f"{model}: {max_attempts} denemede hız sınırı aşılamadı")

_PROVIDER_OPTS = {"openai_reasoning": "minimal", "google_thinking_budget": 0}
OTHER_USAGE = defaultdict(lambda: Counter())
def _note_usage(model, u):
    """OpenAI / Google token sayaçları. Faturalanan çıktı: OpenAI'de completion_tokens (düşünme dâhil), Gemini'de candidates + thoughts."""
    if u is None:
        return
    with USAGE_LOCK:
        c = OTHER_USAGE[model]; c["calls"] += 1
        if hasattr(u, "prompt_tokens"):            # OpenAI
            det = getattr(u, "completion_tokens_details", None); pdt = getattr(u, "prompt_tokens_details", None)
            out = int(getattr(u, "completion_tokens", 0) or 0); think = int(getattr(det, "reasoning_tokens", 0) or 0) if det is not None else 0
            c["input_tokens"] += int(getattr(u, "prompt_tokens", 0) or 0); c["cached_input_tokens"] += int(getattr(pdt, "cached_tokens", 0) or 0) if pdt is not None else 0
            c["output_tokens"] += out - think; c["thinking_tokens"] += think; c["billed_output_tokens"] += out
        else:                                       # Google
            out = int(getattr(u, "candidates_token_count", 0) or 0); think = int(getattr(u, "thoughts_token_count", 0) or 0)
            c["input_tokens"] += int(getattr(u, "prompt_token_count", 0) or 0); c["cached_input_tokens"] += int(getattr(u, "cached_content_token_count", 0) or 0)
            c["output_tokens"] += out; c["thinking_tokens"] += think; c["billed_output_tokens"] += out + think

def cost_from_counter(model, u):
    """Sayaç sözlüğünden tahmini maliyet (USD, liste fiyatı)."""
    p = PRICES_USD_PER_MTOK.get(model)
    if not p or not u:
        return float("nan")
    if model.startswith("claude"):
        return (u.get("input_tokens", 0) * p[0] + u.get("output_tokens", 0) * p[1] + u.get("cache_read_input_tokens", 0) * p[2] + u.get("cache_creation_input_tokens", 0) * p[3]) / 1e6
    cached = u.get("cached_input_tokens", 0)
    return ((u.get("input_tokens", 0) - cached) * p[0] + cached * p[2] + u.get("billed_output_tokens", u.get("output_tokens", 0)) * p[1]) / 1e6

def other_usage_summary(model):
    u = OTHER_USAGE.get(model, {}); n = max(1, u.get("calls", 0)); cost = cost_from_counter(model, u)
    return (f"{u.get('calls', 0)} çağrı | çağrı başına giriş {u.get('input_tokens', 0) / n:.0f} (önbellek {u.get('cached_input_tokens', 0) / n:.0f}), çıktı {u.get('output_tokens', 0) / n:.0f} token "
            f"(düşünme {u.get('thinking_tokens', 0) / n:.0f})" + (f" | tahmini ${cost:.2f}" if cost == cost else ""))
def other_usage_cost(model):
    return cost_from_counter(model, OTHER_USAGE.get(model, {}))

_OPENAI_CLIENT = _GOOGLE_CLIENT = None
def llm_call(provider, model, batch):
    """batch: [(id, text), ...] -> {id: need}"""
    global _OPENAI_CLIENT, _GOOGLE_CLIENT
    user = build_llm_user(batch)
    if SMOKE_TEST:
        _API_TL.last_ms = 0.0
        rng = random.Random(batch[0][0] + hash(model) % 1000)
        return {i: rng.choice(EN_LABELS) for i, _ in batch}
    if provider == "anthropic":
        data = claude_json(SYSTEM_PROMPT_LLM, user, LLM_SCHEMA, model, max_tokens=150 * len(batch) + 1500, temperature=0.0, effort=LLM_EVAL_EFFORT)
    elif provider == "openai":
        if _OPENAI_CLIENT is None:
            from openai import OpenAI
            _OPENAI_CLIENT = OpenAI(api_key=OPENAI_API_KEY, max_retries=0)
        kwargs = dict(model=model, messages=[{"role": "system", "content": SYSTEM_PROMPT_LLM}, {"role": "user", "content": user}],
                      response_format={"type": "json_schema", "json_schema": {"name": "preds", "strict": True, "schema": LLM_SCHEMA}})
        if _PROVIDER_OPTS["openai_reasoning"] and (model.startswith("gpt-5") or model.startswith("o")):
            kwargs["reasoning_effort"] = _PROVIDER_OPTS["openai_reasoning"]   # en düşük düşünme: ucuz ve hızlı (Claude'daki effort="low" karşılığı)
        try:
            _t0 = time.perf_counter(); r = _OPENAI_CLIENT.chat.completions.create(**kwargs); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
        except Exception as e:
            if "reasoning_effort" in kwargs and ("reasoning" in str(e).lower() or "unsupported" in str(e).lower()) and "429" not in str(e):
                log(f"{model}: reasoning_effort kabul edilmedi ({str(e)[:100]}); parametresiz devam"); _PROVIDER_OPTS["openai_reasoning"] = None
                kwargs.pop("reasoning_effort"); _t0 = time.perf_counter(); r = _OPENAI_CLIENT.chat.completions.create(**kwargs); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
            else:
                raise
        _note_usage(model, getattr(r, "usage", None))
        data = extract_json(r.choices[0].message.content)
    elif provider == "google":
        if _GOOGLE_CLIENT is None:
            from google import genai
            _GOOGLE_CLIENT = genai.Client(api_key=GOOGLE_API_KEY)
        cfg = {"system_instruction": SYSTEM_PROMPT_LLM, "response_mime_type": "application/json", "response_schema": strip_additional(LLM_SCHEMA), "temperature": 0.0}
        if _PROVIDER_OPTS["google_thinking_budget"] is not None:
            cfg["thinking_config"] = {"thinking_budget": _PROVIDER_OPTS["google_thinking_budget"]}   # 0: düşünme kapalı (2.5 Flash)
        try:
            _t0 = time.perf_counter(); r = _GOOGLE_CLIENT.models.generate_content(model=model, contents=user, config=cfg); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
        except Exception as e:
            if "thinking_config" in cfg and "thinking" in str(e).lower() and "429" not in str(e):
                log(f"{model}: thinking_budget kabul edilmedi ({str(e)[:100]}); parametresiz devam"); _PROVIDER_OPTS["google_thinking_budget"] = None
                cfg.pop("thinking_config"); _t0 = time.perf_counter(); r = _GOOGLE_CLIENT.models.generate_content(model=model, contents=user, config=cfg); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
            else:
                raise
        _note_usage(model, getattr(r, "usage_metadata", None))
        data = extract_json(r.text)
    else:
        raise ValueError(provider)
    return {int(p["id"]): normalize_need(p.get("need")) for p in data.get("predictions", [])}

def llm_cache_path(provider, model, set_name):
    sig = hashlib.md5(json.dumps([set_name, GEN_SIGNATURE, LLM_EVAL_MESSAGES_PER_CALL, LLM_PROMPT_VERSION, FIX_NEED_TRANSLATIONS]).encode()).hexdigest()[:8]
    return os.path.join(DIRS["llm_cache"], f"{set_name}_{provider}_{model}_{sig}.jsonl")

def llm_proto_tag(provider, model):
    """Önbellek satırına yazılan protokol etiketi (denetim için)."""
    if provider == "anthropic":
        return f"effort={LLM_EVAL_EFFORT};thinking={thinking_param(model, THINKING_MODE)}"
    if provider == "openai":
        return f"reasoning_effort={_PROVIDER_OPTS['openai_reasoning']}"
    if provider == "google":
        return f"thinking_budget={_PROVIDER_OPTS['google_thinking_budget']};temperature=0"
    return ""

def llm_evaluate(provider, model, dfe, set_name):
    """Dönen: top1 listesi, medyan ms/mesaj, başarısız çağrı sayısı, tahminsiz mesaj sayısı, durma nedeni (None: tamamlandı)."""
    ids, texts = dfe["id"].tolist(), dfe["English"].tolist()
    cache_path = llm_cache_path(provider, model, set_name)
    cache = {}
    if os.path.exists(cache_path):
        with open(cache_path, encoding="utf-8") as f:
            for line in f:
                try:
                    r = json.loads(line)
                    if r.get("need") is not None:
                        cache[int(r["id"])] = r
                except Exception:
                    pass
    todo = [(i, t) for i, t in zip(ids, texts) if i not in cache]
    batches = [todo[k:k + LLM_EVAL_MESSAGES_PER_CALL] for k in range(0, len(todo), LLM_EVAL_MESSAGES_PER_CALL)]
    if cache:
        log(f"{model} ({set_name}): {len(cache)} mesaj önbellekten, {len(todo)} mesaj ({len(batches)} istek) API'den")
    lock = threading.Lock(); n_fail = 0; n_ok = 0; stop_reason = None
    workers = LLM_EVAL_WORKERS_BY_PROVIDER.get(provider, LLM_EVAL_WORKERS) if provider != "anthropic" else LLM_EVAL_WORKERS
    proto = llm_proto_tag(provider, model)
    def work(batch):
        preds, api_ms = llm_call_paced(provider, model, batch); ms = api_ms / len(batch)   # yalnız API süresi (hız sınırı beklemesi hariç)
        return [{"id": i, "need": preds.get(i), "ms": ms, "proto": proto} for i, _ in batch]
    if batches:
        ex = ThreadPoolExecutor(max_workers=workers)
        fout = open(cache_path, "a", encoding="utf-8")
        futs = [ex.submit(work, b) for b in batches]
        bar = tqdm(total=len(futs), desc=f"BDM {model} ({set_name})")
        try:
            for fut in as_completed(futs):
                try:
                    rows = fut.result()
                except (LLMQuotaExhausted, BudgetExceeded) as e:
                    if stop_reason is None:
                        stop_reason = str(e)[:300]
                        log(f"DURDURULDU ({model}, {set_name}): {stop_reason}")
                        ex.shutdown(wait=False, cancel_futures=True)
                    n_fail += 1
                except Exception as e:
                    n_fail += 1; log(f"BDM çağrısı başarısız ({model}): {str(e)[:200]}")
                else:
                    n_ok += 1
                    with lock:
                        for r in rows:
                            if r["need"] is not None:
                                cache[r["id"]] = r
                            fout.write(json.dumps(r, ensure_ascii=False) + "\n")
                bar.update(1); bar.set_postfix({"başarılı": n_ok, "başarısız": n_fail})
                if stop_reason is not None and all(f.done() or f.cancelled() for f in futs):
                    break
        finally:
            bar.close(); fout.close(); ex.shutdown(wait=False, cancel_futures=True)
    top1 = [cache.get(i, {}).get("need") for i in ids]
    ms = [cache[i]["ms"] for i in ids if i in cache and cache[i].get("ms") is not None]
    n_missing = len(ids) - sum(p is not None for p in top1)
    return top1, (float(np.median(ms)) if ms else float("nan")), n_fail, n_missing, stop_reason

# --- Gecikme ve maliyet ölçümü: tüm BDM'ler için AYNI paylaşımlar, sıralı (tek iş parçacığı) çağrılar ---------------
def _usage_snapshot(provider, model):
    src = USAGE if provider == "anthropic" else OTHER_USAGE
    return dict(src.get(model, {}))

def _usage_delta(before, after):
    return {k: after.get(k, 0) - before.get(k, 0) for k in set(before) | set(after)}

def llm_latency_benchmark(provider, model, dfe, n_batch_req, n_single_req, force=False):
    """Toplu (LLM_EVAL_MESSAGES_PER_CALL paylaşım/istek) ve tekil (1 paylaşım/istek) kipte gecikme ve token kullanımı.
    Paylaşımlar sabit tohumla seçilir (tüm modellerde aynı). Hız sınırı beklemesi ölçüme girmez. Sonuç Drive'da önbelleğe alınır."""
    mpc = LLM_EVAL_MESSAGES_PER_CALL
    bench_dir = os.path.join(DATA_DIR, "bdm_bench"); os.makedirs(bench_dir, exist_ok=True)
    sig = hashlib.md5(json.dumps([GEN_SIGNATURE, mpc, n_batch_req, n_single_req, LLM_PROMPT_VERSION, llm_proto_tag(provider, model)]).encode()).hexdigest()[:8]
    out_path = os.path.join(bench_dir, f"{provider}_{model}_{sig}.json")
    if os.path.exists(out_path) and not force:
        return json.load(open(out_path, encoding="utf-8"))
    rng = np.random.default_rng(SEED + 2026)
    pool = rng.choice(dfe["id"].values, size=min(len(dfe), n_batch_req * mpc + n_single_req), replace=False)
    text = dict(zip(dfe["id"].tolist(), dfe["English"].tolist()))
    batch_reqs = [[(int(i), text[int(i)]) for i in pool[k * mpc:(k + 1) * mpc]] for k in range(n_batch_req)]
    single_reqs = [[(int(i), text[int(i)])] for i in pool[n_batch_req * mpc:n_batch_req * mpc + n_single_req]]
    res = {"model": model, "saglayici": provider, "protokol": llm_proto_tag(provider, model), "tamam": True, "neden": None}
    for mode, reqs in (("toplu", batch_reqs), ("tekil", single_reqs)):
        times, n_posts, n_ok = [], 0, 0
        before = _usage_snapshot(provider, model)
        for req in tqdm(reqs, desc=f"Ölçüm {model} ({mode})", leave=False):
            try:
                preds, api_ms = llm_call_paced(provider, model, req)
            except (LLMQuotaExhausted, BudgetExceeded) as e:
                res["tamam"] = False; res["neden"] = str(e)[:200]; log(f"Ölçüm durdu ({model}, {mode}): {res['neden']}"); break
            except Exception as e:
                log(f"Ölçüm çağrısı başarısız ({model}, {mode}): {str(e)[:150]}"); continue
            times.append(api_ms); n_posts += len(req); n_ok += sum(preds.get(i) is not None for i, _ in req)
        u = _usage_delta(before, _usage_snapshot(provider, model))
        cost = cost_from_counter(model, u)
        t = np.array([x for x in times if x == x], dtype=float); k = max(1, len(times))
        res[mode] = {"istek": len(times), "paylasim": n_posts, "yanitlanan": n_ok,
                     "ms_istek_medyan": float(np.median(t)) if len(t) else float("nan"), "ms_istek_p90": float(np.percentile(t, 90)) if len(t) else float("nan"),
                     "ms_paylasim_medyan": float(np.median(t)) / (mpc if mode == "toplu" else 1) if len(t) else float("nan"),
                     "giris_token_istek": u.get("input_tokens", 0) / k, "cikis_token_istek": (u.get("billed_output_tokens", u.get("output_tokens", 0))) / k,
                     "dusunme_token_istek": u.get("thinking_tokens", 0) / k,
                     "usd": cost, "usd_1000_paylasim": cost / n_posts * 1000 if n_posts else float("nan")}
        if not res["tamam"]:
            break
    if res["tamam"]:
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(res, f, ensure_ascii=False, indent=1)
    return res

def run_latency_benchmarks(models, dfe):
    rows = []
    for provider, model in models:
        key = {"anthropic": ANTHROPIC_API_KEY, "openai": OPENAI_API_KEY, "google": GOOGLE_API_KEY}.get(provider)
        if not key and not SMOKE_TEST:
            log(f"Ölçüm: {model} atlandı (anahtar yok)"); continue
        r = llm_latency_benchmark(provider, model, dfe, BENCH_BATCH_REQUESTS, BENCH_SINGLE_REQUESTS, force=BENCH_FORCE)
        row = {"Model": model, "Sağlayıcı": provider, "Protokol": r.get("protokol"), "Tamam": r.get("tamam")}
        for mode in ("toplu", "tekil"):
            m = r.get(mode, {})
            row.update({f"{mode}_istek": m.get("istek"), f"{mode}_ms_istek_medyan": m.get("ms_istek_medyan"), f"{mode}_ms_istek_p90": m.get("ms_istek_p90"),
                        f"{mode}_ms_paylasim": m.get("ms_paylasim_medyan"), f"{mode}_giris_token": m.get("giris_token_istek"),
                        f"{mode}_cikis_token": m.get("cikis_token_istek"), f"{mode}_dusunme_token": m.get("dusunme_token_istek"),
                        f"{mode}_usd_1000_paylasim": m.get("usd_1000_paylasim")})
        rows.append(row)
        def _f(x, nd=0):
            return "?" if x is None or x != x else f"{x:.{nd}f}"
        log(f"Ölçüm {model}: toplu {_f(row.get('toplu_ms_paylasim'))} ms/paylaşım (istek medyanı {_f(row.get('toplu_ms_istek_medyan'))} ms), "
            f"tekil {_f(row.get('tekil_ms_istek_medyan'))} ms, {_f(row.get('toplu_usd_1000_paylasim'), 3)} $/1.000 paylaşım" + ("" if r.get("tamam") else " (EKSİK)"))
    if rows:
        t = pd.DataFrame(rows)
        save_table(t, "tablo6b_bdm_gecikme_maliyet_olcumu")
        print(t.round(3).to_string(index=False))
    return rows


## Revizyon deneyleri
Aşağıdaki hücreler yukarıdaki tanımları kullanır. Her hücre kendi sonuçlarını `runs/revizyon/results/revizyon/` ve `runs/revizyon/results/tahminler/` altına yazar; tüm sonuçlar `revizyon_tum_sonuclar.csv` dosyasında birikir.

In [ ]:
# =============================================================================
# R1. ORTAK ALTYAPI: sağlayıcıdan bağımsız JSON çağrısı, bütçe, önbellek, ölçütler, kayıt
# =============================================================================
import threading, zlib
from concurrent.futures import CancelledError

REV_CACHE = os.path.join(DATA_DIR, "revizyon_cache"); os.makedirs(REV_CACHE, exist_ok=True)
DIRS["rev"] = os.path.join(DIRS["results"], "revizyon"); os.makedirs(DIRS["rev"], exist_ok=True)
REV_KEYS = {"anthropic": ANTHROPIC_API_KEY, "openai": OPENAI_API_KEY, "google": GOOGLE_API_KEY}

def has_key(provider):
    return SMOKE_TEST or bool(REV_KEYS.get(provider))

def stable_seed(*parts):
    """Süreçten bağımsız tohum (Python'un hash() işlevi her oturumda değişir)."""
    return zlib.crc32("|".join(map(str, parts)).encode("utf-8")) % (2 ** 31)

# --- Bütçe: özgün denetim yalnız Anthropic'i sayıyordu; burada tüm sağlayıcılar sayılır -------------
def total_cost_all():
    c = total_cost_usd()
    for m in list(OTHER_USAGE):
        v = other_usage_cost(m)
        if v == v:
            c += v
    return float(c)

def check_budget():
    if BUDGET_GUARD_USD is not None and total_cost_all() >= BUDGET_GUARD_USD:
        if not _BUDGET_LOGGED[0]:
            _BUDGET_LOGGED[0] = True
            log(f"BÜTÇE FRENİ: tahmini toplam maliyet ${total_cost_all():.2f} >= {BUDGET_GUARD_USD} $. Yeni API çağrısı yapılmıyor; "
                f"BUDGET_GUARD_USD'yi artırıp hücreyi yeniden çalıştırın (yanıtlar önbellekte).")
        raise BudgetExceeded("bütçe aşıldı")

# --- SMOKE_TEST için şemaya uyan sahte yanıtlar ----------------------------------------------------------
def _mock_value(sch, rng):
    t = sch.get("type")
    if "enum" in sch:
        return rng.choice(sch["enum"])
    if t == "boolean":
        return rng.random() < 0.6
    if t == "integer":
        return rng.randint(0, 9)
    if t == "array":
        return [_mock_value(sch.get("items", {}), rng) for _ in range(rng.randint(1, 3))]
    if t == "object":
        return {k: _mock_value(v, rng) for k, v in sch.get("properties", {}).items()}
    return "x"

def _mock_json(schema, user_text):
    rng = random.Random(stable_seed(user_text))
    props = schema.get("properties", {})
    lines = [l for l in user_text.splitlines() if l.startswith("{")]
    if "records" in props:                                   # üretim
        return mock_generate([json.loads(l) for l in lines])
    if "results" in props:                                   # kalite kontrol
        return {"results": [{"id": json.loads(l)["id"], "all_needs_expressed": rng.random() < 0.92, "missing": [], "extra": [],
                             "district_ok": rng.random() < 0.97} for l in lines]}
    if "predictions" in props:                               # sınıflandırma
        item = props["predictions"]["items"]["properties"]
        out = []
        for i in [int(x) for x in re.findall(r"[\[(]id (\d+)[\])]", user_text)]:
            d = {"id": i}
            for k, sub in item.items():
                if k != "id":
                    d[k] = _mock_value(sub, rng)
            out.append(d)
        return {"predictions": out}
    return _mock_value(schema, rng)

# --- Sağlayıcıdan bağımsız yapılandırılmış JSON çağrısı ----------------------------------------------------
def api_json(provider, model, system_text, user_text, schema, max_tokens=4000, temperature=0.0, effort=None):
    """Anthropic: makaledeki claude_json (effort=low, en düşük düşünme). OpenAI: katı JSON şeması, reasoning_effort=minimal,
    sıcaklık verilmez. Google: yanıt şeması, düşünme bütçesi 0; temperature=None ise sıcaklık verilmez (üretim)."""
    global _OPENAI_CLIENT, _GOOGLE_CLIENT
    if SMOKE_TEST:
        _API_TL.last_ms = 0.0
        return _mock_json(schema, user_text)
    check_budget()
    if provider == "anthropic":
        return claude_json(system_text, user_text, schema, model, max_tokens=max_tokens, temperature=temperature,
                           effort=effort or LLM_EVAL_EFFORT)
    if provider == "openai":
        if _OPENAI_CLIENT is None:
            from openai import OpenAI
            _OPENAI_CLIENT = OpenAI(api_key=OPENAI_API_KEY, max_retries=0)
        kwargs = dict(model=model, messages=[{"role": "system", "content": system_text}, {"role": "user", "content": user_text}],
                      response_format={"type": "json_schema", "json_schema": {"name": "out", "strict": True, "schema": schema}})
        if _PROVIDER_OPTS["openai_reasoning"] and (model.startswith("gpt-5") or model.startswith("o")):
            kwargs["reasoning_effort"] = _PROVIDER_OPTS["openai_reasoning"]
        try:
            _t0 = time.perf_counter(); r = _OPENAI_CLIENT.chat.completions.create(**kwargs); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
        except Exception as e:
            if "reasoning_effort" in kwargs and ("reasoning" in str(e).lower() or "unsupported" in str(e).lower()) and "429" not in str(e):
                log(f"{model}: reasoning_effort kabul edilmedi; parametresiz devam"); _PROVIDER_OPTS["openai_reasoning"] = None
                kwargs.pop("reasoning_effort")
                _t0 = time.perf_counter(); r = _OPENAI_CLIENT.chat.completions.create(**kwargs); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
            else:
                raise
        _note_usage(model, getattr(r, "usage", None))
        ch = r.choices[0]
        if getattr(ch, "finish_reason", "") == "length":
            raise RuntimeError("OpenAI yanıtı uzunluk sınırında kesildi")
        return extract_json(ch.message.content)
    if provider == "google":
        if _GOOGLE_CLIENT is None:
            from google import genai
            _GOOGLE_CLIENT = genai.Client(api_key=GOOGLE_API_KEY)
        cfg = {"system_instruction": system_text, "response_mime_type": "application/json", "response_schema": strip_additional(schema)}
        if temperature is not None:
            cfg["temperature"] = temperature
        if _PROVIDER_OPTS["google_thinking_budget"] is not None:
            cfg["thinking_config"] = {"thinking_budget": _PROVIDER_OPTS["google_thinking_budget"]}
        try:
            _t0 = time.perf_counter(); r = _GOOGLE_CLIENT.models.generate_content(model=model, contents=user_text, config=cfg); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
        except Exception as e:
            if "thinking_config" in cfg and "thinking" in str(e).lower() and "429" not in str(e):
                log(f"{model}: thinking_budget kabul edilmedi; parametresiz devam"); _PROVIDER_OPTS["google_thinking_budget"] = None
                cfg.pop("thinking_config")
                _t0 = time.perf_counter(); r = _GOOGLE_CLIENT.models.generate_content(model=model, contents=user_text, config=cfg); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
            else:
                raise
        _note_usage(model, getattr(r, "usage_metadata", None))
        return extract_json(r.text)
    raise ValueError(provider)

def paced_call(provider, fn, max_attempts=8):
    """fn()'i sağlayıcının hız sınırına uyarak çağırır; dakikalık 429'da bekler, kota/günlük sınırda LLMQuotaExhausted fırlatır.
    Dönen: (sonuç, yalnız API süresi ms)."""
    pacer = pacer_for(provider)
    for attempt in range(max_attempts):
        if not SMOKE_TEST:
            pacer.wait()
        try:
            _API_TL.last_ms = float("nan")
            out = fn()
            return out, float(getattr(_API_TL, "last_ms", float("nan")))
        except (LLMQuotaExhausted, BudgetExceeded):
            raise
        except Exception as e:
            kind = classify_api_error(e)
            if kind in ("quota", "daily"):
                raise LLMQuotaExhausted(f"{kind}: {str(e)[:300]}")
            wait_s = _retry_after_seconds(str(e))
            if kind == "rate":
                wait_s = wait_s if wait_s is not None else min(60.0, 3.0 * (2 ** attempt))
                pacer.hold(wait_s + 0.5)
                time.sleep(wait_s + 0.5 + random.random())
                continue
            if attempt >= 2:
                raise
            time.sleep(2.0 * (attempt + 1))
    raise RuntimeError(f"{provider}: {max_attempts} denemede hız sınırı aşılamadı")

def run_llm(provider, model, items, variant, set_name, system_text, schema, user_builder, parse_item,
            per_call=10, temperature=0.0, max_tokens_per_item=150):
    """items: [(id, metin)]. Dönen: ({id: çözümlenmiş değer}, bilgi sözlüğü).
    Ham yanıtlar REV_CACHE altında JSONL olarak tutulur; yeniden çalıştırmada yalnız eksikler istenir."""
    proto = llm_proto_tag(provider, model)
    sig = hashlib.md5(json.dumps([variant, set_name, system_text, schema, per_call, proto, temperature, SMOKE_TEST],
                                 ensure_ascii=False, sort_keys=True, default=str).encode()).hexdigest()[:8]
    path = os.path.join(REV_CACHE, re.sub(r"[^\w.-]+", "_", f"{set_name}__{variant}__{provider}_{model}") + f"_{sig}.jsonl")
    cache = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                try:
                    r = json.loads(line)
                except Exception:
                    continue
                if r.get("ok"):
                    cache[int(r["id"])] = r
    items = [(int(i), str(t)) for i, t in items]
    todo = [(i, t) for i, t in items if i not in cache]
    batches = [todo[k:k + per_call] for k in range(0, len(todo), per_call)]
    info = {"model": model, "saglayici": provider, "varyant": variant, "veri": set_name, "istenen": len(items),
            "onbellekten": len(cache), "yeni_istek": len(batches), "basarisiz_istek": 0, "durma": None}
    if batches and not has_key(provider):
        info["durma"] = "anahtar yok"; log(f"{model}: anahtar yok, {variant}/{set_name} atlandı"); batches = []
    if batches:
        log(f"BDM {variant} | {set_name} | {model}: {len(cache)} önbellekten, {len(todo)} paylaşım ({len(batches)} istek) API'den")
        workers = LLM_EVAL_WORKERS if provider == "anthropic" else LLM_EVAL_WORKERS_BY_PROVIDER.get(provider, 2)
        lock = threading.Lock()

        def work(batch):
            user = user_builder(batch)
            data, api_ms = paced_call(provider, lambda: api_json(provider, model, system_text, user, schema,
                                                                 max_tokens=max_tokens_per_item * len(batch) + 1500, temperature=temperature))
            preds = {}
            for p in (data or {}).get("predictions", []) if isinstance(data, dict) else []:
                try:
                    preds[int(p.get("id"))] = p
                except Exception:
                    pass
            rows = []
            for i, _ in batch:
                raw = preds.get(i)
                val = parse_item(raw) if raw is not None else None
                rows.append({"id": i, "ok": val is not None, "val": val, "raw": raw,
                             "ms": (api_ms / len(batch)) if api_ms == api_ms else None, "proto": proto})
            return rows

        ex = ThreadPoolExecutor(max_workers=workers)
        futs = [ex.submit(work, b) for b in batches]
        bar = tqdm(total=len(futs), desc=f"{variant} {model} ({set_name})")
        try:
            with open(path, "a", encoding="utf-8") as fout:
                for fut in as_completed(futs):
                    try:
                        rows = fut.result()
                    except CancelledError:
                        pass
                    except (LLMQuotaExhausted, BudgetExceeded) as e:
                        info["basarisiz_istek"] += 1
                        if info["durma"] is None:
                            info["durma"] = str(e)[:200]
                            log(f"DURDURULDU ({model}, {variant}/{set_name}): {info['durma']}")
                            for f_ in futs:
                                f_.cancel()
                    except Exception as e:
                        info["basarisiz_istek"] += 1
                        log(f"BDM isteği başarısız ({model}, {variant}/{set_name}): {str(e)[:200]}")
                    else:
                        with lock:
                            for r in rows:
                                if r["ok"]:
                                    cache[r["id"]] = r
                                fout.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
                    bar.update(1)
        finally:
            bar.close(); ex.shutdown(wait=False, cancel_futures=True)
    out = {i: cache[i]["val"] for i, _ in items if i in cache}
    ms = [cache[i].get("ms") for i in out if cache[i].get("ms") is not None]
    info.update({"yanitlanan": len(out), "kapsam": len(out) / max(1, len(items)),
                 "ms_paylasim_medyan": float(np.median(ms)) if ms else float("nan"), "dosya": os.path.basename(path)})
    if info["kapsam"] < LLM_MIN_COVERAGE:
        log(f"UYARI: {model} {variant}/{set_name} kapsamı {info['kapsam']:.3f} (< {LLM_MIN_COVERAGE}); hücreyi yeniden çalıştırınca eksikler istenir")
    return out, info

def run_jobs_by_provider(jobs):
    """jobs: [(provider, fonksiyon)]. Sağlayıcılar paralel, her sağlayıcının işleri sırayla çalışır (hız sınırları ayrıdır)."""
    by_p = defaultdict(list)
    for p, fn in jobs:
        by_p[p].append(fn)
    def run_seq(fns):
        for fn in fns:
            try:
                fn()
            except Exception as e:
                log(f"İş başarısız: {repr(e)[:300]}")
    with ThreadPoolExecutor(max_workers=max(1, len(by_p))) as ex:
        list(ex.map(run_seq, by_p.values()))

# --- Ölçütler ----------------------------------------------------------------------------------------------
def hit_scores(labels_en, top1, topk=None):
    hit1 = np.array([(p is not None) and (p in L) for p, L in zip(top1, labels_en)], dtype=float)
    lo, hi = bootstrap_ci(hit1)
    res = {"n_mesaj": int(len(hit1)), "hit1": float(hit1.mean()) if len(hit1) else float("nan"),
           "hit1_ga_alt": lo, "hit1_ga_ust": hi, "tahminsiz": int(sum(p is None for p in top1))}
    if topk is not None:
        hit3 = np.array([any((p in L) for p in (P or [])[:3] if p) for P, L in zip(topk, labels_en)], dtype=float)
        res["hit3"] = float(hit3.mean()) if len(hit3) else float("nan")
    return res, hit1

def multilabel_scores(gold_lists, pred_lists):
    """Mikro/makro F1, tüm altın ihtiyaçların bulunma oranı, tam küme eşleşmesi, ortalama tahmin sayısı."""
    tp, fp, fn = Counter(), Counter(), Counter()
    allf, exact = [], []
    for G, P in zip(gold_lists, pred_lists):
        G, P = set(G), set(P or [])
        for x in P & G: tp[x] += 1
        for x in P - G: fp[x] += 1
        for x in G - P: fn[x] += 1
        allf.append(G <= P); exact.append(G == P)
    TP, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    p = TP / (TP + FP) if TP + FP else 0.0; r = TP / (TP + FN) if TP + FN else 0.0
    f1s = []
    for l in set(tp) | set(fn):
        if tp[l] + fn[l] == 0:
            continue
        pl = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0; rl = tp[l] / (tp[l] + fn[l])
        f1s.append(2 * pl * rl / (pl + rl) if pl + rl else 0.0)
    return {"mikro_kesinlik": p, "mikro_duyarlilik": r, "mikro_f1": (2 * p * r / (p + r) if p + r else 0.0),
            "makro_f1": float(np.mean(f1s)) if f1s else float("nan"), "tum_ihtiyaclar_bulundu": float(np.mean(allf)) if allf else float("nan"),
            "tam_kume": float(np.mean(exact)) if exact else float("nan"), "ort_tahmin_sayisi": float(np.mean([len(P or []) for P in pred_lists])) if pred_lists else float("nan")}

def binary_scores(gold, pred):
    gold = np.asarray(gold, bool); pred = np.asarray(pred, bool)
    tp = int((gold & pred).sum()); fp = int((~gold & pred).sum()); fn = int((gold & ~pred).sum()); tn = int((~gold & ~pred).sum())
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    return {"ikili_dogruluk": (tp + tn) / max(1, len(gold)), "ikili_kesinlik": p, "ikili_duyarlilik": r,
            "ikili_f1": (2 * p * r / (p + r) if p + r else 0.0), "dp": tp, "yp": fp, "yn": fn, "dn": tn}

# --- Sonuç kaydı (oturumlar arası birikir) -------------------------------------------------------------------
REV_RESULTS_PATH = os.path.join(DIRS["rev"], "revizyon_tum_sonuclar.csv")
REV_RESULTS = {}
if os.path.exists(REV_RESULTS_PATH):
    for _r in pd.read_csv(REV_RESULTS_PATH, encoding="utf-8-sig").to_dict("records"):
        _k = _r.get("kosul"); _k = "" if not isinstance(_k, str) else _k
        REV_RESULTS[(_r["deney"], _r["veri"], _r["yontem"], _k)] = _r

def rev_record(deney, veri, yontem, kosul="", **m):
    row = {"deney": deney, "veri": veri, "yontem": yontem, "kosul": kosul}
    row.update({k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in m.items()})
    REV_RESULTS[(deney, veri, yontem, kosul)] = row
    pd.DataFrame(list(REV_RESULTS.values())).to_csv(REV_RESULTS_PATH, index=False, encoding="utf-8-sig")
    shown = ", ".join(f"{k}={v:.3f}" for k, v in m.items() if isinstance(v, (float, np.floating)) and k in
                      ("hit1", "hit3", "ilce_dogruluk_makale", "mikro_f1", "makro_f1", "ikili_f1", "dogruluk", "sektor_hit1", "kapsam"))
    log(f"[{deney}] {veri} | {yontem} {kosul}: {shown}")
    return row

def safe_tag(*parts):
    return re.sub(r"[^\w.+-]+", "_", "_".join(str(p) for p in parts if p != ""))

# --- Gömme yardımcıları ------------------------------------------------------------------------------------------
def cos_topk(M, Nv, labels, k=5):
    """Kosinüs benzerliğine göre ilk k benzersiz etiket ve puanları. Sıfır vektörlü paylaşım: boş liste (tahminsiz)."""
    M = np.asarray(M, dtype=np.float32); Nv = np.asarray(Nv, dtype=np.float32)
    mn = np.linalg.norm(M, axis=1, keepdims=True); nn = np.linalg.norm(Nv, axis=1, keepdims=True)
    S = (M / (mn + 1e-9)) @ (Nv / (nn + 1e-9)).T
    order = np.argsort(-S, axis=1)
    tops, scores = [], []
    for i in range(len(M)):
        if mn[i, 0] == 0:
            tops.append([]); scores.append([]); continue
        t, s = [], []
        for j in order[i]:
            if labels[j] in t:
                continue
            t.append(labels[j]); s.append(float(S[i, j]))
            if len(t) == k:
                break
        tops.append(t); scores.append(s)
    return tops, scores

def write_topk(tag, df, tops, scores=None):
    out = pd.DataFrame({"id": df["id"].values, "gercek": [";".join(L) for L in df["labels_en"]]})
    for k in range(5):
        out[f"top{k + 1}"] = [t[k] if len(t) > k else None for t in tops]
    if scores is not None:
        for k in range(5):
            out[f"s{k + 1}"] = [round(s[k], 5) if len(s) > k else None for s in scores]
    out.to_csv(os.path.join(DIRS["preds"], f"skor_{tag}.csv"), index=False, encoding="utf-8-sig")

def score_record(deney, set_name, df, yontem, kosul, tops, scores=None, extra=None):
    """Tahminleri kaydeder ve puanlar. Sentetik/alternatif kümeler: makaledeki ölçütler (hit@1, hit@3, ilçe doğruluğu).
    Gerçek kümeler: listedeki ihtiyacı olan paylaşımlarda hit@1/hit@3 ve somut ihtiyaçlı paylaşımlarda sektör düzeyi hit@1."""
    tag = safe_tag(set_name, yontem, kosul)
    write_topk(tag, df, tops, scores)
    top1 = [t[0] if t else None for t in tops]
    if set_name.startswith("gercek"):
        pos = [i for i, L in enumerate(df["labels_en"]) if L]
        res, _ = hit_scores([df["labels_en"].iat[i] for i in pos], [top1[i] for i in pos], [tops[i] for i in pos])
        hn = [i for i, h in enumerate(df["has_need"]) if h and df["sectors"].iat[i]]
        sec = np.array([(top1[i] is not None) and (SECTOR_OF_EN.get(top1[i]) in df["sectors"].iat[i]) for i in hn], float)
        res.update({"sektor_hit1": float(sec.mean()) if len(sec) else float("nan"), "n_sektor": len(hn)})
    else:
        res, _ = evaluate_predictions(df, top1, [t[:3] for t in tops], tag)
        sec = np.array([(p is not None) and (SECTOR_OF_EN.get(p) in {SECTOR_OF_EN[x] for x in L}) for p, L in zip(top1, df["labels_en"])], float)
        res["sektor_hit1"] = float(sec.mean())
    if extra:
        res.update(extra)
    return rev_record(deney, set_name, yontem, kosul, **res)

# --- Makaledeki tahmin dosyaları ------------------------------------------------------------------------------
def paper_pred(tag):
    """Makaledeki çalıştırmanın paylaşım düzeyi tahminleri (results/tahminler/mesaj_sentetik_<tag>.csv)."""
    p = paper_file(f"results/tahminler/mesaj_sentetik_{tag}.csv", required=False)
    if p is None:
        return None
    d = pd.read_csv(p, encoding="utf-8-sig")
    d["top1"] = d["top1"].where(d["top1"].notna(), None)
    return d.sort_values("id").reset_index(drop=True)

PAPER_LLM_TAGS = {m: f"BDM_{m}" for _, m in REV_LLMS}
log(f"Revizyon altyapısı hazır. Önbellek: {REV_CACHE} | Sonuçlar: {DIRS['rev']}")

In [ ]:
# =============================================================================
# R2. İHTİYAÇ TANIMLARI VE SEKTÖR ŞEMASI
# -----------------------------------------------------------------------------
# * Tanımlar: her maddeye kısa, genel bir tanım (Türkçe ve İngilizce). Gömme yöntemlerinde "ad + tanım"
#   koşulunda ihtiyaç metni olarak kullanılır. Varsayılan tanımlar Drive'a CSV olarak yazılır
#   (data/ihtiyac_tanimlari.csv); düzenlerseniz düzenlenmiş dosya kullanılır.
# * Sektörler: 66 madde sabittir; üstüne IASC küme yaklaşımına dayalı 13 sektörlük bir üst düzey eklenmiştir.
#   Hiyerarşik (kısmi) puanlama ve gerçek verideki listede olmayan ihtiyaçların değerlendirilmesi için kullanılır.
# =============================================================================
NEED_DESC_DEFAULT = [   # (madde, tanım TR, tanım EN): "ne olduğu; kimin için / hangi durumda"
    ("içme suyu", "içilebilir temiz su; şebeke suyu kesilen ya da kirlenen yerler için", "clean drinking water; for places where tap water is cut off or contaminated"),
    ("bebek maması", "bebek maması ya da formül süt; anne sütü alamayan bebekler için", "infant formula or baby food; for babies who cannot be breastfed"),
    ("biberon", "bebeklerin süt ya da mama içtiği şişe; bebeği olan aileler için", "feeding bottle; for families with babies"),
    ("ıslak mendil", "tek kullanımlık nemli mendil; su olmadan el ve bebek temizliği için", "disposable wet wipes; for cleaning hands and babies without water"),
    ("peçete", "kâğıt peçete ya da kâğıt havlu; temizlik ve yemek için", "paper napkins or paper towels; for cleaning and meals"),
    ("gıda", "yiyecek ve erzak; aç kalan ya da yiyeceği tükenen insanlar için", "food and provisions; for people who are hungry or have run out of food"),
    ("fener", "taşınabilir lamba; elektriği olmayan çadır ya da oda için", "portable lamp; for a tent or room without electricity"),
    ("elektrikli ısıtıcı", "elektrikle çalışan ısıtıcı; elektriği olan ama soğuk yerler için", "electric heater; for cold places that have power"),
    ("soba", "odun ya da kömürle yanan soba; soğukta çadırda ya da barakada kalanlar için", "wood or coal stove; for people staying in tents or huts in the cold"),
    ("odun", "yakacak odun; sobası ya da ateşi olup yakıtı olmayanlar için", "firewood; for people with a stove or fire but no fuel"),
    ("pil", "kuru pil; el feneri ve radyo gibi cihazları elektriksiz çalıştırmak için", "batteries; to run flashlights, radios and other devices without power"),
    ("battaniye", "battaniye ya da yorgan; soğukta üşüyen, açıkta ya da çadırda kalanlar için", "blankets or quilts; for people who are cold, outdoors or in tents"),
    ("pilli radyo", "pille çalışan radyo; elektrik ve internet yokken haber almak için", "battery-powered radio; to get news without power or internet"),
    ("diş fırçası", "diş fırçası; evinden ayrılmak zorunda kalanların kişisel temizliği için", "toothbrush; for the personal hygiene of displaced people"),
    ("diş macunu", "diş macunu; evinden ayrılmak zorunda kalanların kişisel temizliği için", "toothpaste; for the personal hygiene of displaced people"),
    ("hijyenik ped", "regl pedi; kadınların hijyen ihtiyacı için", "sanitary pads; for women's menstrual hygiene"),
    ("bebek bezi", "tek kullanımlık bez; bebekler ve bakıma muhtaç yetişkinler için", "disposable diapers; for babies and adults who need care"),
    ("sabun", "sabun; el ve vücut temizliği ve hastalıkları önlemek için", "soap; for washing hands and body and preventing disease"),
    ("yedek giysi", "temiz kıyafet, iç çamaşırı ve çorap; giysisi kirlenen, ıslanan ya da eşyasını alamayanlar için", "clean clothes, underwear and socks; for people whose clothes are dirty or wet or who lost their belongings"),
    ("mont", "kalın dış giysi; soğukta açıkta kalanlar için", "warm coat or jacket; for people staying outdoors in the cold"),
    ("maske", "yüz maskesi; enkaz tozuna ve enfeksiyona karşı", "face mask; against rubble dust and infection"),
    ("nakit para", "nakit para; acil alışveriş ve harcamalar için", "cash; for urgent purchases and expenses"),
    ("konserve", "teneke kutuda hazır yiyecek; yemek pişiremeyenler için", "canned food; for people who cannot cook"),
    ("sıcak yemek", "pişmiş sıcak yemek ya da çorba; soğukta ve mutfağı olmayanlar için", "hot cooked meal or soup; for people in the cold without a kitchen"),
    ("ilaç", "ilaç; hasta ya da yaralı olanlar ve düzenli ilaç kullananlar için", "medicine; for sick or injured people and those on regular medication"),
    ("ağrı kesici", "ağrı kesici ilaç; ağrısı olan yaralılar ve hastalar için", "painkiller; for injured or sick people in pain"),
    ("antibiyotik", "antibiyotik; mikrop kapmış yaralar ve enfeksiyonlar için", "antibiotic; for infected wounds and bacterial infections"),
    ("soğuk algınlığı ilacı", "nezle, grip ve öksürük ilacı; soğukta hastalananlar için", "cold, flu and cough medicine; for people who fell ill in the cold"),
    ("ateş düşürücü", "ateş düşürücü ilaç; ateşi çıkanlar, özellikle çocuklar için", "fever reducer; for people, especially children, with a high temperature"),
    ("yarabandı", "yapışkanlı küçük bant; küçük kesik ve sıyrıklar için", "adhesive bandage; for small cuts and scrapes"),
    ("bandaj", "sargı bezi ya da gazlı bez; yaraları sarmak için", "bandage or gauze; for dressing wounds"),
    ("alerji ilaçları", "alerji ilacı; alerjik tepki gösterenler için", "allergy medication; for people with allergic reactions"),
    ("astım ilacı", "astım ilacı ya da spreyi; nefes darlığı çeken astım hastaları için", "asthma medicine or inhaler; for asthma patients who are short of breath"),
    ("insülin", "iğneyle kullanılan şeker ilacı; ilacı olmayan şeker hastaları için", "injectable diabetes medicine; for people with diabetes who have none left"),
    ("seyyar tuvalet", "taşınabilir tuvalet; tuvaleti olmayan toplanma alanları ve çadırlar için", "portable toilet; for shelters and gathering areas without toilets"),
    ("çadır", "çadır; evi yıkılan ya da evine giremeyen, açıkta kalanlar için", "tent; for people whose homes collapsed or are unsafe and who are sleeping outdoors"),
    ("uyku tulumu", "uyku tulumu; soğukta dışarıda ya da çadırda kalan kişiler için", "sleeping bag; for people staying outside or in tents in the cold"),
    ("arama kurtarma ekibi", "arama kurtarma ekibi; enkaz altında kalan insanlar için", "search and rescue team; for people trapped under rubble"),
    ("sağlık ekibi", "doktor, hemşire ya da sağlık görevlisi; yaralılar ve tıbbi müdahale bekleyenler için", "doctors, nurses or paramedics; for injured people and those waiting for medical care"),
    ("psikolojik destek ekibi", "psikolog ya da destek uzmanı; travma, korku ve yas yaşayanlar için", "psychologists or support specialists; for people facing trauma, fear and grief"),
    ("güvenlik güçleri", "polis ya da jandarma; yağma, hırsızlık ve güvenlik sorunlarına karşı", "police or gendarmerie; against looting, theft and safety problems"),
    ("jenaratör", "jeneratör; elektriği kesilen yerlerde elektrik üretmek için", "generator; to produce electricity where the power is out"),
    ("powerbank", "taşınabilir şarj aleti; şarjı biten telefonlar için", "power bank; for phones whose batteries are dying"),
    ("glutesiz yiyecek", "glutensiz gıda; çölyak hastaları için", "gluten-free food; for people with coeliac disease"),
    ("tuzsuz-şekersiz yiyecek", "tuzsuz ve şekersiz gıda; şeker ve tansiyon hastaları için", "salt-free and sugar-free food; for people with diabetes or hypertension"),
    ("gönüllü", "gönüllü insan gücü; yardım dağıtımı, enkaz ve bakım işleri için", "volunteers; for aid distribution, rubble work and care"),
    ("kuru gıda", "pirinç, makarna, bakliyat gibi kuru erzak; yemek pişirebilenler için", "dry staples such as rice, pasta and pulses; for people who can cook"),
    ("ekmek", "ekmek; günlük temel yiyecek olarak", "bread; as a basic daily food"),
    ("ilk yardım kiti", "ilk yardım çantası; yaralılara ilk müdahale için", "first aid kit; for treating the injured on the spot"),
    ("kap kacak", "tencere ve tava; yemek pişirmek için", "pots and pans; for cooking"),
    ("engelli ekipmanı", "tekerlekli sandalye, koltuk değneği, işitme cihazı gibi araçlar; engelli bireyler için", "wheelchairs, crutches, hearing aids and similar aids; for people with disabilities"),
    ("kan", "kan ya da kan ürünü; kan kaybeden ağır yaralılar için", "blood or blood products; for seriously injured people who have lost blood"),
    ("aydınlatıcı", "projektör ve alan aydınlatması; gece çalışılan enkaz ve toplanma alanları için", "floodlights; for rubble sites and gathering areas at night"),
    ("hayvan maması", "kedi, köpek ve diğer hayvanlar için mama; evcil ve sokak hayvanları için", "food for cats, dogs and other animals; for pets and street animals"),
    ("kalp ilacı", "kalp ilacı; ilacı olmayan kalp hastaları için", "heart medicine; for heart patients who have none left"),
    ("konteynır", "konteyner ev; uzun süre barınma gereken aileler için", "container home; for families who need longer-term shelter"),
    ("temizlik ürünü", "deterjan, çamaşır suyu ve dezenfektan; barınma alanlarının temizliği için", "detergent, bleach and disinfectant; for cleaning shelters"),
    ("hırdavat", "kürek, kazma, çekiç gibi el aletleri; enkaz kaldırma ve onarım için", "hand tools such as shovels, pickaxes and hammers; for clearing rubble and repairs"),
    ("çocuk oyun alanı", "çocukların güvenle oynayabileceği alan ve oyuncak; afetten etkilenen çocuklar için", "safe play area and toys; for children affected by the disaster"),
    ("oksijen cihazı", "oksijen cihazı ya da tüpü; solunum güçlüğü çeken hastalar için", "oxygen device or cylinder; for patients with breathing problems"),
    ("elektrik saha ekibi", "elektrik teknisyenleri; hatları onarıp elektriği geri getirmek için", "electrical technicians; to repair power lines and restore electricity"),
    ("tercüman", "tercüman; Türkçe bilmeyen depremzedeler ve yabancı ekiplerle iletişim için", "interpreter; for communicating with survivors and foreign teams who do not speak Turkish"),
    ("kan toplama merkezi", "kan bağış noktası ya da aracı; kan bağışı yapmak isteyenler için", "blood donation centre or unit; for people who want to donate blood"),
    ("can yeleği", "can yeleği; su baskınında ya da denizden tahliyede batmamak için", "life jacket; to stay afloat during floods or evacuation by sea"),
    ("mutfak eşyası", "tabak, bardak, çatal ve kaşık; yemek yemek için", "plates, cups and cutlery; for eating"),
    ("el feneri", "elde taşınan pilli fener; karanlıkta yol bulmak ve enkazda aramak için", "handheld flashlight; for finding the way in the dark and searching rubble"),
]
assert [n for n, _, _ in NEED_DESC_DEFAULT] == NEEDS_TR, "Tanım listesi ihtiyaç listesiyle aynı sırada olmalı"
DESC_PATH = os.path.join(DATA_DIR, "ihtiyac_tanimlari.csv")
if not os.path.exists(DESC_PATH):
    pd.DataFrame([{"ihtiyac_tr": n, "ihtiyac_en": TR2EN[n], "tanim_tr": t, "tanim_en": e} for n, t, e in NEED_DESC_DEFAULT]
                 ).to_csv(DESC_PATH, index=False, encoding="utf-8-sig")
    log(f"Varsayılan ihtiyaç tanımları yazıldı: {DESC_PATH}")
_desc = pd.read_csv(DESC_PATH, encoding="utf-8-sig")
assert list(_desc["ihtiyac_tr"]) == NEEDS_TR and _desc[["tanim_tr", "tanim_en"]].notna().all().all(), "Tanım dosyası eksik ya da sırası bozuk"
NEED_DESC_TR = dict(zip(_desc["ihtiyac_tr"], _desc["tanim_tr"].astype(str)))
NEED_DESC_EN = {TR2EN[n]: str(e) for n, e in zip(_desc["ihtiyac_tr"], _desc["tanim_en"])}
_desc.to_csv(os.path.join(DIRS["rev"], "ihtiyac_tanimlari.csv"), index=False, encoding="utf-8-sig")
DESC_SIG = hashlib.md5(_desc.to_csv(index=False).encode()).hexdigest()[:6]
log(f"İhtiyaç tanımları: {len(_desc)} madde (imza {DESC_SIG})")

SECTORS = [
    ("S01", "Arama-kurtarma", "Search and rescue", "Enkaz altındakileri bulma, çıkarma ve tahliye; bunun için ekip ve ekipman"),
    ("S02", "Sağlık", "Health", "Tıbbi bakım, ilaç, tıbbi malzeme, kan, ruh sağlığı ve psikososyal destek"),
    ("S03", "Gıda ve beslenme", "Food and nutrition", "Gıda, hazır yemek ve bebek beslenmesi"),
    ("S04", "Su, sanitasyon ve hijyen", "Water, sanitation and hygiene", "İçme suyu, tuvalet, kişisel hijyen ve temizlik"),
    ("S05", "Barınma ve temel eşya", "Shelter and non-food items", "Barınak, ısınma, giyim, uyku, mutfak ve aydınlatma eşyası"),
    ("S06", "Enerji ve iletişim", "Energy and telecommunications", "Elektrik, şarj, haberleşme ve internet"),
    ("S07", "Koruma ve güvenlik", "Protection and security", "Güvenlik, çocukların korunması, kayıp yakınların aranması, dil desteği"),
    ("S08", "Lojistik ve ulaşım", "Logistics and transport", "Ulaşım, nakliye, yol erişimi, yardımın taşınması ve dağıtımı"),
    ("S09", "Eğitim", "Education", "Okul, geçici eğitim alanı ve eğitim malzemesi"),
    ("S10", "Kamp yönetimi", "Camp management", "Çadır kent ve toplu yaşam alanlarının yönetimi"),
    ("S11", "Erken iyileştirme ve altyapı", "Early recovery and infrastructure", "Enkaz kaldırma, hasar tespiti, onarım ve yeniden yapım"),
    ("S12", "Nakit ve gönüllü destek", "Cash and volunteers", "Para yardımı ve gönüllü insan gücü"),
    ("S13", "Diğer", "Other", "Yukarıdakilere girmeyen ihtiyaçlar"),
]
SECTOR_NAMES = [s[1] for s in SECTORS]
_S = {
    "Arama-kurtarma": ["arama kurtarma ekibi", "can yeleği"],
    "Sağlık": ["ilaç", "ağrı kesici", "antibiyotik", "soğuk algınlığı ilacı", "ateş düşürücü", "alerji ilaçları", "astım ilacı", "insülin",
               "kalp ilacı", "yarabandı", "bandaj", "ilk yardım kiti", "kan", "kan toplama merkezi", "oksijen cihazı", "sağlık ekibi",
               "psikolojik destek ekibi", "engelli ekipmanı", "maske"],
    "Gıda ve beslenme": ["gıda", "kuru gıda", "konserve", "sıcak yemek", "ekmek", "glutesiz yiyecek", "tuzsuz-şekersiz yiyecek", "bebek maması", "biberon"],
    "Su, sanitasyon ve hijyen": ["içme suyu", "sabun", "ıslak mendil", "peçete", "diş fırçası", "diş macunu", "hijyenik ped", "bebek bezi",
                                 "seyyar tuvalet", "temizlik ürünü"],
    "Barınma ve temel eşya": ["çadır", "konteynır", "battaniye", "uyku tulumu", "yedek giysi", "mont", "soba", "odun", "elektrikli ısıtıcı",
                              "kap kacak", "mutfak eşyası", "fener", "el feneri", "aydınlatıcı", "pil", "hırdavat"],
    "Enerji ve iletişim": ["jenaratör", "powerbank", "pilli radyo", "elektrik saha ekibi"],
    "Koruma ve güvenlik": ["güvenlik güçleri", "çocuk oyun alanı", "tercüman"],
    "Nakit ve gönüllü destek": ["nakit para", "gönüllü"],
    "Diğer": ["hayvan maması"],
}
ITEM_SECTOR = {n: s for s, L in _S.items() for n in L}
assert sorted(ITEM_SECTOR) == sorted(NEEDS_TR) and set(ITEM_SECTOR.values()) <= set(SECTOR_NAMES), "Sektör şeması 66 maddeyi tam kapsamalı"
SECTOR_OF_EN = {TR2EN[n]: s for n, s in ITEM_SECTOR.items()}
pd.DataFrame([{"ihtiyac_tr": n, "ihtiyac_en": TR2EN[n], "sektor": ITEM_SECTOR[n],
               "sector": dict((s[1], s[2]) for s in SECTORS)[ITEM_SECTOR[n]]} for n in NEEDS_TR]
             ).to_csv(os.path.join(DIRS["rev"], "sektor_semasi.csv"), index=False, encoding="utf-8-sig")
log("Sektör şeması: " + ", ".join(f"{s} ({sum(1 for v in ITEM_SECTOR.values() if v == s)})" for s in SECTOR_NAMES))

In [ ]:
# =============================================================================
# R3. ALTERNATİF ÜRETİCİLER: aynı planlar, farklı model aileleri (öz-tercih yanlılığı denetimi)
# -----------------------------------------------------------------------------
# Makaledeki veriden ilçe başına ALT_PER_DISTRICT kayıt sabit tohumla seçilir. Bu kayıtların planları
# (ilçe, ihtiyaçlar, anlatım biçimi, persona, kayıt dili, uzunluk, biçim özellikleri) AYNEN korunur ve
# makaledeki üretim istemiyle Gemini 3.8 Flash ve GPT-5 mini'ye yeniden yazdırılır. Çıktılar makaledeki
# doğrulama ve tekrar eleme adımlarından geçer, üçüncü bir aileden modelle denetlenir. Böylece aynı plan
# için üç üreticinin metni (Claude, Gemini, GPT) eşleştirilmiş olarak karşılaştırılabilir.
# =============================================================================
ALT_SETS = {}
ALT_QC_ROWS = []
if REV["alternatif_uretici"]:
    _rng = random.Random(SEED + 300)
    ALT_IDS = []
    for _d in DISTRICTS:
        _ids = df_syn.index[df_syn["district"] == _d].tolist()
        ALT_IDS += sorted(_rng.sample(_ids, min(ALT_PER_DISTRICT, len(_ids))))
    ALT_IDS = sorted(ALT_IDS)

    def spec_of(i):
        r = df_syn.loc[i]
        needs = list(r["labels_tr"])
        return {"id": f"a{int(r['id']):04d}", "district": r["district"], "needs_tr": needs, "needs_en": [TR2EN[n] for n in needs],
                "explicitness": r["explicitness"], "persona": r["persona"], "register": r["register"], "length": r["length"],
                "typos": bool(r["typos"]), "no_diacritics": bool(r["no_diacritics"]), "hashtag": bool(r["hashtag"]),
                "emoji": bool(r["emoji"]), "neighbourhood": bool(r["neighbourhood"]), "numbers": bool(r["numbers"])}

    ALT_SPECS = [spec_of(i) for i in ALT_IDS]
    ALT_SIG = hashlib.md5(json.dumps([PAPER_GEN_SIGNATURE, ALT_PER_DISTRICT, SEED, PROMPT_VERSION, SMOKE_TEST]).encode()).hexdigest()[:8]
    ALT_DIR = os.path.join(DATA_DIR, "gen_parts_alt", ALT_SIG); os.makedirs(ALT_DIR, exist_ok=True)
    log(f"Alternatif üretici alt kümesi: {len(ALT_SPECS)} plan ({ALT_PER_DISTRICT}/ilçe); parçalar: {ALT_DIR}")

    def _chunk_name(prefix, chunk):
        return f"{prefix}_{hashlib.md5('|'.join(s['id'] for s in chunk).encode()).hexdigest()[:10]}.json"

    def alt_generate(gname, provider, model, specs):
        chunks = [specs[i:i + GEN_RECORDS_PER_CALL] for i in range(0, len(specs), GEN_RECORDS_PER_CALL)]
        def one(ch):
            pp = os.path.join(ALT_DIR, _chunk_name(f"gen_{gname}", ch))
            if os.path.exists(pp):
                with open(pp, encoding="utf-8") as f:
                    return json.load(f)
            t0 = time.perf_counter()
            data, _ = paced_call(provider, lambda: api_json(provider, model, SYSTEM_PROMPT_GEN, build_user_prompt(ch), GEN_SCHEMA,
                                                            max_tokens=max(3000, GEN_MAX_TOKENS_PER_RECORD * len(ch)), temperature=None))
            part = {"specs": ch, "records": (data or {}).get("records", []), "model": model,
                    "seconds": round(time.perf_counter() - t0, 1), "ts": datetime.datetime.now().isoformat()}
            with open(pp, "w", encoding="utf-8") as f:
                json.dump(part, f, ensure_ascii=False)
            return part
        parts = []
        with ThreadPoolExecutor(max_workers=LLM_EVAL_WORKERS_BY_PROVIDER.get(provider, 2)) as ex:
            futs = [ex.submit(one, ch) for ch in chunks]
            for fut in tqdm(as_completed(futs), total=len(futs), desc=f"Üretim ({gname})"):
                try:
                    parts.append(fut.result())
                except (LLMQuotaExhausted, BudgetExceeded) as e:
                    log(f"Üretim durdu ({gname}): {str(e)[:200]}")
                except Exception as e:
                    log(f"Üretim parçası başarısız ({gname}): {str(e)[:200]}")
        return parts

    def alt_qc(tag, judge, records):
        jp, jm = judge
        chunks = [records[i:i + GEN_RECORDS_PER_CALL] for i in range(0, len(records), GEN_RECORDS_PER_CALL)]
        def one(ch):
            pp = os.path.join(ALT_DIR, _chunk_name(f"qc_{tag}_{jm}", ch))
            if os.path.exists(pp):
                with open(pp, encoding="utf-8") as f:
                    return json.load(f)
            user = "Items (one JSON object per line):\n" + "\n".join(
                json.dumps({k: r[k] for k in ("id", "district", "needs_tr", "needs_en", "explicitness", "message_tr", "message_en")}, ensure_ascii=False)
                for r in ch)
            data, _ = paced_call(jp, lambda: api_json(jp, jm, SYSTEM_PROMPT_QC, user, QC_SCHEMA, max_tokens=200 * len(ch) + 1500,
                                                      temperature=0.0, effort=QC_EFFORT))
            with open(pp, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False)
            return data
        res = {}
        with ThreadPoolExecutor(max_workers=LLM_EVAL_WORKERS_BY_PROVIDER.get(jp, LLM_EVAL_WORKERS)) as ex:
            futs = [ex.submit(one, ch) for ch in chunks]
            for fut in tqdm(as_completed(futs), total=len(futs), desc=f"Denetim ({tag}, {jm})"):
                try:
                    for r in (fut.result() or {}).get("results", []):
                        res[str(r.get("id"))] = r
                except (LLMQuotaExhausted, BudgetExceeded) as e:
                    log(f"Denetim durdu ({tag}): {str(e)[:200]}")
                except Exception as e:
                    log(f"Denetim parçası başarısız ({tag}): {str(e)[:200]}")
        return res

    def qc_fields(q, judge_model):
        return {"qc_model": judge_model, "qc_checked": q is not None,
                "qc_needs_ok": bool(q["all_needs_expressed"]) if q else None, "qc_district_ok": bool(q["district_ok"]) if q else None,
                "qc_ok": bool(q and q["all_needs_expressed"] and q["district_ok"] and not q.get("extra")),
                "qc_missing": "; ".join(q.get("missing", [])) if q else "", "qc_extra": "; ".join(q.get("extra", [])) if q else ""}

    def qc_summary_row(name, gen_model, judge_model, recs):
        chk = [r for r in recs if r.get("qc_checked")]
        return {"metin_ureticisi": name, "uretici_model": gen_model, "denetci_model": judge_model, "plan": len(ALT_SPECS),
                "kabul": len(recs), "denetlenen": len(chk),
                "ihtiyaclar_tam": float(np.mean([r["qc_needs_ok"] for r in chk])) if chk else float("nan"),
                "fazladan_ihtiyac_yok": float(np.mean([not r["qc_extra"] for r in chk])) if chk else float("nan"),
                "ilce_dogru": float(np.mean([r["qc_district_ok"] for r in chk])) if chk else float("nan"),
                "tumu_gecti": float(np.mean([r["qc_ok"] for r in chk])) if chk else float("nan")}

    ALT_TEXTS = {}
    for g in ALT_GENERATORS:
        gname, (gp, gm), (jp, jm) = g["ad"], g["uretici"], g["denetci"]
        if not has_key(gp):
            log(f"Alternatif üretici {gname}: {gp} anahtarı yok, atlandı"); continue
        accepted, reasons_all = {}, Counter()
        for rnd in range(2):                                     # 2. tur: reddedilen ya da eksik planlar yeniden istenir
            todo = [s for s in ALT_SPECS if int(s["id"][1:]) not in accepted]
            if not todo:
                break
            acc, reasons = validate_records(alt_generate(gname if rnd == 0 else f"{gname}_r{rnd}", gp, gm, todo))
            reasons_all.update(reasons)
            for r in acc:
                accepted.setdefault(int(r["id"][1:]), r)
            log(f"{gname} tur {rnd}: {len(acc)} kabul, retler {dict(reasons)}; toplam {len(accepted)}/{len(ALT_SPECS)}")
        recs, n_dup = deduplicate([accepted[i] for i in sorted(accepted)], DEDUP_COSINE_THRESHOLD)
        log(f"{gname}: tekrar elenen {n_dup}, kalan {len(recs)}")
        qc = alt_qc(gname, (jp, jm), recs) if has_key(jp) else {}
        for r in recs:
            r.update(qc_fields(qc.get(r["id"]), jm))
        ALT_TEXTS[gname] = {int(r["id"][1:]): r for r in recs}
        ALT_QC_ROWS.append({**qc_summary_row(gname, gm, jm, recs), "ret_nedenleri": json.dumps(dict(reasons_all), ensure_ascii=False), "tekrar_elenen": n_dup})

    # Aynı alt kümedeki Claude metinlerinin çapraz denetimi (makaledeki denetçi Claude Sonnet 5.5 idi)
    _claude_recs = [{**spec_of(i), "message_tr": df_syn.at[i, "Message"], "message_en": df_syn.at[i, "English"]} for i in ALT_IDS]
    _qc_c = alt_qc("claude", ALT_CLAUDE_QC, _claude_recs) if has_key(ALT_CLAUDE_QC[0]) else {}
    for r in _claude_recs:
        r.update(qc_fields(_qc_c.get(r["id"]), ALT_CLAUDE_QC[1]))
    ALT_QC_ROWS.append({**qc_summary_row("claude", GEN_MODEL, ALT_CLAUDE_QC[1], _claude_recs), "ret_nedenleri": "", "tekrar_elenen": 0})
    _paper_qc = df_syn.loc[ALT_IDS, "qc_ok"].astype(str).str.lower().eq("true")
    ALT_QC_ROWS.append({"metin_ureticisi": "claude (makaledeki denetim)", "uretici_model": GEN_MODEL, "denetci_model": QC_MODEL,
                        "plan": len(ALT_IDS), "kabul": len(ALT_IDS), "denetlenen": len(ALT_IDS), "tumu_gecti": float(_paper_qc.mean())})

    # Eşleştirilmiş küme: tüm üreticilerde metni olan planlar
    _common = set(ALT_IDS)
    for _m in ALT_TEXTS.values():
        _common &= set(_m)
    ALT_COMMON_IDS = sorted(_common)
    _cqc = {int(r["id"][1:]): r["qc_ok"] for r in _claude_recs}

    def alt_frame(gname, recmap):
        rows = []
        for i in ALT_COMMON_IDS:
            b = df_syn.loc[i]; r = recmap[i] if recmap is not None else None
            rows.append({"id": int(i), "Message": r["message_tr"] if r else b["Message"], "English": r["message_en"] if r else b["English"],
                         "district": b["district"], "label1": b["label1"], "label2": b["label2"], "label3": b["label3"],
                         "labels_tr": list(b["labels_tr"]), "labels_en": list(b["labels_en"]), "n_needs": int(b["n_needs"]),
                         "explicitness": b["explicitness"], "persona": b["persona"], "register": b["register"], "length": b["length"],
                         "uretici": gname, "qc_ok": r["qc_ok"] if r else _cqc.get(i),
                         "qc_ok_makale": (str(b["qc_ok"]).lower() == "true") if r is None else None})
        d = pd.DataFrame(rows)
        if recmap is None:                                       # Claude metinleri: makaledeki ön işlenmiş sütunlar
            for c in ("text_plain", "text_lemma", "text_stem"):
                d[c] = df_syn.loc[ALT_COMMON_IDS, c].values
            return d
        return add_text_columns(d)

    if ALT_TEXTS:
        ALT_SETS["alt_claude"] = alt_frame("claude", None)
        for gname, recmap in ALT_TEXTS.items():
            ALT_SETS[f"alt_{gname}"] = alt_frame(gname, recmap)
        for name, d in ALT_SETS.items():
            d.drop(columns=["labels_tr", "labels_en"]).to_csv(os.path.join(DIRS["rev"], f"{name}_veri.csv"), index=False, encoding="utf-8-sig")
        log(f"Eşleştirilmiş alternatif kümeler: {list(ALT_SETS)}; ortak plan sayısı {len(ALT_COMMON_IDS)}")
    if ALT_QC_ROWS:
        save_table(pd.DataFrame(ALT_QC_ROWS), "revizyon/alternatif_uretici_denetim")
        print(pd.DataFrame(ALT_QC_ROWS).round(3).to_string(index=False))
else:
    log("Alternatif üretici deneyi kapalı (REV['alternatif_uretici']=False)")

In [ ]:
# =============================================================================
# R4. GERÇEK VERİ: iki bağımsız etiketleyicinin uzlaştırılmış altın etiketleri
# -----------------------------------------------------------------------------
# Kümeler: (i) HumAID deprem bölümünün "requests_or_urgent_needs" sınıfındaki İngilizce tweetler,
# (ii) Kaggle "Turkey and Syria Earthquake Tweets" veri setindeki Türkçe tweetler.
# Altın dosya (gercek_veri_altin.csv) uzlaştırmadan sonra hazırlanır ve YALNIZ sizin Drive'ınızda durur
# (tweet metinleri yayımlanmaz; depoya yalnız kimlikler ve etiketler konur). Sütunlar:
#   kume (EN/TR), sira, tweet_id, metin, somut_ihtiyac (Evet/Hayır), ihtiyaclar_tr (';' ile, listedeki adlar),
#   listede_olmayan (';' ile), listede_olmayan_sektor (';' ile, 13 sektörden), anlatim, il, ilce
# Türkçe tweetler yalnız İngilizce çalışan modeller için makine çevirisiyle (MT_MODEL) İngilizceye çevrilir;
# çok dilli modeller ve BDM'ler özgün Türkçe metni de kullanır.
# =============================================================================
REAL_SETS = {}
REAL_GOLD = None
REAL_DIR = os.path.join(BASE_DIR, REAL_DIR_NAME); os.makedirs(REAL_DIR, exist_ok=True)
REAL_PATH = os.path.join(REAL_DIR, REAL_GOLD_FILE)

def _split_list(x):
    if not isinstance(x, str):
        return []
    return [t.strip() for t in x.split(";") if t.strip()]

def translate_tr_en(d):
    """Türkçe -> İngilizce makine çevirisi; tweet kimliğine göre Drive'da önbelleğe alınır."""
    cache_p = os.path.join(REAL_DIR, "makine_cevirisi_tr_en.csv")
    cache = {}
    if os.path.exists(cache_p):
        _c = pd.read_csv(cache_p, encoding="utf-8-sig", dtype={"tweet_id": str})
        cache = dict(zip(_c["tweet_id"], _c["ceviri"].astype(str)))
    keys = d["tweet_id"].astype(str).tolist()
    todo = [(k, t) for k, t in zip(keys, d["Message"]) if k not in cache]
    if todo:
        if SMOKE_TEST:
            for k, t in todo:
                cache[k] = t
        else:
            texts = [t for _, t in todo]
            try:
                from transformers import pipeline
                mt = pipeline("translation", model=MT_MODEL, device=0 if DEVICE == "cuda" else -1)
                outs = [o["translation_text"] for o in mt(texts, batch_size=16, max_length=512, truncation=True)]
            except Exception as e:                           # yedek: MarianMT ile doğrudan üretim
                log(f"Çeviri hattı kurulamadı ({repr(e)[:150]}); MarianMT ile doğrudan çevriliyor")
                from transformers import MarianMTModel, MarianTokenizer
                tok = MarianTokenizer.from_pretrained(MT_MODEL); mt = MarianMTModel.from_pretrained(MT_MODEL).to(DEVICE)
                outs = []
                for k0 in range(0, len(texts), 16):
                    enc = tok(texts[k0:k0 + 16], return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
                    outs += tok.batch_decode(mt.generate(**enc, max_new_tokens=512), skip_special_tokens=True)
            for (k, _), o in zip(todo, outs):
                cache[k] = o
            del mt; gc.collect()
        pd.DataFrame({"tweet_id": list(cache), "ceviri": list(cache.values())}).to_csv(cache_p, index=False, encoding="utf-8-sig")
        log(f"Makine çevirisi: {len(todo)} yeni tweet ({MT_MODEL})")
    return [cache[k] for k in keys]

if REV["gercek_veri"]:
    if SMOKE_TEST:                                               # akış denemesi için sentetik paylaşımlardan sahte "gerçek" küme
        _rng = random.Random(7); _rows = []
        for kume, col in (("EN", "English"), ("TR", "Message")):
            for k, (_, r) in enumerate(df_syn.sample(40, random_state=11 if kume == "EN" else 12).iterrows()):
                yes = _rng.random() < 0.6
                _rows.append({"kume": kume, "sira": f"{kume}-{k + 1:03d}", "tweet_id": str(10 ** 17 + k), "metin": r[col],
                              "somut_ihtiyac": "Evet" if yes else "Hayır", "ihtiyaclar_tr": ";".join(r["labels_tr"]) if yes else "",
                              "listede_olmayan": "barınma" if yes and _rng.random() < 0.2 else "",
                              "listede_olmayan_sektor": "", "anlatim": "Açık" if yes else "",
                              "il": "İstanbul" if yes else "", "ilce": r["district"] if yes else ""})
                if _rows[-1]["listede_olmayan"]:
                    _rows[-1]["listede_olmayan_sektor"] = "Barınma ve temel eşya"
        REAL_GOLD = pd.DataFrame(_rows)
    elif os.path.exists(REAL_PATH):
        REAL_GOLD = pd.read_csv(REAL_PATH, encoding="utf-8-sig", dtype={"tweet_id": str, "il": str, "ilce": str})
    else:
        log(f"Gerçek veri altın dosyası yok ({REAL_PATH}); gerçek veri adımları atlanıyor. "
            f"Dosyayı bu klasöre koyup bu hücreden itibaren yeniden çalıştırabilirsiniz.")

if REAL_GOLD is not None:
    _need = {"kume", "sira", "tweet_id", "metin", "somut_ihtiyac", "ihtiyaclar_tr", "listede_olmayan", "listede_olmayan_sektor", "anlatim", "il", "ilce"}
    assert not (_need - set(REAL_GOLD.columns)), f"Altın dosyada eksik sütunlar: {_need - set(REAL_GOLD.columns)}"
    for kume in ("EN", "TR"):
        d = REAL_GOLD[REAL_GOLD["kume"].astype(str).str.upper() == kume].copy().reset_index(drop=True)
        if d.empty:
            continue
        d["id"] = np.arange(len(d))
        d["labels_tr"] = d["ihtiyaclar_tr"].apply(lambda x: list(dict.fromkeys(c for c in (canon_need_tr(t) for t in _split_list(x)) if c)))
        _bad = [t for x in d["ihtiyaclar_tr"] for t in _split_list(x) if canon_need_tr(t) is None]
        assert not _bad, f"Listede olmayan madde adı: {_bad[:5]}"
        d["labels_en"] = d["labels_tr"].apply(lambda L: list(dict.fromkeys(TR2EN[t] for t in L)))
        d["has_need"] = d["somut_ihtiyac"].astype(str).str.strip().eq("Evet")
        d["ly_list"] = d["listede_olmayan"].apply(_split_list)
        d["ly_sektor"] = d["listede_olmayan_sektor"].apply(_split_list)
        _badsec = [s for L in d["ly_sektor"] for s in L if s not in SECTOR_NAMES]
        assert not _badsec, f"Geçersiz sektör adı: {_badsec[:5]}"
        d["sectors"] = [sorted(set(ITEM_SECTOR[t] for t in L) | set(S)) for L, S in zip(d["labels_tr"], d["ly_sektor"])]
        d["district"] = "-"
        for c in ("il", "ilce"):
            d[c] = d[c].fillna("").astype(str).replace("nan", "")
        if kume == "TR":
            d["Message"] = d["metin"].astype(str)
            d["English"] = translate_tr_en(d)
        else:
            d["English"] = d["metin"].astype(str); d["Message"] = d["English"]
        d = add_text_columns(d)
        REAL_SETS["gercek_" + kume.lower()] = d
        log(f"Gerçek küme {kume}: {len(d)} paylaşım, somut ihtiyaç {int(d['has_need'].sum())}, listedeki ihtiyacı olan "
            f"{int((d['labels_en'].str.len() > 0).sum())}, yalnız listede olmayan ihtiyaçlı "
            f"{int((d['has_need'] & (d['labels_en'].str.len() == 0)).sum())}")
        _cov = Counter(s for S in d.loc[d["has_need"], "sectors"] for s in S)
        log(f"  sektör dağılımı: {dict(_cov)}")


# --- HumAID: insan (kitle) etiketleriyle dış geçerlilik --------------------------------------------------------
# HumAID deprem bölümündeki tweetler üç kitle etiketleyicinin çoğunluk oyuyla sınıflandırılmıştır (Alam vd., 2021).
# "requests_or_urgent_needs" sınıfının tamamı ve diğer sınıflardan HUMAID_NEG_RATIO katı rastgele tweet alınır;
# BDM'lerin açık küme yanıtındaki "somut ihtiyaç var" kararı bu insan etiketiyle karşılaştırılır (R8).
HUMAID_SET = None
HUMAID_DIR = os.path.join(BASE_DIR, HUMAID_DIR_NAME)
HUMAID_ZIP_URL = "https://crisisnlp.qcri.org/data/humaid/humaid_data_all.zip"

def clean_tweet(t):
    t = re.sub(r"^RT @\w+:\s*", "", str(t)); t = re.sub(r"https?://\S+", "", t)
    return re.sub(r"\s+", " ", t).strip().strip('"').strip()

def _read_records(p):
    if p.endswith(".tsv"):
        return pd.read_csv(p, sep="\t", dtype=str, quoting=3, on_bad_lines="skip").to_dict("records")
    txt = open(p, encoding="utf-8", errors="ignore").read()
    try:
        d = json.loads(txt)
    except json.JSONDecodeError:
        d = [json.loads(l) for l in txt.splitlines() if l.strip()]
    if isinstance(d, dict):
        d = d.get("data", list(d.values()))
    return d

def humaid_files():
    out = []
    for sp in ("train", "dev", "test"):
        for ext in (".json", ".jsonl", ".tsv"):
            p = os.path.join(HUMAID_DIR, sp + ext)
            if os.path.exists(p):
                out.append((sp, p)); break
    if out:
        return out
    try:                                                     # Drive'da yoksa resmî zip; deprem dosyaları seçilir
        os.makedirs(HUMAID_DIR, exist_ok=True)
        zp = os.path.join(HUMAID_DIR, "humaid_data_all.zip")
        if not os.path.exists(zp):
            urllib.request.urlretrieve(HUMAID_ZIP_URL, zp + ".part"); os.replace(zp + ".part", zp)
        with zipfile.ZipFile(zp) as z:
            names = [n for n in z.namelist() if re.search(r"(train|dev|test)\.(tsv|jsonl?|json)$", n, re.I)]
            et = [n for n in names if re.search(r"(^|/)earthquake[^/]*_(train|dev|test)\.", n, re.I)]
            ev = [n for n in names if re.search(r"earthquake[^/]*_(train|dev|test)\.", n, re.I)]
            pick = et or ev
            for n in pick:
                sp = re.search(r"_(train|dev|test)\.", n, re.I).group(1).lower()
                dst = os.path.join(HUMAID_DIR, "zip_" + re.sub(r"[^\w.]+", "_", n))
                with open(dst, "wb") as f:
                    f.write(z.read(n))
                out.append((sp, dst))
        log(f"HumAID zip'inden {len(out)} deprem dosyası alındı")
    except Exception as e:
        log(f"HumAID verisi bulunamadı ({repr(e)[:150]}); '{HUMAID_DIR}' klasörüne train/dev/test dosyalarını koyabilirsiniz")
    return out

if REV.get("humaid_insan"):
    if SMOKE_TEST:
        _h = df_syn.sample(150, random_state=5)[["English"]].rename(columns={"English": "metin"})
        _h["sinif"] = ["requests_or_urgent_needs" if k % 3 == 0 else "sympathy_and_support" for k in range(len(_h))]
        _h["tweet_id"] = [str(10 ** 15 + k) for k in range(len(_h))]; _h["bolum"] = "test"
        _hum = _h
    else:
        _rows = []
        for sp, p in humaid_files():
            for x in _read_records(p):
                t = clean_tweet(x.get("tweet_text", "")); lab = str(x.get("class_label", "")).strip()
                if t and lab and lab.lower() != "nan":
                    _rows.append({"tweet_id": str(x.get("tweet_id", "")), "metin": t, "sinif": lab, "bolum": sp})
        _hum = pd.DataFrame(_rows)
    if len(_hum):
        _hum = _hum.drop_duplicates("metin")
        _req = _hum[_hum["sinif"] == "requests_or_urgent_needs"]; _oth = _hum[_hum["sinif"] != "requests_or_urgent_needs"]
        _oth = _oth.sample(min(len(_oth), HUMAID_NEG_RATIO * len(_req)), random_state=SEED)
        HUMAID_SET = pd.concat([_req, _oth]).sample(frac=1, random_state=SEED).reset_index(drop=True)
        HUMAID_SET["id"] = np.arange(len(HUMAID_SET)); HUMAID_SET["English"] = HUMAID_SET["metin"]
        HUMAID_SET["is_request"] = HUMAID_SET["sinif"].eq("requests_or_urgent_needs")
        HUMAID_SET[["id", "tweet_id", "sinif", "bolum"]].to_csv(os.path.join(DIRS["rev"], "humaid_ornek_kimlikler.csv"), index=False, encoding="utf-8-sig")
        log(f"HumAID örneği: {len(HUMAID_SET)} tweet ({int(HUMAID_SET['is_request'].sum())} istek/acil ihtiyaç); sınıflar: "
            f"{HUMAID_SET['sinif'].value_counts().to_dict()}")

In [ ]:
# =============================================================================
# R5. GÖMME YÖNTEMLERİ
# -----------------------------------------------------------------------------
# (a) Yeni kodlayıcılar (mE5-large-instruct, BGE-M3, Qwen3-Embedding 0.6B/4B): İngilizce ve Türkçe metin,
#     makaledeki düz metin ve ön işlemesiz (ham) metin, ihtiyaç adı ve ad + tanım koşulları. Karar kuralı makaledeki
#     gibidir (en yüksek kosinüs benzerliği); talimatlı modellerde paylaşımlara görev talimatı eklenir.
# (b) İnce ayar ablasyonu: RoBERTa ham (roberta-base) ve güncel karşıtsal eşi; makaledeki BERT-raw / BERT (NLI) /
#     ALBERT / ALBERT-paraphrase çiftleriyle birlikte ham -> NLI -> karşıtsal ince ayar karşılaştırması.
# (c) Makaledeki 12 model ve temel çizgiler: ad + tanım koşulu (sentetik), alternatif üretici kümeleri ve gerçek veri.
# Her yapılandırmanın ilk 5 tahmini ve kosinüs puanları tahminler/skor_*.csv dosyalarına yazılır (kademeli analiz için).
# =============================================================================
class PromptSTEmbedder:
    """Talimatlı/güncel sentence-transformers kodlayıcısı: sorgulara (paylaşımlara) görev talimatı eklenir, belgeler (ihtiyaçlar) yalın."""
    def __init__(self, name, query_prompt="", fp16=False):
        import torch as _t
        from sentence_transformers import SentenceTransformer
        kw = {}
        if fp16 and DEVICE == "cuda":
            kw["model_kwargs"] = {"torch_dtype": _t.float16}
        if "qwen" in name.lower():
            kw["tokenizer_kwargs"] = {"padding_side": "left"}
        try:
            self.model = SentenceTransformer(name, device=DEVICE, **kw)
        except TypeError:
            self.model = SentenceTransformer(name, device=DEVICE)
        try:
            self.model.max_seq_length = min(int(self.model.max_seq_length or 512), 512)
        except Exception:
            pass
        self.qp = query_prompt.replace("{task}", EMB_TASK) if query_prompt else ""
        self.info = f"sentence-transformers:{name}"
        self.n_params = int(sum(p.numel() for p in self.model.parameters()))
        self.dim = int(self.model.get_sentence_embedding_dimension() or 0)
    def _enc(self, texts, bs=32):
        return np.asarray(self.model.encode(list(texts), batch_size=bs, convert_to_numpy=True, show_progress_bar=False,
                                            normalize_embeddings=True), dtype=np.float32)
    def encode(self, texts):
        return self._enc(texts)
    def encode_query(self, texts):
        return self._enc([self.qp + t for t in texts]) if self.qp else self._enc(texts)
    def set_device(self, dev):
        self.model.to(dev)
        if dev == "cpu":
            self.model.float()
    def close(self):
        del self.model

def message_texts(df, lang, mode):
    if lang == "en":
        return df["English"].astype(str).tolist() if mode == "raw" else df["text_" + mode].astype(str).tolist()
    return df["Message"].astype(str).tolist() if mode == "raw" else df["Message"].apply(plain_text_tr).tolist()

def need_inputs(lang, mode, need_mode):
    """İhtiyaç metinleri ve karşılık gelen İngilizce etiketler. 'ad': makaledeki protokol; 'ad+tanim': ad + kısa tanım."""
    if lang == "en":
        if need_mode == "ad":
            texts = list(EN_LABELS) if mode == "raw" else need_texts(mode)
        else:
            base = [f"{n}: {NEED_DESC_EN[n]}" for n in EN_LABELS]
            texts = base if mode == "raw" else [TEXT_FUNCS[mode](t) or t.lower() for t in base]
        return texts, list(EN_LABELS)
    base = [n if need_mode == "ad" else f"{n}: {NEED_DESC_TR[n]}" for n in NEEDS_TR]
    texts = base if mode == "raw" else [plain_text_tr(t) for t in base]
    return texts, [TR2EN[n] for n in NEEDS_TR]

def emb_eval(emb, model_name, deney, set_name, df, lang, mode, need_mode, mcache):
    msgs = message_texts(df, lang, mode)
    needs, labels = need_inputs(lang, mode, need_mode)
    if hasattr(emb, "fit"):                                    # TF-IDF: paylaşımlar + ihtiyaç metinleri üzerinde
        emb.fit(msgs + needs)
        M = emb.encode(msgs)
    else:
        key = (set_name, lang, mode)
        if key not in mcache:
            t0 = time.perf_counter()
            mcache[key] = emb.encode_query(msgs) if hasattr(emb, "encode_query") else emb.encode(msgs)
            mcache[("_ms", ) + key] = (time.perf_counter() - t0) / max(1, len(msgs)) * 1000
        M = mcache[key]
    tops, scores = cos_topk(M, emb.encode(needs), labels)
    return score_record(deney, set_name, df, model_name, f"{lang}/{mode}/{need_mode}", tops, scores)

def jobs_for(kind, purpose):
    """Bir modelin çalışacağı (deney, küme, dil, metin, ihtiyaç) birleşimleri."""
    J = []
    tt3 = list(TEXT_TYPES)
    if purpose == "yeni":
        for lang in ("en", "tr"):
            for mode in ("plain", "raw"):
                for nm in ("ad", "ad+tanim"):
                    J.append(("gomme_sentetik", "sentetik", lang, mode, nm))
                for s in ALT_SETS:
                    J.append(("gomme_alternatif", s, lang, mode, "ad"))
        for s in REAL_SETS:
            langs = ("en",) if s == "gercek_en" else ("tr", "en")
            for lang in langs:
                for mode in ("plain", "raw"):
                    for nm in (("ad", "ad+tanim") if not (s == "gercek_tr" and lang == "en") else ("ad",)):
                        J.append(("gomme_gercek", s, lang, mode, nm))
        return J
    if purpose == "ablasyon":
        return [("gomme_sentetik", "sentetik", "en", m, "ad") for m in tt3]
    # makaledeki modeller
    if kind == "st_tr":
        if REV["tanimli_ihtiyaclar"]:
            J.append(("gomme_sentetik", "sentetik", "tr", "plain", "ad+tanim"))
        J += [("gomme_alternatif", s, "tr", "plain", "ad") for s in ALT_SETS]
        if "gercek_tr" in REAL_SETS:
            J += [("gomme_gercek", "gercek_tr", "tr", "plain", nm) for nm in ("ad", "ad+tanim")]
        return J
    if REV["tanimli_ihtiyaclar"]:
        J += [("gomme_sentetik", "sentetik", "en", m, "ad+tanim") for m in tt3]
    J += [("gomme_alternatif", s, "en", m, "ad") for s in ALT_SETS for m in tt3]
    J += [("gomme_gercek", s, "en", m, nm) for s in REAL_SETS for m in tt3 for nm in ("ad", "ad+tanim")]
    return J

def run_model_jobs(model_name, loader, jobs, latency=False):
    if not jobs:
        return None
    t0 = time.perf_counter()
    try:
        emb = loader()
    except Exception as e:
        log(f"Model yüklenemedi ({model_name}): {repr(e)[:300]}"); return None
    log(f"Gömme: {model_name} ({getattr(emb, 'info', '')}) yüklendi ({time.perf_counter() - t0:.0f} sn); {len(jobs)} yapılandırma")
    mcache, lat = {}, None
    sets = {"sentetik": df_syn, **ALT_SETS, **REAL_SETS}
    for deney, s, lang, mode, nm in jobs:
        try:
            emb_eval(emb, model_name, deney, s, sets[s], lang, mode, nm, mcache)
        except Exception as e:
            log(f"Yapılandırma başarısız ({model_name}, {s}, {lang}/{mode}/{nm}): {repr(e)[:300]}")
    if latency:
        sample = random.Random(SEED).sample(df_syn["English"].astype(str).tolist(), 30)
        _q = getattr(emb, "encode_query", emb.encode)
        def _lat():
            v = []
            for s in sample:
                t1 = time.perf_counter(); _q([s]); v.append((time.perf_counter() - t1) * 1000)
            return float(np.median(v))
        lat = {"Model": model_name, "kaynak": getattr(emb, "info", ""), "parametre": getattr(emb, "n_params", None), "boyut": getattr(emb, "dim", None),
               "toplu_ms_paylasim_" + DEVICE: mcache.get(("_ms", "sentetik", "en", "plain")), "tek_paylasim_ms_" + DEVICE: _lat()}
        if DEVICE == "cuda" and (getattr(emb, "n_params", 0) or 0) < CPU_LATENCY_MAX_PARAMS and not SMOKE_TEST:
            try:
                emb.set_device("cpu"); lat["tek_paylasim_ms_cpu"] = _lat()
            except Exception as e:
                log(f"İşlemci gecikmesi ölçülemedi ({model_name}): {e}")
    try:
        emb.close()
    except Exception:
        pass
    del emb; gc.collect()
    if torch is not None and DEVICE == "cuda":
        torch.cuda.empty_cache()
    return lat

LATENCY_ROWS = []
# (a) yeni kodlayıcılar
if REV["yeni_kodlayicilar"]:
    for name, spec in NEW_ENCODERS.items():
        loader = (lambda n=name: MockEmbedder(n)) if SMOKE_TEST else \
                 (lambda s=spec: PromptSTEmbedder(s["name"], s.get("query_prompt", ""), s.get("fp16", False)))
        r = run_model_jobs(name, loader, jobs_for("st", "yeni"), latency=True)
        if r:
            LATENCY_ROWS.append(r)
    if LATENCY_ROWS:
        save_table(pd.DataFrame(LATENCY_ROWS), "revizyon/yeni_kodlayici_gecikme")
# (b) ince ayar ablasyonu
if REV["ince_ayar_ablasyonu"]:
    for name, hf in ABLATION_MODELS.items():
        loader = (lambda n=name: MockEmbedder(n)) if SMOKE_TEST else (lambda h=hf: STEmbedder(h))
        run_model_jobs(name, loader, jobs_for("st", "ablasyon"))
# (c) makaledeki modeller: tanımlı ihtiyaçlar, alternatif kümeler, gerçek veri
for name in ALL_MODELS:
    kind = MODEL_SPECS[name]["kind"]
    if kind in ("random", "majority"):
        for s, d in {**ALT_SETS, **REAL_SETS}.items():
            top1, top3 = baseline_predictions(kind, d)
            score_record("gomme_alternatif" if s.startswith("alt") else "gomme_gercek", s, d, name, "en/plain/ad", top3)
        continue
    jobs = jobs_for(kind, "makale")
    if jobs:
        run_model_jobs(name, (lambda n=name: load_model(n)), jobs)
log("Gömme deneyleri tamam.")

In [ ]:
# =============================================================================
# R6. AZ ÖRNEKLİ ÖĞRENME: SetFit ve dondurulmuş gömme + lojistik regresyon
# -----------------------------------------------------------------------------
# Bölme (sabit tohum): her ihtiyaç için FEW_K paylaşım eğitime ayrılır (tek ihtiyaçlı paylaşımlar öncelikli,
# 66 x 8 = 528); geri kalan paylaşımlar test kümesidir. Sıfır atış yöntemler (makaledeki tahminler) ve
# 66 örnekli BDM istemi (R8) AYNI test kümesinde değerlendirilir.
# SetFit yordamı (Tunstall vd., 2022): aynı/farklı ihtiyaçlı paylaşım çiftleriyle kosinüs benzerliği üzerinden
# karşıtsal ince ayar (her yinelemede her örnek için bir pozitif ve bir negatif çift), ardından lojistik regresyon
# başlığı. sentence-transformers ile doğrudan uygulanmıştır (setfit paketinin sürüm uyumsuzluklarından bağımsız).
# =============================================================================
FEW_TRAIN, FEW_TEST, FEW_TARGET, FEW_EXAMPLES = [], [], {}, {}
if REV["setfit"] or REV["bdm_az_ornekli"]:
    _rng = random.Random(SEED + 500)
    for n in NEEDS_TR:
        en = TR2EN[n]
        c1 = [i for i in df_syn.index if int(df_syn.at[i, "n_needs"]) == 1 and df_syn.at[i, "label1"] == n and i not in FEW_TARGET]
        c2 = [i for i in df_syn.index if n in df_syn.at[i, "labels_tr"] and i not in FEW_TARGET and i not in c1]
        _rng.shuffle(c1); _rng.shuffle(c2)
        pick = (c1 + c2)[:FEW_K]
        for i in pick:
            FEW_TARGET[i] = en
        FEW_EXAMPLES[en] = pick[0]
    FEW_TRAIN = sorted(FEW_TARGET); FEW_TEST = sorted(set(df_syn.index) - set(FEW_TRAIN))
    DF_FEW_TEST = df_syn.loc[FEW_TEST].reset_index(drop=True)
    pd.DataFrame({"id": df_syn["id"], "bolum": ["egitim" if i in FEW_TARGET else "test" for i in df_syn.index],
                  "egitim_ihtiyaci": [FEW_TARGET.get(i, "") for i in df_syn.index],
                  "bdm_ornegi": [i in set(FEW_EXAMPLES.values()) for i in df_syn.index]}
                 ).to_csv(os.path.join(DIRS["rev"], "az_ornekli_bolum.csv"), index=False, encoding="utf-8-sig")
    log(f"Az örnekli bölme: eğitim {len(FEW_TRAIN)}, test {len(FEW_TEST)} paylaşım; BDM istemi için {len(FEW_EXAMPLES)} örnek")

    # Sıfır atış referansı: makaledeki gömme ve temel çizgi tahminleri aynı test kümesinde
    _paper_emb = [m for m in MAIN_MODELS + EXTRA_MODELS + ["Temel: TF-IDF"] if m in MODEL_SPECS]
    _tags = [(m, tt, f"{m.replace(':', '_')}_{tt}") for m in _paper_emb for tt in TEXT_TYPES] + \
            [(m, "plain", f"{m}_plain") for m in TURKISH_MODELS]
    _idx = {i: k for k, i in enumerate(df_syn["id"])}
    for m, tt, tag in _tags:
        d = paper_pred(tag)
        if d is None:
            continue
        d = d.set_index("id").reindex(DF_FEW_TEST["id"])
        tops = [[x for x in (r.top1, r.top2, r.top3) if isinstance(x, str)] for r in d.itertuples()]
        score_record("az_ornekli", "sentetik_test", DF_FEW_TEST, m, f"sifir_atis/{tt}", tops)

def setfit_train(base, texts, labels, seed, iterations):
    """SetFit karşıtsal ince ayarı (CosineSimilarityLoss eşdeğeri: kosinüs benzerliği ile 1/0 hedefi arasında MSE)."""
    import torch as _t
    from sentence_transformers import SentenceTransformer
    _t.manual_seed(seed); np.random.seed(seed); rng = random.Random(seed)
    model = SentenceTransformer(base, device=DEVICE)
    model.max_seq_length = min(int(model.max_seq_length or 256), 256)
    by = defaultdict(list)
    for i, y in enumerate(labels):
        by[y].append(i)
    pairs = []
    for _ in range(iterations):
        for i, y in enumerate(labels):
            pos = [j for j in by[y] if j != i]
            if pos:
                pairs.append((i, rng.choice(pos), 1.0))
            j = rng.randrange(len(labels))
            while labels[j] == y:
                j = rng.randrange(len(labels))
            pairs.append((i, j, 0.0))
    rng.shuffle(pairs)
    steps = math.ceil(len(pairs) / SETFIT_BATCH) * SETFIT_EPOCHS; warm = max(1, int(0.1 * steps))
    opt = _t.optim.AdamW(model.parameters(), lr=SETFIT_LR)
    sched = _t.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (steps - s) / max(1, steps - warm)))
    use_bf16 = DEVICE == "cuda" and _t.cuda.is_bf16_supported()
    _prep = getattr(model, "preprocess", None) or model.tokenize
    dev = model.device
    model.train()
    for _ in range(SETFIT_EPOCHS):
        for b in tqdm(range(0, len(pairs), SETFIT_BATCH), desc=f"SetFit {base.split('/')[-1]} tohum {seed}", leave=False):
            bp = pairs[b:b + SETFIT_BATCH]
            # sentence-transformers >= 6: preprocess() (tokenize() eskidi); çıktıda "modality" gibi metin alanları da vardır,
            # yalnız tensörler cihaza taşınır, diğer alanlar olduğu gibi forward'a geçer
            fa = {k: (v.to(dev) if hasattr(v, "to") else v) for k, v in _prep([texts[i] for i, _, _ in bp]).items()}
            fb = {k: (v.to(dev) if hasattr(v, "to") else v) for k, v in _prep([texts[j] for _, j, _ in bp]).items()}
            y = _t.tensor([l for _, _, l in bp], dtype=_t.float32, device=dev)
            with _t.autocast(device_type="cuda", dtype=_t.bfloat16, enabled=use_bf16):
                ea = model(fa)["sentence_embedding"]; eb = model(fb)["sentence_embedding"]
            loss = _t.nn.functional.mse_loss(_t.nn.functional.cosine_similarity(ea.float(), eb.float()), y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    model.eval()
    return model

def few_model(lang, texts, labels, seed, iterations, train=True):
    if SMOKE_TEST:
        return MockEmbedder(f"setfit-{lang}-{seed}-{train}")
    if not train:
        from sentence_transformers import SentenceTransformer
        return SentenceTransformer(SETFIT_BASE[lang], device=DEVICE)
    return setfit_train(SETFIT_BASE[lang], texts, labels, seed, iterations)

def few_embed(model, texts):
    if isinstance(model, MockEmbedder):
        return model.encode(list(texts))
    return np.asarray(model.encode(list(texts), batch_size=64, convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)

def lr_topk(Xtr, ytr, X, seed, k=5):
    from sklearn.linear_model import LogisticRegression
    clf = LogisticRegression(max_iter=5000, random_state=seed).fit(Xtr, ytr)
    P = clf.predict_proba(X); order = np.argsort(-P, axis=1)[:, :k]
    return [[clf.classes_[j] for j in row] for row in order], [[float(P[i, j]) for j in row] for i, row in enumerate(order)]

def few_real_sets(lang):
    """Eğitilen modelin uygulanacağı gerçek kümeler: İngilizce model -> İngilizce metin (Türkçe kümede makine çevirisi)."""
    out = []
    for s, d in REAL_SETS.items():
        if lang == "en":
            out.append((s, d, d["English"].astype(str).tolist(), "en"))
        elif s == "gercek_tr":
            out.append((s, d, d["Message"].astype(str).tolist(), "tr"))
    return out

def free_model(m):
    del m; gc.collect()
    if torch is not None and DEVICE == "cuda":
        torch.cuda.empty_cache()

if REV["setfit"] and FEW_TRAIN:
    _ytr = [FEW_TARGET[i] for i in FEW_TRAIN]
    for lang in ("en", "tr"):
        col = "English" if lang == "en" else "Message"
        Xtr_txt = df_syn.loc[FEW_TRAIN, col].astype(str).tolist(); Xte_txt = DF_FEW_TEST[col].astype(str).tolist()
        try:                                                     # dondurulmuş gömme + lojistik regresyon
            base = few_model(lang, None, None, SEED, 0, train=False)
            Etr, Ete = few_embed(base, Xtr_txt), few_embed(base, Xte_txt)
            tops, sc = lr_topk(Etr, _ytr, Ete, SEED)
            score_record("az_ornekli", "sentetik_test", DF_FEW_TEST, f"Dondurulmuş+LR-{lang.upper()}", f"K{FEW_K}", tops, sc)
            for s, d, txt, _ in few_real_sets(lang):
                tops, sc = lr_topk(Etr, _ytr, few_embed(base, txt), SEED)
                score_record("az_ornekli_gercek", s, d, f"Dondurulmuş+LR-{lang.upper()}", f"K{FEW_K}", tops, sc)
            free_model(base)
        except Exception as e:
            log(f"Dondurulmuş gömme + LR başarısız ({lang}): {repr(e)[:300]}")
        for seed in SETFIT_SEEDS:
            try:
                t0 = time.perf_counter()
                m = few_model(lang, Xtr_txt, _ytr, seed, SETFIT_ITERATIONS)
                Etr = few_embed(m, Xtr_txt)
                tops, sc = lr_topk(Etr, _ytr, few_embed(m, Xte_txt), seed)
                score_record("az_ornekli", "sentetik_test", DF_FEW_TEST, f"SetFit-{lang.upper()}", f"K{FEW_K}/tohum{seed}", tops, sc)
                for s, d, txt, _ in few_real_sets(lang):
                    tops, sc = lr_topk(Etr, _ytr, few_embed(m, txt), seed)
                    score_record("az_ornekli_gercek", s, d, f"SetFit-{lang.upper()}", f"K{FEW_K}/tohum{seed}", tops, sc)
                log(f"SetFit {lang} tohum {seed}: {time.perf_counter() - t0:.0f} sn")
                free_model(m)
            except Exception as e:
                log(f"SetFit başarısız ({lang}, tohum {seed}): {repr(e)[:300]}")
        if SETFIT_FULL_FOR_REAL and REAL_SETS:                   # 2.000 sentetik paylaşımla eğitilen model -> gerçek veri
            try:
                txt_all = df_syn[col].astype(str).tolist(); y_all = [TR2EN[x] for x in df_syn["label1"]]
                m = few_model(lang, txt_all, y_all, SEED, SETFIT_FULL_ITERATIONS)
                E_all = few_embed(m, txt_all)
                for s, d, txt, _ in few_real_sets(lang):
                    tops, sc = lr_topk(E_all, y_all, few_embed(m, txt), SEED)
                    score_record("az_ornekli_gercek", s, d, f"SetFit-{lang.upper()}", "tam_sentetik", tops, sc)
                free_model(m)
            except Exception as e:
                log(f"SetFit (tam sentetik) başarısız ({lang}): {repr(e)[:300]}")
log("Az örnekli öğrenme tamam.")

In [ ]:
# =============================================================================
# R7. İLÇE VE KONUM ÇIKARIMI: alt dize (makale), bulanık sözlük eşlemesi, Türkçe NER + sözlük
# -----------------------------------------------------------------------------
# (a) Sentetik paylaşımlarda ilçe adı bilinçli olarak bozulur: küçük harf, Türkçe karakter yok, ekin kesme işaretsiz
#     bitişmesi, bir harflik yazım hatası. Her yöntemin doğru ilçeyi tek başına bulma oranı ölçülür.
# (b) Gerçek Türkçe tweetlerde (altın dosyada il/ilçe varsa) Türkiye geneli il/ilçe sözlüğüyle konum çıkarımı.
# =============================================================================
from rapidfuzz import fuzz, process as rf_process

_TR_ASCII = str.maketrans({"ı": "i", "ş": "s", "ğ": "g", "ç": "c", "ö": "o", "ü": "u", "â": "a", "î": "i", "û": "u",
                           "İ": "I", "Ş": "S", "Ğ": "G", "Ç": "C", "Ö": "O", "Ü": "U", "Â": "A"})
def ascii_tr(s):
    return str(s).translate(_TR_ASCII)

def norm_tok(s):
    return ascii_tr(tr_lower(s)).replace("'", "").replace("’", "").replace("`", "")

# Yer adlarından sonra gelebilen yaygın ekler (Türkçe karakterleri ASCII'ye indirgenmiş)
SUFFIX_SET = {"", "de", "da", "te", "ta", "den", "dan", "ten", "tan", "e", "a", "ye", "ya", "i", "u", "yi", "yu", "in", "un", "nin",
              "nun", "deki", "daki", "teki", "taki", "li", "lu", "liler", "lular", "lilar", "luler", "dir", "dur", "tir", "tur",
              "dekiler", "dakiler", "ne", "na", "nde", "nda", "nden", "ndan", "ndeki", "ndaki", "nin", "dakilere", "dekilere"}

def make_gazetteer(names):
    return {norm_tok(n): n for n in names if str(n).strip()}

def match_token(tok, gaz, fuzzy=True):
    t = norm_tok(tok)
    if len(t) < 3:
        return None
    if t in gaz:
        return gaz[t]
    for g, name in gaz.items():                                   # ekli biçim: Kadıköy'de -> kadikoyde
        if len(g) >= 4 and t.startswith(g) and t[len(g):] in SUFFIX_SET:
            return name
    if fuzzy and len(t) >= 5:                                     # yazım hatası: ekini atıp benzerlik
        cands = [t] + [t[:-len(s)] for s in SUFFIX_SET if s and t.endswith(s) and len(t) - len(s) >= 5]
        best = None
        for c in cands:
            m = rf_process.extractOne(c, list(gaz), scorer=fuzz.ratio)
            if m and m[1] >= FUZZY_MIN and m[0][0] == c[0] and abs(len(m[0]) - len(c)) <= 2 and (best is None or m[1] > best[1]):
                best = m
        if best:
            return gaz[best[0]]
    return None

_TOKEN_RE = re.compile(r"[A-Za-zÇĞİÖŞÜçğıöşüÂâÎîÛû'’]+")
def extract_fuzzy(text, gaz):
    out = []
    for tok in _TOKEN_RE.findall(str(text)):
        m = match_token(tok, gaz)
        if m and m not in out:
            out.append(m)
    return out

def extract_substring(text, names):
    t = tr_lower(text)
    return [n for n in names if tr_lower(n) in t]

NER_PIPE, NER_NAME = None, None
def load_ner():
    global NER_PIPE, NER_NAME
    if NER_PIPE is not None or SMOKE_TEST:
        return
    from transformers import pipeline
    for nm in NER_MODELS:
        try:
            NER_PIPE = pipeline("ner", model=nm, aggregation_strategy="simple", device=0 if DEVICE == "cuda" else -1)
            NER_NAME = nm; log(f"NER modeli: {nm}"); return
        except Exception as e:
            log(f"NER modeli yüklenemedi ({nm}): {repr(e)[:200]}")

def ner_locations(texts):
    if SMOKE_TEST:
        return [[w for w in _TOKEN_RE.findall(t) if w[:1].isupper()] for t in texts]
    load_ner()
    if NER_PIPE is None:
        return [[] for _ in texts]
    res = []
    for o in NER_PIPE(list(texts), batch_size=32):
        res.append([e["word"] for e in o if str(e.get("entity_group", "")).upper().replace("B-", "").replace("I-", "") in ("LOC", "LOCATION", "GPE")])
    return res

def extract_ner(locs, gaz):
    out = []
    for ent in locs:
        for tok in _TOKEN_RE.findall(ent.replace("##", "")):
            m = match_token(tok, gaz)
            if m and m not in out:
                out.append(m)
    return out

def perturb(text, d, kind, rng):
    """Paylaşımdaki ilçe adının ilk geçişini bozar."""
    k = tr_lower(text).find(tr_lower(d))
    if k < 0 or kind == "orijinal":
        return text
    occ, after = text[k:k + len(d)], text[k + len(d):]
    if kind == "kucuk_harf":
        occ = tr_lower(occ)
    elif kind == "aksansiz":
        occ = ascii_tr(occ)
    elif kind == "ek_bitisik":
        after = after[1:] if after[:1] in ("'", "’") else after
    elif kind == "yazim_hatasi":
        if len(occ) >= 5:
            p = rng.randrange(1, len(occ) - 1)
            occ = occ[:p] + occ[p + 1:] if rng.random() < 0.5 else occ[:p - 1] + occ[p] + occ[p - 1] + occ[p + 1:]
    return text[:k] + occ + after

if REV["ner_ilce"]:
    IST_GAZ = make_gazetteer(DISTRICTS)
    VARIANTS = ["orijinal", "kucuk_harf", "aksansiz", "ek_bitisik", "yazim_hatasi"]
    _gold = df_syn["district"].tolist()
    _rows = []
    for kind in VARIANTS:
        rng = random.Random(stable_seed("bozma", kind))
        texts = [perturb(t, d, kind, rng) for t, d in zip(df_syn["Message"], _gold)]
        changed = float(np.mean([a != b for a, b in zip(texts, df_syn["Message"])]))
        preds = {"alt_dize (makale)": [detect_districts(t) for t in texts],
                 "bulanik_sozluk": [extract_fuzzy(t, IST_GAZ) for t in texts]}
        try:
            preds["ner+sozluk"] = [extract_ner(L, IST_GAZ) for L in ner_locations(texts)]
        except Exception as e:
            log(f"NER başarısız ({kind}): {repr(e)[:200]}")
        for meth, P in preds.items():
            rev_record("ilce_cikarimi", "sentetik", meth, kind, n=len(texts), degisen_oran=changed,
                       dogruluk=float(np.mean([p == [g] for p, g in zip(P, _gold)])),
                       ilk_tahmin_dogru=float(np.mean([bool(p) and p[0] == g for p, g in zip(P, _gold)])),
                       dogru_iceren=float(np.mean([g in p for p, g in zip(P, _gold)])),
                       tahminsiz=float(np.mean([len(p) == 0 for p in P])))
            _rows.append(pd.DataFrame({"id": df_syn["id"], "bozma": kind, "yontem": meth, "gercek": _gold, "tahmin": [";".join(p) for p in P]}))
    pd.concat(_rows).to_csv(os.path.join(DIRS["preds"], "ilce_cikarimi_sentetik.csv"), index=False, encoding="utf-8-sig")

    # (b) gerçek Türkçe tweetler: Türkiye geneli il ve ilçe sözlüğü (geoBoundaries gbOpen TUR ADM1/ADM2 adları)
    def tr_admin_names(level):
        p = os.path.join(GEO_DIR, f"tur_{level.lower()}_adlari.json")
        if os.path.exists(p):
            with open(p, encoding="utf-8") as f:
                return json.load(f)
        if SMOKE_TEST:
            return ["İstanbul"] if level == "ADM1" else list(DISTRICTS)
        import requests
        meta = requests.get(f"https://www.geoboundaries.org/api/current/gbOpen/TUR/{level}/", timeout=60).json()
        gj = requests.get(meta.get("simplifiedGeometryGeoJSON") or meta.get("gjDownloadURL"), timeout=300).json()
        names = sorted({f["properties"].get("shapeName", "").strip() for f in gj["features"] if f["properties"].get("shapeName")})
        with open(p, "w", encoding="utf-8") as f:
            json.dump(names, f, ensure_ascii=False)
        return names

    if "gercek_tr" in REAL_SETS:
        try:
            IL_NAMES, ILCE_NAMES = tr_admin_names("ADM1"), tr_admin_names("ADM2")
            IL_GAZ, ILCE_GAZ = make_gazetteer(IL_NAMES), make_gazetteer(ILCE_NAMES)
            d = REAL_SETS["gercek_tr"]
            texts = d["Message"].astype(str).tolist()
            locs = ner_locations(texts)
            _out = {"id": d["id"], "il_gercek": d["il"], "ilce_gercek": d["ilce"]}
            for level, names, gaz, gold_col in (("il", IL_NAMES, IL_GAZ, "il"), ("ilce", ILCE_NAMES, ILCE_GAZ, "ilce")):
                P = {"alt_dize": [extract_substring(t, names) for t in texts],
                     "bulanik_sozluk": [extract_fuzzy(t, gaz) for t in texts],
                     "ner+sozluk": [extract_ner(L, gaz) for L in locs]}
                idx = [i for i, g in enumerate(d[gold_col]) if g]
                for meth, pr in P.items():
                    _out[f"{level}_{meth}"] = [";".join(p) for p in pr]
                    if idx:
                        g = [norm_tok(d[gold_col].iat[i]) for i in idx]
                        rev_record("konum_cikarimi", "gercek_tr", meth, level, n=len(idx),
                                   dogruluk=float(np.mean([[norm_tok(x) for x in pr[i]] == [gg] for i, gg in zip(idx, g)])),
                                   ilk_tahmin_dogru=float(np.mean([bool(pr[i]) and norm_tok(pr[i][0]) == gg for i, gg in zip(idx, g)])),
                                   dogru_iceren=float(np.mean([gg in [norm_tok(x) for x in pr[i]] for i, gg in zip(idx, g)])),
                                   tahminsiz=float(np.mean([len(pr[i]) == 0 for i in idx])))
            pd.DataFrame(_out).to_csv(os.path.join(DIRS["preds"], "konum_cikarimi_gercek_tr.csv"), index=False, encoding="utf-8-sig")
            log(f"Konum sözlüğü: {len(IL_NAMES)} il, {len(ILCE_NAMES)} ilçe adı (geoBoundaries)")
        except Exception as e:
            log(f"Gerçek veride konum çıkarımı başarısız: {repr(e)[:300]}")
log("İlçe/konum çıkarımı tamam.")

In [ ]:
# =============================================================================
# R8. BDM DENEYLERİ (makaledeki protokol: 10 paylaşım/istek, JSON şeması, en düşük düşünme)
# -----------------------------------------------------------------------------
# * Tekrarlar: makaledeki istemle BDM_TEKRAR ek çalıştırma; çalıştırmalar arası değişkenlik (hit@1 ortalama/SS,
#   tüm tekrarlarda aynı tahmin oranı, Fleiss kappa).
# * GPT-5 mini: ilk denemede varsayılan düşünmeyle yanıtlanan 210 paylaşım minimal düşünmeyle yeniden sorulur.
# * Türkçe: özgün Türkçe paylaşım + makaledeki istemin Türkçe çevirisi + Türkçe liste.
# * Çok etiketli: en fazla 3 sıralı ihtiyaç (İngilizce ve Türkçe); mikro/makro F1, tüm ihtiyaçları bulma oranı.
# * Az örnekli: her ihtiyaçtan bir örnek (66 örnek) içeren istem; R6'daki test kümesinde.
# * Alternatif üreticiler: aynı planlarla Gemini/GPT'nin yazdığı metinler (Claude metinleri için makaledeki tahminler).
# * Gerçek veri: kapalı küme (makaledeki istem) ve açık küme (somut ihtiyaç var mı + listeden ihtiyaçlar + "not in list").
# İşler sağlayıcılara göre paralel yürür; yanıtlar Drive'da önbelleğe alınır, yeniden çalıştırmada yalnız eksikler istenir.
# =============================================================================
_TR_NEED_LOWER = {" ".join(tr_lower(n).split()): n for n in NEEDS_TR}

def normalize_need_tr(x):
    """Türkçe ihtiyaç adı -> İngilizce etiket (makaledeki çıktı uzayı)."""
    if not isinstance(x, str):
        return None
    k = " ".join(tr_lower(x).split()); k = NEED_ALIASES_TR.get(k, k)
    if k in _TR_NEED_LOWER:
        return TR2EN[_TR_NEED_LOWER[k]]
    m = difflib.get_close_matches(k, list(_TR_NEED_LOWER), n=1, cutoff=0.6)
    return TR2EN[_TR_NEED_LOWER[m[0]]] if m else None

def _schema(item_props):
    return {"type": "object", "properties": {"predictions": {"type": "array", "items": {
        "type": "object", "properties": {"id": {"type": "integer"}, **item_props},
        "required": ["id"] + list(item_props), "additionalProperties": False}}}, "required": ["predictions"], "additionalProperties": False}

_LIST_TR = "\n".join(f"- {n}" for n in NEEDS_TR)
_LIST_EN = "\n".join(f"- {n}" for n in EN_LABELS)
SYSTEM_PROMPT_LLM_TR = ("Sen bir afet müdahale asistanısın. Bir depremzedenin sosyal medya paylaşımına dayanarak, aşağıdaki afet yardım "
                        "ihtiyaçları listesinden en acil gereksinim duyulan tek yardım kalemini belirle:\n" + _LIST_TR +
                        "\nHer paylaşım için yalnızca ihtiyacın adıyla, listede yazıldığı biçimde yanıt ver.")
SYSTEM_PROMPT_MULTI_EN = ("You are a disaster response assistant. Based on the social media message of an earthquake victim, identify the relief "
                          "items from the following list of disaster relief needs that the message asks for or clearly implies, ordered from the "
                          "most to the least urgent (at most 3 items):\n" + _LIST_EN +
                          "\nFor each message respond only with the names of the needs, exactly as written in the list.")
SYSTEM_PROMPT_MULTI_TR = ("Sen bir afet müdahale asistanısın. Bir depremzedenin sosyal medya paylaşımına dayanarak, aşağıdaki afet yardım "
                          "ihtiyaçları listesinden paylaşımda istenen ya da açıkça ima edilen yardım kalemlerini en acilden en az acile doğru "
                          "sıralayarak belirle (en fazla 3 kalem):\n" + _LIST_TR +
                          "\nHer paylaşım için yalnızca ihtiyaçların adlarıyla, listede yazıldığı biçimde yanıt ver.")
OPEN_LABELS = list(EN_LABELS) + ["not in list"]
SYSTEM_PROMPT_OPEN = (
    "You are a disaster response assistant. You will read social media posts written after earthquakes. For each post decide:\n"
    "1. has_need: true if the post reports a concrete need of people affected by the earthquake: an item, service or team that is "
    "needed or requested, for the author or for others (including rescue teams for people trapped under rubble, and aid-collection "
    "lists that name concrete items). false for news, donation announcements that name no concrete item, prayers and solidarity "
    "messages, political comments, announcements of aid that has arrived, and general calls for help that name no concrete need.\n"
    "2. needs: if has_need is true, the relief items from the list below that the post asks for or clearly implies, from the most to "
    "the least urgent (at most 3). If a needed item has no equivalent in the list, use \"not in list\". If has_need is false, return "
    "an empty list.\nList of relief items:\n" + _LIST_EN + "\nRespond only with item names exactly as written in the list."
    + (("\nAdditional rules:\n" + GERCEK_ISTEM_EK) if GERCEK_ISTEM_EK else ""))
LLM_SCHEMA_TR = _schema({"need": {"type": "string", "enum": list(NEEDS_TR)}})
LLM_SCHEMA_MULTI_EN = _schema({"needs": {"type": "array", "items": {"type": "string", "enum": list(EN_LABELS)}}})
LLM_SCHEMA_MULTI_TR = _schema({"needs": {"type": "array", "items": {"type": "string", "enum": list(NEEDS_TR)}}})
LLM_SCHEMA_OPEN = _schema({"has_need": {"type": "boolean"}, "needs": {"type": "array", "items": {"type": "string", "enum": OPEN_LABELS}}})

ub_en = build_llm_user          # makaledeki kullanıcı istemi (birebir aynı)
def ub_tr(batch):
    return "Paylaşımlar:\n" + "\n".join(f"[id {i}] {t}" for i, t in batch) + \
           "\nHer paylaşım kimliği için bir giriş içeren JSON döndür: {\"predictions\":[{\"id\":..,\"need\":..}, ...]}."
def ub_multi_en(batch):
    return "Messages:\n" + "\n".join(f"[id {i}] {t}" for i, t in batch) + \
           "\nReturn JSON {\"predictions\":[{\"id\":..,\"needs\":[..]}, ...]} with one entry per message id."
def ub_multi_tr(batch):
    return "Paylaşımlar:\n" + "\n".join(f"[id {i}] {t}" for i, t in batch) + \
           "\nHer paylaşım kimliği için bir giriş içeren JSON döndür: {\"predictions\":[{\"id\":..,\"needs\":[..]}, ...]}."
def ub_open(batch):
    return "Posts:\n" + "\n".join(f"[id {i}] {t}" for i, t in batch) + \
           "\nReturn JSON {\"predictions\":[{\"id\":..,\"has_need\":..,\"needs\":[..]}, ...]} with one entry per post id."

def p_single_en(p):
    return normalize_need(p.get("need"))
def p_single_tr(p):
    return normalize_need_tr(p.get("need"))
def _p_multi(p, norm):
    L = p.get("needs")
    if not isinstance(L, list):
        return None
    return [x for x in dict.fromkeys(norm(v) for v in L) if x][:3]
def p_multi_en(p):
    return _p_multi(p, normalize_need)
def p_multi_tr(p):
    return _p_multi(p, normalize_need_tr)
def p_open(p):
    hn = p.get("has_need")
    if not isinstance(hn, bool):
        return None
    out = []
    for x in p.get("needs") or []:
        if isinstance(x, str) and x.strip().lower() == "not in list":
            out.append("not in list")
        else:
            n = normalize_need(x)
            if n:
                out.append(n)
    return {"has_need": hn, "needs": list(dict.fromkeys(out))[:3] if hn else []}

# 66 örnekli istem: R6'daki eğitim bölümünden her ihtiyaç için bir örnek (tek ihtiyaçlı paylaşımlar öncelikli)
SYSTEM_PROMPT_FEW = SYSTEM_PROMPT_LLM
if FEW_EXAMPLES:
    SYSTEM_PROMPT_FEW = SYSTEM_PROMPT_LLM + "\n\nExamples (message -> need):\n" + "\n".join(
        f"Message: {json.dumps(str(df_syn.at[i, 'English']), ensure_ascii=False)} -> {en}" for en, i in FEW_EXAMPLES.items())
with open(os.path.join(DIRS["rev"], "bdm_istemleri.txt"), "w", encoding="utf-8") as _f:
    for _n, _p in (("tek_en (makale)", SYSTEM_PROMPT_LLM), ("tek_tr", SYSTEM_PROMPT_LLM_TR), ("coklu_en", SYSTEM_PROMPT_MULTI_EN),
                   ("coklu_tr", SYSTEM_PROMPT_MULTI_TR), ("az_ornekli_en", SYSTEM_PROMPT_FEW), ("acik_kume", SYSTEM_PROMPT_OPEN)):
        _f.write(f"===== {_n} =====\n{_p}\n\n")

# GPT-5 mini'nin ilk denemede varsayılan düşünmeyle yanıtlanan 210 paylaşımı (makaledeki çalıştırma günlüğünden)
REDO_IDS_GPT = list(range(20, 30)) + list(range(1710, 1870)) + list(range(1880, 1910)) + list(range(1940, 1950))
assert len(REDO_IDS_GPT) == 210

LLM_OUT = {}
def _items(df, col, ids=None):
    d = df if ids is None else df[df["id"].isin(ids)]
    return list(zip(d["id"].astype(int), d[col].astype(str)))

def add_job(jobs, provider, model, set_name, variant, items, system, schema, ub, parser, mt=150):
    if not items:
        return
    def fn():
        LLM_OUT[(set_name, variant, model)] = run_llm(provider, model, items, variant, set_name, system, schema, ub, parser,
                                                      max_tokens_per_item=mt)
    jobs.append((provider, fn))

_jobs = []
_syn_en, _syn_tr = _items(df_syn, "English"), _items(df_syn, "Message")
for provider, model in REV_LLMS:
    if not has_key(provider):
        log(f"{model}: {provider} anahtarı yok, BDM deneyleri atlanıyor"); continue
    for s, d in REAL_SETS.items():                                   # gerçek veri (küçük, önce)
        pos_ids = d.loc[d["has_need"], "id"].tolist()
        if s == "gercek_en":
            add_job(_jobs, provider, model, s, "tek_en", _items(d, "English", pos_ids), SYSTEM_PROMPT_LLM, LLM_SCHEMA, ub_en, p_single_en)
            add_job(_jobs, provider, model, s, "acik_kume", _items(d, "English"), SYSTEM_PROMPT_OPEN, LLM_SCHEMA_OPEN, ub_open, p_open, mt=200)
        else:
            add_job(_jobs, provider, model, s, "tek_tr", _items(d, "Message", pos_ids), SYSTEM_PROMPT_LLM_TR, LLM_SCHEMA_TR, ub_tr, p_single_tr)
            add_job(_jobs, provider, model, s, "tek_en_mt", _items(d, "English", pos_ids), SYSTEM_PROMPT_LLM, LLM_SCHEMA, ub_en, p_single_en)
            add_job(_jobs, provider, model, s, "acik_kume", _items(d, "Message"), SYSTEM_PROMPT_OPEN, LLM_SCHEMA_OPEN, ub_open, p_open, mt=200)
    if HUMAID_SET is not None:
        add_job(_jobs, provider, model, "humaid", "acik_kume", _items(HUMAID_SET, "English"), SYSTEM_PROMPT_OPEN, LLM_SCHEMA_OPEN, ub_open, p_open, mt=200)
    if REV["bdm_gpt210"] and model == "gpt-5-mini":
        add_job(_jobs, provider, model, "sentetik", "tek_en_redo210", _items(df_syn, "English", REDO_IDS_GPT),
                SYSTEM_PROMPT_LLM, LLM_SCHEMA, ub_en, p_single_en)
    for s, d in ALT_SETS.items():
        if s != "alt_claude":
            add_job(_jobs, provider, model, s, "tek_en", _items(d, "English"), SYSTEM_PROMPT_LLM, LLM_SCHEMA, ub_en, p_single_en)
    if REV["bdm_turkce"]:
        add_job(_jobs, provider, model, "sentetik", "tek_tr", _syn_tr, SYSTEM_PROMPT_LLM_TR, LLM_SCHEMA_TR, ub_tr, p_single_tr)
    if REV["bdm_tekrar"]:
        for t in range(2, 2 + BDM_TEKRAR):
            add_job(_jobs, provider, model, "sentetik", f"tek_en_t{t}", _syn_en, SYSTEM_PROMPT_LLM, LLM_SCHEMA, ub_en, p_single_en)
    if REV["bdm_coklu"]:
        add_job(_jobs, provider, model, "sentetik", "coklu_en", _syn_en, SYSTEM_PROMPT_MULTI_EN, LLM_SCHEMA_MULTI_EN, ub_multi_en, p_multi_en, mt=200)
        add_job(_jobs, provider, model, "sentetik", "coklu_tr", _syn_tr, SYSTEM_PROMPT_MULTI_TR, LLM_SCHEMA_MULTI_TR, ub_multi_tr, p_multi_tr, mt=200)
    if REV["bdm_az_ornekli"] and FEW_TEST:
        add_job(_jobs, provider, model, "sentetik_test", "az_ornekli_en", _items(df_syn, "English", FEW_TEST),
                SYSTEM_PROMPT_FEW, LLM_SCHEMA, ub_en, p_single_en)
log(f"BDM işleri: {len(_jobs)} (sağlayıcılar paralel, her sağlayıcı kendi içinde sırayla)")
_t0 = time.perf_counter()
run_jobs_by_provider(_jobs)
log(f"BDM çağrıları bitti: {(time.perf_counter() - _t0) / 60:.1f} dk; tahmini toplam API maliyeti ${total_cost_all():.2f}")

# --- Puanlama ----------------------------------------------------------------------------------------------------
def tops_single(preds, ids):
    return [[preds[i]] if preds.get(i) else [] for i in ids]

def info_extra(key):
    inf = LLM_OUT.get(key, (None, {}))[1] or {}
    return {"kapsam": inf.get("kapsam"), "ms_paylasim": inf.get("ms_paylasim_medyan"), "durma": inf.get("durma")}

# Makaledeki çalıştırma (1. çalıştırma) ve GPT-5 mini için protokol tutarlı birleşim
PAPER_LLM = {}
for _, m in REV_LLMS:
    d = paper_pred(PAPER_LLM_TAGS[m])
    if d is not None:
        PAPER_LLM[m] = {int(i): (t if isinstance(t, str) else None) for i, t in zip(d["id"], d["top1"])}
RUN1 = dict(PAPER_LLM)
_k = ("sentetik", "tek_en_redo210", "gpt-5-mini")
if _k in LLM_OUT and "gpt-5-mini" in PAPER_LLM:
    redo = LLM_OUT[_k][0]
    RUN1["gpt-5-mini"] = {**PAPER_LLM["gpt-5-mini"], **redo}
    sub = df_syn[df_syn["id"].isin(REDO_IDS_GPT)].reset_index(drop=True); ids = sub["id"].astype(int).tolist()
    old = [PAPER_LLM["gpt-5-mini"].get(i) for i in ids]; new = [redo.get(i) for i in ids]
    h_old, _ = hit_scores(sub["labels_en"].tolist(), old); h_new, _ = hit_scores(sub["labels_en"].tolist(), new)
    rev_record("bdm_gpt210", "sentetik_210", "BDM gpt-5-mini", "varsayilan_dusunme(makale)", **h_old)
    rev_record("bdm_gpt210", "sentetik_210", "BDM gpt-5-mini", "minimal_dusunme(yeniden)", **h_new,
               degisen_tahmin=int(sum(a != b for a, b in zip(old, new))), **info_extra(_k))
    ids_all = df_syn["id"].astype(int).tolist()
    score_record("bdm_gpt210", "sentetik", df_syn, "BDM gpt-5-mini", "makale(karisik_protokol)", tops_single(PAPER_LLM["gpt-5-mini"], ids_all))
    score_record("bdm_gpt210", "sentetik", df_syn, "BDM gpt-5-mini", "protokol_tutarli", tops_single(RUN1["gpt-5-mini"], ids_all))

def fleiss_kappa(runs, ids):
    cats = sorted({p.get(i) for p in runs for i in ids if p.get(i)})
    ci = {c: k for k, c in enumerate(cats)}; n = len(runs)
    C = np.zeros((len(ids), len(cats)))
    for r, i in enumerate(ids):
        for p in runs:
            if p.get(i) in ci:
                C[r, ci[p.get(i)]] += 1
    C = C[C.sum(axis=1) == n]
    if len(C) == 0 or n < 2:
        return float("nan")
    P_i = ((C ** 2).sum(axis=1) - n) / (n * (n - 1)); p_j = C.sum(axis=0) / (len(C) * n); Pe = (p_j ** 2).sum()
    return float((P_i.mean() - Pe) / (1 - Pe)) if Pe < 1 else float("nan")

ids_all = df_syn["id"].astype(int).tolist()
_rep_rows = []
for _, m in REV_LLMS:
    labeled = [(1, RUN1.get(m))] + [(t, LLM_OUT.get(("sentetik", f"tek_en_t{t}", m), (None,))[0]) for t in range(2, 2 + BDM_TEKRAR)]
    labeled = [(t, r) for t, r in labeled if r]
    runs = [r for _, r in labeled]
    if not runs:
        continue
    for t, r in labeled:
        if t > 1:
            score_record("bdm_tekrar", "sentetik", df_syn, f"BDM {m}", f"tekrar{t}", tops_single(r, ids_all),
                         extra=info_extra(("sentetik", f"tek_en_t{t}", m)))
    h = [hit_scores(df_syn["labels_en"].tolist(), [r.get(i) for i in ids_all])[0]["hit1"] for r in runs]
    full = [i for i in ids_all if all(r.get(i) for r in runs)]
    row = {"Model": m, "calistirma": len(runs), "hit1_ort": float(np.mean(h)), "hit1_ss": float(np.std(h, ddof=1)) if len(h) > 1 else float("nan"),
           "hit1_min": float(np.min(h)), "hit1_max": float(np.max(h)),
           "tum_tekrarlarda_ayni": float(np.mean([len({r[i] for r in runs}) == 1 for i in full])) if len(runs) > 1 else float("nan"),
           "fleiss_kappa": fleiss_kappa(runs, full) if len(runs) > 1 else float("nan"), "ortak_yanitli": len(full)}
    _rep_rows.append(row)
    rev_record("bdm_tekrar", "sentetik", f"BDM {m}", "ozet", **{k: v for k, v in row.items() if k != "Model"})
if _rep_rows:
    save_table(pd.DataFrame(_rep_rows), "revizyon/bdm_tekrar_degiskenligi")

for _, m in REV_LLMS:
    k = ("sentetik", "tek_tr", m)
    if k in LLM_OUT:
        score_record("bdm_turkce", "sentetik", df_syn, f"BDM {m}", "tr/tek", tops_single(LLM_OUT[k][0], ids_all), extra=info_extra(k))
    for lang in ("en", "tr"):
        k = ("sentetik", f"coklu_{lang}", m)
        if k in LLM_OUT:
            preds = LLM_OUT[k][0]; tops = [list(preds.get(i) or []) for i in ids_all]
            ml = multilabel_scores(df_syn["labels_en"].tolist(), tops)
            score_record("bdm_coklu", "sentetik", df_syn, f"BDM {m}", f"{lang}/coklu", tops, extra={**ml, **info_extra(k)})
    k = ("sentetik_test", "az_ornekli_en", m)
    if FEW_TEST and k in LLM_OUT:
        tids = DF_FEW_TEST["id"].astype(int).tolist()
        score_record("az_ornekli", "sentetik_test", DF_FEW_TEST, f"BDM {m}", "66_ornekli", tops_single(LLM_OUT[k][0], tids), extra=info_extra(k))
        if m in RUN1:
            score_record("az_ornekli", "sentetik_test", DF_FEW_TEST, f"BDM {m}", "sifir_atis/makale", tops_single(RUN1[m], tids))

# Alternatif üreticiler: Claude metinleri için makaledeki (GPT için protokol tutarlı) tahminler, diğerleri için yeni çalıştırma
_cross = []
for s, d in ALT_SETS.items():
    sids = d["id"].astype(int).tolist()
    for _, m in REV_LLMS:
        if s == "alt_claude":
            if m not in RUN1:
                continue
            row = score_record("bdm_alternatif", s, d, f"BDM {m}", "tek_en(makale)", tops_single(RUN1[m], sids))
        else:
            k = (s, "tek_en", m)
            if k not in LLM_OUT:
                continue
            row = score_record("bdm_alternatif", s, d, f"BDM {m}", "tek_en", tops_single(LLM_OUT[k][0], sids), extra=info_extra(k))
        _cross.append({"metin_ureticisi": s.replace("alt_", ""), "degerlendiren": m, "hit1": row.get("hit1"), "hit3": row.get("hit3"),
                       "ilce_dogruluk": row.get("ilce_dogruluk_makale"), "n": row.get("n_mesaj")})
if _cross:
    _c = pd.DataFrame(_cross)
    save_table(_c, "revizyon/bdm_uretici_capraz")
    print(_c.pivot_table(index="degerlendiren", columns="metin_ureticisi", values="hit1").round(3).to_string())

# Gerçek veri
for s, d in REAL_SETS.items():
    pos = d[d["has_need"]].reset_index(drop=True); pids = pos["id"].astype(int).tolist()
    for _, m in REV_LLMS:
        for var in ("tek_en", "tek_tr", "tek_en_mt"):
            k = (s, var, m)
            if k in LLM_OUT:
                score_record("bdm_gercek", s, pos, f"BDM {m}", f"kapali/{var}", tops_single(LLM_OUT[k][0], pids), extra=info_extra(k))
        k = (s, "acik_kume", m)
        if k in LLM_OUT:
            preds = LLM_OUT[k][0]
            sub = d[d["id"].astype(int).isin(preds)].reset_index(drop=True)
            P = [preds[int(i)] for i in sub["id"]]
            res = binary_scores(sub["has_need"].tolist(), [p["has_need"] for p in P])
            pk = [j for j, h in enumerate(sub["has_need"]) if h]
            gold = [list(sub["labels_en"].iat[j]) + (["not in list"] if sub["ly_list"].iat[j] else []) for j in pk]
            ml = multilabel_scores(gold, [P[j]["needs"] for j in pk])
            ly = binary_scores([bool(sub["ly_list"].iat[j]) for j in pk], ["not in list" in P[j]["needs"] for j in pk])
            gsec = [list(sub["sectors"].iat[j]) for j in pk]
            psec = [sorted({SECTOR_OF_EN[x] for x in P[j]["needs"] if x in SECTOR_OF_EN}) for j in pk]
            sec = multilabel_scores(gsec, psec)
            rev_record("bdm_gercek", s, f"BDM {m}", "acik_kume", n=len(sub), **res,
                       **{f"ihtiyac_{a}": b for a, b in ml.items()}, **{f"sektor_{a}": b for a, b in sec.items()},
                       listede_yok_kesinlik=ly["ikili_kesinlik"], listede_yok_duyarlilik=ly["ikili_duyarlilik"], listede_yok_f1=ly["ikili_f1"],
                       **info_extra(k))
            pd.DataFrame({"id": sub["id"], "gercek_somut": sub["has_need"], "tahmin_somut": [p["has_need"] for p in P],
                          "gercek_ihtiyac": [";".join(L) for L in sub["labels_en"]], "tahmin_ihtiyac": [";".join(p["needs"]) for p in P]}
                         ).to_csv(os.path.join(DIRS["preds"], f"mesaj_{safe_tag(s, 'BDM', m, 'acik_kume')}.csv"), index=False, encoding="utf-8-sig")

# HumAID: BDM'nin "somut ihtiyaç var" kararı ile insan etiketi (requests_or_urgent_needs) karşılaştırması
if HUMAID_SET is not None:
    _hrows = []
    for _, m in REV_LLMS:
        k = ("humaid", "acik_kume", m)
        if k not in LLM_OUT:
            continue
        preds = LLM_OUT[k][0]
        sub = HUMAID_SET[HUMAID_SET["id"].astype(int).isin(preds)].reset_index(drop=True)
        pred_need = [preds[int(i)]["has_need"] for i in sub["id"]]
        res = binary_scores(sub["is_request"].tolist(), pred_need)
        rev_record("bdm_humaid", "humaid", f"BDM {m}", "acik_kume/istek_ayrimi", n=len(sub), **res, **info_extra(k))
        for c, g in pd.DataFrame({"sinif": sub["sinif"], "p": pred_need}).groupby("sinif"):
            _hrows.append({"Model": m, "sinif": c, "n": len(g), "somut_ihtiyac_dedi": float(np.mean(g["p"]))})
    if _hrows:
        save_table(pd.DataFrame(_hrows), "revizyon/humaid_sinif_bazinda")

# Kullanım ve maliyet özeti
_u = []
for m in sorted(set(USAGE) | set(OTHER_USAGE)):
    src = USAGE.get(m) or OTHER_USAGE.get(m) or {}
    _u.append({"Model": m, "cagri": src.get("calls", 0), "giris_token": src.get("input_tokens", 0), "cikis_token": src.get("output_tokens", 0),
               "dusunme_token": src.get("thinking_tokens", 0), "tahmini_usd": usage_cost_usd(m) if m in USAGE else other_usage_cost(m)})
if _u:
    save_table(pd.DataFrame(_u), "revizyon/api_kullanim_maliyet")
log(f"BDM deneyleri tamam. Oturumdaki tahmini toplam API maliyeti: ${total_cost_all():.2f}")

In [ ]:
# =============================================================================
# R9. ÖZET TABLOLAR VE SONUÇ ZIP'İ
# =============================================================================
_res = pd.DataFrame(list(REV_RESULTS.values()))
if not _res.empty:
    save_table(_res, "revizyon/revizyon_tum_sonuclar")
    _show = [c for c in ("n_mesaj", "hit1", "hit1_ga_alt", "hit1_ga_ust", "hit3", "ilce_dogruluk_makale", "sektor_hit1", "mikro_f1", "makro_f1",
                         "tum_ihtiyaclar_bulundu", "ikili_f1", "dogruluk", "kapsam") if c in _res.columns]
    pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 40)
    for dn, g in _res.groupby("deney", sort=False):
        key = "hit1" if "hit1" in g and g["hit1"].notna().any() else ("dogruluk" if "dogruluk" in g and g["dogruluk"].notna().any() else None)
        g2 = g.sort_values(["veri", key], ascending=[True, False]) if key else g
        cols = ["veri", "yontem", "kosul"] + [c for c in _show if c in g2 and g2[c].notna().any()]
        print(f"\n===== {dn} ({len(g2)} satır) =====")
        print(g2[cols].head(40).round(3).to_string(index=False))

_zip = os.path.join(BASE_DIR, f"revizyon_sonuclari_{RUN_TAG}.zip")
_roots = [RUN_DIR, REV_CACHE] + ([ALT_DIR] if REV.get("alternatif_uretici") and "ALT_DIR" in globals() else [])
with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as _z:
    for _root in _roots:
        for _dp, _, _fs in os.walk(_root):
            for _fn in _fs:
                _full = os.path.join(_dp, _fn)
                _z.write(_full, os.path.relpath(_full, BASE_DIR))
    _mt = os.path.join(BASE_DIR, REAL_DIR_NAME, "makine_cevirisi_tr_en.csv")   # çeviri kalitesinin denetimi için
    if os.path.exists(_mt):
        _z.write(_mt, os.path.relpath(_mt, BASE_DIR))
log(f"Sonuç zip'i: {_zip} ({os.path.getsize(_zip) / 1e6:.1f} MB). Tahmini toplam API maliyeti: ${total_cost_all():.2f}")
log("Tüm sonuçlar tek zip dosyası olarak hazırlandı.")
if IN_COLAB and AUTO_DOWNLOAD_ZIP:
    try:
        files.download(_zip)
    except Exception as e:
        log("Zip otomatik indirilemedi (Drive'daki dosyayı kullanın):", e)